# 🎬🎧 StemTube Studio | YouTube, áudio e IA em um só lugar

Cole um **link do YouTube** (vídeo, Shorts, playlist ou canal) ou envie um **arquivo do computador** e escolha o que fazer:

- 🎬 **Baixar o vídeo em MP4** em qualquer resolução disponível (144p a 8K), com playlists inteiras e seleção item a item.
- 🎵 **Baixar só o áudio** em MP3, M4A/AAC, OPUS, FLAC, WAV ou no formato original, com escolha de qualidade e capa embutida.
- ✂️ **Recortar um trecho** por tempo (início e fim), em vídeo ou áudio, com recodificação para corte preciso (no formato de áudio original, o corte é aproximado).
- 🎧 **Separar voz e instrumentos com IA** em 2, 4 ou 6 faixas, ouvir no **mixer** dentro da página, silenciar/isolar faixas e **renderizar sua própria mixagem**.

Tudo com prévia antes de começar, fila com progresso em tempo real, ZIP de tudo e cópia opcional no Google Drive.

**Como abrir o Studio**

1. Rode a célula **1️⃣** — faça login na conta Google (obrigatório). Ela instala as dependências. Desmarque `instalar_separador_ia` se quiser só os downloads.
2. Rode a célula **2️⃣** — clique em **Abrir StemTube Studio em nova aba**. Mantenha o notebook aberto e conectado enquanto usa o Studio. Cole o link ou envie o arquivo e siga as etapas.

> 💡 Ou use **Ambiente de execução ▸ Executar tudo** (`Ctrl+F9`).

> ⚡ **A separação de faixas usa a GPU.** O notebook solicita uma T4, mas a disponibilidade depende do Colab. Se o chip no topo do app mostrar *Sem GPU*, vá em **Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU** e rode as células de novo. Os downloads funcionam com ou sem GPU.

**Vídeos privados, +18 ou só para membros:** dentro do app, abra a seção *Cookies da sua conta* e carregue o `cookies.txt` exportado do navegador uma única vez. Com o Drive conectado ele fica salvo em `Meu Drive/StemTube/cookies.txt` (cookies das versões anteriores, StemLab e YT Downloader, são migrados automaticamente).

> ⚠️ Uso **pessoal**. Respeite os termos do YouTube e os direitos autorais dos criadores. Nunca compartilhe seu arquivo de cookies.

---
*Desenvolvido por **@emersonms** - 2026*


In [ ]:
#@title 1️⃣ Preparar o ambiente e entrar com a conta Google  { display-mode: "form" }
#@markdown Clique no ▶ à esquerda. Uma janela do Google vai pedir autorização — isso é obrigatório para usar o app. A instalação leva de 1 a 3 minutos.
conectar_google_drive = True  #@param {type:"boolean"}
#@markdown *Com o Drive conectado, os cookies ficam salvos entre sessões e os arquivos podem ser copiados para `Meu Drive/StemTube`.*
instalar_separador_ia = True  #@param {type:"boolean"}
#@markdown *Desmarque para preparar só os downloads (mais rápido). A separação de faixas com IA precisa deste componente.*

import os, sys, shutil, subprocess, json, time
if globals().get('_STEMTUBE_STATE'):
    raise RuntimeError('O ambiente já foi preparado. Para atualizar dependências, reinicie a sessão. Para reabrir a interface, execute somente a célula 2.')
_STEMTUBE_SETUP_READY = False
from IPython.display import HTML, display, clear_output

def _box(msg, kind='info'):
    cor = {'info': '#3b82f6', 'ok': '#22c55e', 'warn': '#f59e0b', 'err': '#ef4444'}[kind]
    display(HTML(f'<div style="font-family:system-ui,sans-serif;font-size:14px;padding:10px 14px;margin:6px 0;border-radius:10px;'
                 f'border-left:4px solid {cor};background:{cor}14;color:inherit">{msg}</div>'))

# ---------- 1. Login Google (obrigatório) ----------
_box('🔐 Aguardando login na conta Google… aceite a janela que abriu.')
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    raise SystemExit(f'Login cancelado ou falhou: {e}. Rode a célula novamente e conclua o login.')

USER_EMAIL = None
try:
    import google.auth, google.auth.transport.requests, requests
    _creds, _ = google.auth.default()
    _creds.refresh(google.auth.transport.requests.Request())
    USER_EMAIL = requests.get('https://www.googleapis.com/oauth2/v3/userinfo',
                              headers={'Authorization': f'Bearer {_creds.token}'}, timeout=15).json().get('email')
except Exception:
    pass
USER_EMAIL = USER_EMAIL or 'Conta Google conectada'

# ---------- 2. Google Drive (opcional) ----------
DRIVE_OK = False
if conectar_google_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_OK = os.path.isdir('/content/drive/MyDrive')
    except Exception as e:
        _box(f'Drive não conectado: {e}', 'warn')

# ---------- 3. GPU ----------
GPU_NAME = None
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
except Exception:
    pass
if instalar_separador_ia and not GPU_NAME:
    _box('⚠️ <b>Nenhuma GPU nesta sessão.</b> Downloads funcionam normalmente, mas a separação de faixas vai ficar muito lenta. '
         'Vá em <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU</b> e rode esta célula de novo.', 'warn')

# ---------- 4. Dependências ----------
_t0 = time.time()
_pkgs = ['yt-dlp[default]'] + (['audio-separator[cpu]', 'audioread'] if instalar_separador_ia else [])
_box('📦 Instalando ' + ('yt-dlp, o separador de faixas (audio-separator) e o runtime JavaScript… 1 a 3 min' if instalar_separador_ia else 'yt-dlp e o runtime JavaScript… ~30 s'))
_r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U'] + _pkgs, capture_output=True, text=True, timeout=900)
if _r.returncode != 0:
    _box('Falha ao instalar as dependências. Copie a mensagem abaixo e envie para suporte.', 'err')
    print(_r.stderr[-3000:])
    raise SystemExit('pip falhou')
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True, timeout=300)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True, timeout=300)
if not shutil.which('deno'):
    try:
        subprocess.run(['curl', '-fSL', '--max-time', '120', 'https://deno.land/install.sh',
                        '-o', '/tmp/stemtube_deno_install.sh'], check=True, timeout=150)
        subprocess.run(['sh', '/tmp/stemtube_deno_install.sh'], env=dict(os.environ, DENO_INSTALL='/usr/local'),
                       stdin=subprocess.DEVNULL, check=True, timeout=300, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception as e:
        _box(f'O runtime JavaScript (deno) não pôde ser instalado ({e}). Alguns vídeos podem falhar; o app continua funcionando.', 'warn')
os.environ['PATH'] += ':/usr/local/bin'
import importlib, yt_dlp
importlib.reload(yt_dlp)

SEP_VERSION = None
if instalar_separador_ia:
    try:
        import audio_separator
        SEP_VERSION = getattr(audio_separator, '__version__', None) or 'ok'
    except Exception as e:
        raise RuntimeError('O separador não pôde ser importado. Reinicie a sessão e execute novamente a preparação.') from e
    _cli_check = subprocess.run(['audio-separator', '--help'], capture_output=True, text=True, timeout=120)
    if _cli_check.returncode or '--custom_output_names' not in _cli_check.stdout:
        raise RuntimeError('O comando audio-separator não está operacional ou é incompatível. Reinicie a sessão.')

_STEMTUBE_SETUP_READY = True
clear_output()
_box(f'✅ <b>Login OK</b> — {USER_EMAIL}', 'ok')
_box(('✅ <b>Google Drive conectado</b> — cookies, downloads, faixas e mixagens podem ser salvos em <code>Meu Drive/StemTube</code>' if DRIVE_OK
      else 'ℹ️ Google Drive <b>não conectado</b> — os cookies valerão só nesta sessão.'), 'ok' if DRIVE_OK else 'info')
if instalar_separador_ia:
    _box((f'⚡ GPU: <b>{GPU_NAME}</b>' if GPU_NAME else '⚠️ <b>Sem GPU</b> — ative em Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU'), 'ok' if GPU_NAME else 'warn')
_box(f'✅ yt-dlp <b>{yt_dlp.version.__version__}</b> · separador de IA {"<b>instalado</b>" if SEP_VERSION else "<b>não instalado</b> (só downloads)"} · '
     f'ffmpeg {"OK" if shutil.which("ffmpeg") else "NÃO encontrado"} · deno {"OK" if shutil.which("deno") else "não instalado"} · {int(time.time()-_t0)} s', 'ok')
_box('👉 Agora rode a célula <b>2️⃣ Abrir o aplicativo</b>.' + (' O modelo de IA de voz começa a ser baixado em segundo plano assim que o app abrir.' if SEP_VERSION else ''), 'info')


In [ ]:
#@title 2️⃣ Abrir o aplicativo  { display-mode: "form" }
#@markdown Execute esta célula e clique em **Abrir StemTube Studio em nova aba**. Se o ambiente for reiniciado, rode a célula 1 e depois esta.
abrir_em_nova_aba = True  #@param {type:"boolean"}
#@markdown Se o navegador bloquear a nova aba, desmarque a opção para usar a interface nesta célula.
import json as _json
from IPython.display import HTML as _HTML, display as _display

def _erro(msg):
    _display(_HTML(f'<div style="font-family:system-ui,sans-serif;padding:14px 16px;border-radius:12px;border-left:4px solid #ef4444;'
                   f'background:#ef444414;font-size:14px">⛔ <b>{msg}</b></div>'))

_pronto = True
if not globals().get('_STEMTUBE_SETUP_READY'):
    _erro('Execute primeiro a célula 1️⃣ (login na conta Google).'); _pronto = False
else:
    try:
        import google.auth as _ga; _ga.default()
    except Exception:
        _erro('Login na conta Google necessário. Rode a célula 1️⃣ novamente.'); _pronto = False
    try:
        import yt_dlp as _ytdlp
    except Exception:
        _erro('O yt-dlp não está instalado. Rode a célula 1️⃣ novamente e aguarde a instalação terminar.'); _pronto = False

if _pronto:
    try:
        # ============================================================
        #  StemTube Studio — backend (roda dentro do Google Colab)
        #  Expõe funções Python para a interface web via
        #  google.colab.output.register_callback / kernel.invokeFunction
        #
        #  Reúne, em um único aplicativo:
        #    • download de vídeos, playlists e canais do YouTube (MP4 ou áudio)
        #    • recorte por tempo de um vídeo
        #    • separação de voz e instrumentos por IA (audio-separator)
        #    • mixer e renderização de mixagens
        #
        #  Desenvolvido por @emersonms - 2026
        # ============================================================
        import os, re, json, time, uuid, shutil, threading, socketserver, http.server
        import urllib.parse, traceback, queue, subprocess, base64, sys, math, tempfile

        import zipfile

        import yt_dlp
        from IPython.display import JSON

        APP_NAME    = 'StemTube Studio'
        APP_VERSION = '1.0.0'

        # ---------- caminhos ----------
        BASE_DIR      = '/content/stemtube'
        WORK_DIR      = os.path.join(BASE_DIR, 'work')       # áudio de origem (wav) e uploads
        OUT_DIR       = os.path.join(BASE_DIR, 'out')        # resultados por tarefa (servidos ao navegador)
        MODELS_DIR    = os.path.join(BASE_DIR, 'models')     # modelos de IA baixados
        COOKIES_PATH  = os.path.join(BASE_DIR, 'cookies.txt')
        DRIVE_MYDRIVE = '/content/drive/MyDrive'
        DRIVE_ROOT    = os.path.join(DRIVE_MYDRIVE, 'StemTube')
        LEGACY_DRIVE_ROOTS = [os.path.join(DRIVE_MYDRIVE, 'StemLab'), os.path.join(DRIVE_MYDRIVE, 'YT_Downloader')]
        FILE_PORT     = 8767
        MAX_UPLOAD_MB = 1024
        MAX_DURATION  = 25 * 60   # 25 min: acima disso a GPU gratuita costuma estourar a memória na separação
        MAX_PLAYLIST  = 1000
        MOCK          = bool(os.environ.get('STEMTUBE_MOCK'))  # modo de teste local sem GPU/modelos
        for _d in (WORK_DIR, OUT_DIR, MODELS_DIR):
            os.makedirs(_d, exist_ok=True)

        try:
            import audio_separator as _audio_separator
            SEP_AVAILABLE = True
        except Exception:
            SEP_AVAILABLE = MOCK

        STATE = globals().get('_STEMTUBE_STATE') or {
            'jobs': {}, 'sources': {}, 'uploads': {},
            'user': globals().get('USER_EMAIL') or 'Conta Google conectada',
            'js_runtime': None, 'gpu': None, 'prefetch': {},
        }
        _STEMTUBE_STATE = STATE
        MODEL_LOCK = STATE.setdefault('model_lock', threading.Lock())

        # ---------- modos de separação ----------
        STEM_PT = {'Vocals': 'Voz', 'Instrumental': 'Playback', 'Drums': 'Bateria', 'Bass': 'Baixo',
                   'Guitar': 'Guitarra', 'Piano': 'Piano', 'Other': 'Outros'}
        STEM_ICON = {'Vocals': '🎤', 'Instrumental': '🎹', 'Drums': '🥁', 'Bass': '🎸', 'Guitar': '🎸', 'Piano': '🎹', 'Other': '🎼'}
        STEM_COLOR = {'Vocals': '#f472b6', 'Instrumental': '#38bdf8', 'Drums': '#fb923c', 'Bass': '#a78bfa',
                      'Guitar': '#facc15', 'Piano': '#34d399', 'Other': '#94a3b8'}
        MODES = {
            '2stem': {'label': 'Voz + Playback', 'model': 'model_bs_roformer_ep_317_sdr_12.9755.ckpt',
                      'stems': ['Vocals', 'Instrumental'], 'factor': 1.2, 'base': 30},
            '4stem': {'label': 'Banda (4 faixas)', 'model': 'htdemucs_ft.yaml',
                      'stems': ['Vocals', 'Drums', 'Bass', 'Other'], 'factor': 1.2, 'base': 40},
            '6stem': {'label': 'Completo (6 faixas)', 'model': 'htdemucs_6s.yaml',
                      'stems': ['Vocals', 'Drums', 'Bass', 'Guitar', 'Piano', 'Other'], 'factor': 0.5, 'base': 30},
        }
        OUT_FORMATS   = {'wav': 'WAV', 'flac': 'FLAC', 'mp3': 'MP3'}
        AUDIO_FORMATS = ('mp3', 'm4a', 'opus', 'flac', 'wav', 'best')
        AUDIO_QUALITIES = ('best', '320', '256', '192', '128', '96')
        UPLOAD_EXTS   = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.opus', '.wma', '.aiff', '.aif', '.mp4', '.webm', '.mkv', '.mov')
        YT_HOSTS      = ('youtube.com', 'www.youtube.com', 'm.youtube.com', 'music.youtube.com',
                         'youtube-nocookie.com', 'www.youtube-nocookie.com')

        # ---------- utilidades ----------
        def _drive_on():
            return os.path.isdir(DRIVE_MYDRIVE)

        def _detect_js_runtime():
            if STATE['js_runtime'] is not None:
                return STATE['js_runtime']
            rt = {}
            for name in ('deno', 'node', 'bun'):
                p = shutil.which(name)
                if p:
                    rt = {name: {'path': p}}
                    break
            STATE['js_runtime'] = rt
            return rt

        def _gpu_info(force=False):
            if STATE['gpu'] is not None and not force:
                return STATE['gpu']
            info = {'available': False, 'name': None, 'memory_gb': None, 'mock': MOCK}
            try:
                import torch
                if torch.cuda.is_available():
                    p = torch.cuda.get_device_properties(0)
                    info.update({'available': True, 'name': p.name, 'memory_gb': round(p.total_memory / 1024 ** 3, 1)})
            except Exception:
                pass
            STATE['gpu'] = info
            return info

        def _friendly_error(msg):
            m = (msg or '').lower()
            if 'sign in to confirm' in m or 'not a bot' in m or ('bot' in m and 'confirm' in m):
                return ('O YouTube pediu confirmação de que você não é um robô. Isso é comum em servidores do Colab. '
                        'Carregue os cookies da sua conta na seção "Cookies" e tente novamente.')
            if 'private video' in m or 'this video is private' in m:
                return 'Vídeo privado. Carregue os cookies de uma conta com acesso para usá-lo.'
            if 'members-only' in m or 'join this channel' in m:
                return 'Conteúdo exclusivo para membros. Carregue os cookies da conta que é membro do canal.'
            if 'age' in m and ('restrict' in m or 'confirm your age' in m or 'inappropriate' in m):
                return 'Vídeo com restrição de idade. Carregue os cookies da sua conta para confirmar a idade.'
            if 'video unavailable' in m or 'video is unavailable' in m or 'not available' in m:
                return 'Vídeo indisponível (removido, bloqueado na região ou link inválido).'
            if 'unsupported url' in m or 'is not a valid url' in m:
                return 'Este link não é reconhecido. Cole um link de vídeo, playlist ou canal do YouTube.'
            if 'requested format is not available' in m:
                return 'O formato escolhido não está disponível para este vídeo. Tente outra resolução ou formato.'
            if 'http error 429' in m or 'too many requests' in m:
                return 'O YouTube limitou as requisições (429). Aguarde alguns minutos ou use cookies.'
            if 'live' in m and ('is a live' in m or 'not yet' in m or 'premiere' in m):
                return 'Transmissões ao vivo ou estreias ainda não disponíveis não podem ser processadas.'
            if 'out of memory' in m or 'cuda error' in m:
                return 'A memória da GPU estourou. Tente uma música mais curta ou o modo "Voz + Playback".'
            return re.sub(r'^\s*ERROR:\s*', '', msg or 'Erro desconhecido').strip()[:400]

        class _SilentLogger:
            """Engole a saída do yt-dlp, mas guarda erros para reportar na interface."""
            def __init__(self): self.errors, self.warnings = [], []
            def debug(self, msg):   pass
            def info(self, msg):    pass
            def warning(self, msg): self.warnings.append(str(msg))
            def error(self, msg):   self.errors.append(str(msg))

        def _base_opts():
            o = {
                'quiet': True, 'no_warnings': True, 'noprogress': True, 'nocheckcertificate': False,
                'extractor_retries': 3, 'retries': 5, 'fragment_retries': 5, 'socket_timeout': 30,
                'geo_bypass': True, 'logger': _SilentLogger(),
            }
            if os.path.exists(COOKIES_PATH):
                o['cookiefile'] = COOKIES_PATH
            rt = _detect_js_runtime()
            if rt:
                o['js_runtimes'] = rt
            return o

        def _fmt_size(b):
            if not b: return None
            for unit in ('B', 'KB', 'MB', 'GB'):
                if b < 1024: return f'{b:.0f} {unit}' if unit == 'B' else f'{b:.1f} {unit}'
                b /= 1024
            return f'{b:.1f} TB'

        def _hms(sec):
            sec = int(round(sec or 0)); h, m, s = sec // 3600, sec % 3600 // 60, sec % 60
            return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

        def _thumb(info):
            t = info.get('thumbnail')
            if not t and info.get('thumbnails'):
                t = info['thumbnails'][-1].get('url')
            if not t and info.get('id'):
                t = f"https://i.ytimg.com/vi/{info['id']}/hqdefault.jpg"
            return t

        def _safe_name(s, n=90):
            s = yt_dlp.utils.sanitize_filename(str(s or 'audio'), restricted=False)
            s = re.sub(r'\s+', ' ', s).strip(' .')
            return s[:n] or 'audio'

        def _ffprobe_duration(path):
            try:
                out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', path],
                                     capture_output=True, text=True, timeout=60).stdout.strip()
                duration = float(out) if out else 0.0
                return duration if math.isfinite(duration) and duration > 0 else 0.0
            except Exception:
                return 0.0

        def _output_path(rel, preview=False):
            """Resolve um caminho relativo dentro de OUT_DIR, bloqueando travessia e links externos."""
            if not isinstance(rel, str) or not rel or os.path.isabs(rel):
                raise ValueError('Caminho de arquivo inválido.')
            root = os.path.realpath(OUT_DIR)
            path = os.path.realpath(os.path.join(root, rel))
            if os.path.commonpath((root, path)) != root or not os.path.isfile(path):
                raise FileNotFoundError('Arquivo não encontrado nesta sessão.')
            if preview and not path.endswith(('.preview.mp3', '.preview.mp4')):
                raise ValueError('Prévia inválida ou grande demais para a ponte do Colab.')
            return path

        def _check_cancel(job):
            if job.get('cancel'):
                raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')

        def _run_process(job, cmd, timeout=1800, progress_file=None, progress_total=None, on_progress=None):
            """Processo com leitura contínua, prazo e cancelamento, inclusive sem logs."""
            _check_cancel(job)
            with tempfile.TemporaryFile(mode='w+b') as log:
                proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
                job['proc'] = proc
                started = time.monotonic()
                try:
                    while proc.poll() is None:
                        _check_cancel(job)
                        if time.monotonic() - started > timeout:
                            raise TimeoutError('O processamento ultrapassou o tempo limite.')
                        total = progress_total or job.get('duration')
                        if progress_file and total:
                            try:
                                with open(progress_file) as f: values = re.findall(r'out_time_us=(\d+)', f.read())
                                if values:
                                    pct = min(99, int(values[-1]) / 1000000 / total * 100)
                                    if on_progress: on_progress(pct)
                                    else: job['percent'] = max(job.get('percent', 0), pct)
                            except OSError: pass
                        time.sleep(0.1)
                    _check_cancel(job)
                    if proc.returncode:
                        log.seek(max(0, log.tell() - 3000))
                        raise RuntimeError(log.read().decode('utf-8', errors='replace')[-3000:])
                finally:
                    if proc.poll() is None: proc.kill()
                    proc.wait()
                    job['proc'] = None

        # ---------- links do YouTube ----------
        def _parse_url(url, mode='auto'):
            """Descobre se o link aponta para vídeo, playlist, canal ou uma combinação.
            mode: 'auto' (detecta), 'video' (só o vídeo) ou 'playlist' (só a playlist/canal)."""
            u = (url or '').strip()
            if not re.match(r'^https?://', u, re.I):
                u = 'https://' + u
            p = urllib.parse.urlparse(u)
            host = (p.hostname or '').lower()
            if p.username or p.password or p.port not in (None, 80, 443):
                raise ValueError('unsupported url: endereço inválido.')
            qs = urllib.parse.parse_qs(p.query)
            video_id = None
            playlist_id = (qs.get('list') or [None])[0]
            if host == 'youtu.be':
                video_id = p.path.strip('/').split('/')[0] or None
            elif host in YT_HOSTS:
                if p.path == '/watch':
                    video_id = (qs.get('v') or [None])[0]
                else:
                    m = re.match(r'^/(shorts|live|embed|v)/([A-Za-z0-9_-]{11})', p.path)
                    if m: video_id = m.group(2)
                if p.path.startswith('/playlist'):
                    video_id = None
            else:
                raise ValueError('unsupported url: apenas links do YouTube são aceitos.')
            if video_id and not re.fullmatch(r'[A-Za-z0-9_-]{11}', video_id):
                raise ValueError('unsupported url: identificador de vídeo inválido.')
            if playlist_id and not re.fullmatch(r'[A-Za-z0-9_-]{2,128}', playlist_id):
                raise ValueError('unsupported url: identificador de playlist inválido.')
            is_mix = bool(playlist_id and playlist_id.startswith('RD'))
            channel_url = u if (not video_id and not playlist_id and re.match(r'^/(@[\w.-]+|channel/|c/|user/)', p.path)) else None
            if is_mix and video_id:
                playlist_url = f'https://www.youtube.com/watch?v={video_id}&list={playlist_id}'
            elif playlist_id:
                playlist_url = f'https://www.youtube.com/playlist?list={playlist_id}'
            elif channel_url:
                playlist_url = channel_url.rstrip('/')
                if not re.search(r'/(videos|shorts|streams|playlists)$', playlist_url):
                    playlist_url += '/videos'
            else:
                playlist_url = None
            if mode == 'video':
                if not video_id:
                    raise ValueError('Você escolheu "somente o vídeo", mas este link não aponta para um vídeo. Cole o link de um vídeo (watch?v=…, youtu.be/… ou /shorts/…).')
                playlist_id, playlist_url, channel_url = None, None, None
            elif mode == 'playlist':
                if not playlist_url:
                    raise ValueError('Você escolheu "playlist completa", mas este link não contém uma playlist. Abra a playlist no YouTube e copie o link que tem "list=" (ou o link de um canal).')
                video_id = None
            if not video_id and not playlist_url:
                raise ValueError('unsupported url: não encontrei um vídeo ou playlist nesse link.')
            return {'url': u, 'video_id': video_id, 'playlist_id': playlist_id, 'playlist_url': playlist_url,
                    'channel_url': channel_url, 'is_mix': is_mix}

        # ---------- cookies ----------
        def _cookie_domain_ok(domain):
            domain = domain.lower().lstrip('.')
            return any(domain == base or domain.endswith('.' + base) for base in ('youtube.com', 'google.com'))

        def _validate_cookie_content(content):
            rows = ['# Netscape HTTP Cookie File']
            for line in content.splitlines():
                raw = line[len('#HttpOnly_'):] if line.startswith('#HttpOnly_') else line
                if not raw.strip() or raw.startswith('#'):
                    continue
                parts = raw.split('\t')
                if len(parts) != 7 or parts[1] not in ('TRUE', 'FALSE') or parts[3] not in ('TRUE', 'FALSE'):
                    raise ValueError('Registro Netscape inválido: confira a exportação dos cookies.')
                if not parts[4].isdigit() or not parts[5]:
                    raise ValueError('Nome ou validade de cookie inválido.')
                if _cookie_domain_ok(parts[0]):
                    rows.append(line)
            if len(rows) == 1:
                raise ValueError('Não há cookies do YouTube ou Google neste arquivo.')
            return '\n'.join(rows) + '\n'

        def _json_cookies_to_netscape(items):
            lines = ['# Netscape HTTP Cookie File', f'# gerado pelo {APP_NAME}', '']
            for c in items:
                dom = c.get('domain') or '.youtube.com'
                flag = 'TRUE' if dom.startswith('.') else 'FALSE'
                path = c.get('path') or '/'
                secure = 'TRUE' if c.get('secure') else 'FALSE'
                exp = c.get('expirationDate') or c.get('expires') or c.get('expiry') or 0
                try: exp = int(float(exp))
                except Exception: exp = 0
                lines.append('\t'.join([dom, flag, path, secure, str(exp), str(c.get('name', '')), str(c.get('value', ''))]))
            return '\n'.join(lines) + '\n'

        def _header_cookies_to_netscape(header):
            header = re.sub(r'^\s*cookie\s*:\s*', '', header.strip(), flags=re.I)
            items = []
            for part in header.split(';'):
                if '=' in part:
                    n, v = part.strip().split('=', 1)
                    items.append({'domain': '.youtube.com', 'path': '/', 'secure': True,
                                  'expirationDate': int(time.time()) + 365 * 86400, 'name': n.strip(), 'value': v.strip()})
            return _json_cookies_to_netscape(items)

        def _normalize_cookies(text):
            if not isinstance(text, str) or len(text.encode('utf-8')) > 2 * 1024 * 1024:
                raise ValueError('Cookies devem ser texto com até 2 MB.')
            t = text.strip().lstrip('﻿')
            if not t:
                raise ValueError('Arquivo de cookies vazio.')
            if t.startswith('[') or t.startswith('{'):
                data = json.loads(t)
                if isinstance(data, dict):
                    data = data.get('cookies') or data.get('items') or list(data.values())
                return _json_cookies_to_netscape(data)
            if '\t' in t:
                if not t.startswith('#'):
                    t = '# Netscape HTTP Cookie File\n' + t
                return t + '\n'
            if '=' in t and ';' in t and '\n' not in t.strip():
                return _header_cookies_to_netscape(t)
            raise ValueError('Formato de cookies não reconhecido. Use um arquivo cookies.txt (formato Netscape), o JSON exportado por extensão ou o cabeçalho Cookie:.')

        def _cookie_summary():
            if not os.path.exists(COOKIES_PATH):
                return {'found': False}
            names, total = set(), 0
            with open(COOKIES_PATH, encoding='utf-8', errors='ignore') as f:
                for line in f:
                    if line.startswith('#HttpOnly_'): line = line[len('#HttpOnly_'):]
                    if line.startswith('#') or not line.strip(): continue
                    parts = line.rstrip('\n').split('\t')
                    if len(parts) >= 7 and _cookie_domain_ok(parts[0]):
                        total += 1
                        names.add(parts[5])
            logged = bool(names & {'SID', '__Secure-3PSID', 'SAPISID', '__Secure-3PAPISID', 'LOGIN_INFO'})
            src = 'drive' if os.path.exists(os.path.join(DRIVE_ROOT, 'cookies.txt')) else 'sessao'
            return {'found': True, 'total': total, 'logged_in': logged, 'source': src,
                    'updated': time.strftime('%d/%m/%Y %H:%M', time.localtime(os.path.getmtime(COOKIES_PATH)))}

        def _restore_cookies_from_drive():
            """Carrega cookies salvos no Drive (inclusive das versões anteriores: StemLab e YT Downloader)."""
            if os.path.exists(COOKIES_PATH) or not _drive_on():
                return None
            for root in [DRIVE_ROOT] + LEGACY_DRIVE_ROOTS:
                saved = os.path.join(root, 'cookies.txt')
                if os.path.exists(saved):
                    try:
                        content = _validate_cookie_content(open(saved, encoding='utf-8', errors='ignore').read())
                    except Exception:
                        continue
                    with open(COOKIES_PATH, 'w', encoding='utf-8') as f:
                        f.write(content)
                    os.chmod(COOKIES_PATH, 0o600)
                    if root != DRIVE_ROOT:   # migra para a pasta nova
                        try:
                            os.makedirs(DRIVE_ROOT, exist_ok=True)
                            shutil.copy(COOKIES_PATH, os.path.join(DRIVE_ROOT, 'cookies.txt'))
                        except Exception:
                            pass
                    return os.path.basename(root)
            return None

        def cb_save_cookies(text, persist_drive=True):
            try:
                content = _validate_cookie_content(_normalize_cookies(text))
                with open(COOKIES_PATH, 'w', encoding='utf-8') as f:
                    f.write(content)
                os.chmod(COOKIES_PATH, 0o600)
                saved_drive = False
                if persist_drive and _drive_on():
                    os.makedirs(DRIVE_ROOT, exist_ok=True)
                    shutil.copy(COOKIES_PATH, os.path.join(DRIVE_ROOT, 'cookies.txt'))
                    saved_drive = True
                s = _cookie_summary(); s['saved_drive'] = saved_drive
                return JSON({'ok': True, 'cookies': s})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        def cb_clear_cookies(also_drive=False):
            try:
                if os.path.exists(COOKIES_PATH): os.remove(COOKIES_PATH)
                if also_drive:
                    for root in [DRIVE_ROOT] + LEGACY_DRIVE_ROOTS:
                        p = os.path.join(root, 'cookies.txt')
                        if os.path.exists(p): os.remove(p)
                return JSON({'ok': True, 'cookies': _cookie_summary()})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_test_cookies():
            try:
                if not os.path.exists(COOKIES_PATH):
                    return JSON({'ok': False, 'error': 'Nenhum cookie carregado.'})
                opts = _base_opts(); opts.update({'extract_flat': True, 'playlistend': 1, 'ignoreerrors': True})
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info('https://www.youtube.com/feed/history', download=False)
                ok = bool(info and (info.get('entries') is not None))
                return JSON({'ok': True, 'valid': ok,
                             'message': 'Cookies válidos: sua conta foi reconhecida pelo YouTube.' if ok
                             else 'O YouTube não reconheceu a sessão. Exporte os cookies novamente com a conta logada.'})
            except Exception as e:
                return JSON({'ok': True, 'valid': False, 'message': _friendly_error(str(e))})

        # ---------- bootstrap ----------
        def _model_ready(model):
            return model in STATE.setdefault('ready_models', set())

        def cb_bootstrap():
            try:
                migrated = _restore_cookies_from_drive()
                return JSON({'ok': True, 'app': APP_NAME, 'version': APP_VERSION, 'user': STATE['user'], 'drive': _drive_on(),
                             'cookies': _cookie_summary(), 'cookies_migrated_from': migrated,
                             'ytdlp': yt_dlp.version.__version__, 'js_runtime': next(iter(_detect_js_runtime()), None),
                             'gpu': _gpu_info(), 'port': FILE_PORT, 'mock': MOCK, 'max_upload_mb': MAX_UPLOAD_MB,
                             'max_duration': MAX_DURATION, 'separator': SEP_AVAILABLE,
                             'modes': {k: {'label': v['label'], 'stems': v['stems'], 'factor': v['factor'], 'base': v['base'],
                                           'ready': _model_ready(v['model'])} for k, v in MODES.items()},
                             'stem_pt': STEM_PT, 'stem_icon': STEM_ICON, 'stem_color': STEM_COLOR})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- análise / prévia ----------
        def _res_label(h):
            names = {4320: '8K', 2160: '4K UHD', 1440: '2K QHD', 1080: 'Full HD', 720: 'HD', 480: 'SD', 360: 'Baixa', 240: 'Muito baixa', 144: 'Mínima'}
            return names.get(h, '')

        def _summarize_video(info):
            vids, auds, seen_aud = {}, [], set()
            duration = info.get('duration') or 0
            for f in info.get('formats') or []:
                vcodec, acodec = f.get('vcodec') or 'none', f.get('acodec') or 'none'
                size = f.get('filesize') or f.get('filesize_approx') or 0
                if not size and duration and f.get('tbr'):
                    size = int(f['tbr'] * 1000 / 8 * duration)  # estimativa por bitrate
                if vcodec != 'none' and f.get('height'):
                    h = int(f['height'])
                    cand = {
                        'height': h, 'fps': int(round(f.get('fps') or 0)), 'ext': f.get('ext'),
                        'vcodec': vcodec.split('.')[0], 'size': size, 'tbr': f.get('tbr') or 0,
                        'hdr': (f.get('dynamic_range') or 'SDR') != 'SDR', 'has_audio': acodec != 'none',
                    }
                    cur = vids.get(h)
                    if (cur is None or (cand['fps'], cand['tbr']) > (cur['fps'], cur['tbr'])):
                        vids[h] = cand
                elif acodec != 'none' and vcodec == 'none':
                    if 'drc' in (f.get('format_id') or '') or 'DRC' in (f.get('format_note') or ''):
                        continue
                    abr = int(round(f.get('abr') or f.get('tbr') or 0))
                    key = (f.get('ext'), acodec.split('.')[0], abr // 8, f.get('language'))
                    if key in seen_aud: continue
                    seen_aud.add(key)
                    auds.append({'format_id': f.get('format_id'), 'ext': f.get('ext'), 'acodec': acodec.split('.')[0],
                                 'abr': abr, 'size': size, 'lang': f.get('language'), 'note': f.get('format_note')})
            best_audio = max([a['size'] for a in auds] or [0])
            resolutions = []
            for h in sorted(vids, reverse=True):
                v = vids[h]
                est = (v['size'] + (0 if v['has_audio'] else best_audio)) if v['size'] else 0
                resolutions.append({**v, 'size_label': _fmt_size(est), 'size': est, 'label': _res_label(h)})
            auds.sort(key=lambda a: -a['abr'])
            return {
                'id': info.get('id'), 'title': info.get('title'), 'channel': info.get('uploader') or info.get('channel'),
                'channel_url': info.get('uploader_url') or info.get('channel_url'),
                'duration': duration, 'duration_str': _hms(duration),
                'views': info.get('view_count'), 'likes': info.get('like_count'),
                'upload_date': info.get('upload_date'), 'thumbnail': _thumb(info),
                'description': (info.get('description') or '')[:600],
                'is_live': bool(info.get('is_live') or info.get('live_status') in ('is_live', 'is_upcoming')),
                'age_limit': info.get('age_limit') or 0, 'availability': info.get('availability'),
                'url': info.get('webpage_url') or f"https://www.youtube.com/watch?v={info.get('id')}",
                'resolutions': resolutions,
                'audio_formats': [{**a, 'size_label': _fmt_size(a['size'])} for a in auds],
                'too_long': duration > MAX_DURATION,
            }

        def _register_youtube_source(v):
            """Registra o vídeo analisado como fonte para a separação de faixas."""
            src_id = uuid.uuid4().hex[:10]
            STATE['sources'][src_id] = {
                'id': src_id, 'kind': 'youtube', 'video_id': v['id'], 'title': v['title'], 'channel': v['channel'],
                'duration': v['duration'], 'duration_str': v['duration_str'], 'thumbnail': v['thumbnail'], 'url': v['url'],
                'wav': None, 'created': time.time(), 'too_long': v['too_long'],
            }
            return src_id

        def _summarize_entry(e, idx):
            vid = e.get('id')
            return {
                'index': idx, 'id': vid, 'title': e.get('title') or f'Vídeo {idx}',
                'url': e.get('url') if (e.get('url') or '').startswith('http') else f'https://www.youtube.com/watch?v={vid}',
                'duration': e.get('duration'), 'channel': e.get('uploader') or e.get('channel'),
                'views': e.get('view_count'),
                'thumbnail': _thumb(e) or f'https://i.ytimg.com/vi/{vid}/mqdefault.jpg',
                'unavailable': (e.get('title') in ('[Private video]', '[Deleted video]')) or e.get('availability') in ('private', 'premium_only', 'needs_auth', 'subscriber_only'),
            }

        def _extract_playlist(pl_url, is_mix=False):
            opts = _base_opts(); opts.update({'extract_flat': 'in_playlist', 'ignoreerrors': True, 'playlistend': MAX_PLAYLIST})
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(pl_url, download=False)
            mix_msg = ('Este link é um Mix automático do YouTube (list=RD…). O YouTube só gera Mixes dentro do player e não permite '
                       'listá-los por fora, então não dá para baixá-lo como playlist. Você pode baixar o vídeo, ou salvar os vídeos '
                       'do Mix em uma playlist sua no YouTube e colar o link dela aqui.')
            if not info:
                raise RuntimeError(mix_msg if is_mix else 'O YouTube não retornou a playlist (pode exigir login/cookies ou estar indisponível).')
            if info.get('_type') != 'playlist' and not info.get('entries'):
                raise RuntimeError(mix_msg if is_mix else 'Este link não retornou uma playlist.')
            entries = [e for e in (info.get('entries') or []) if e]
            items = [_summarize_entry(e, i + 1) for i, e in enumerate(entries)]
            if not items:
                raise RuntimeError('A playlist veio vazia. Se ela for privada ou "não listada", carregue os cookies da sua conta.')
            total_dur = sum((i['duration'] or 0) for i in items)
            return {
                'id': info.get('id'), 'title': info.get('title') or ('Mix do YouTube' if is_mix else 'Playlist'),
                'channel': info.get('uploader') or info.get('channel') or ('YouTube' if is_mix else None), 'url': pl_url,
                'count': len(items), 'total_duration': total_dur, 'is_mix': is_mix,
                'thumbnail': _thumb(info) or (items[0]['thumbnail'] if items else None),
                'description': (info.get('description') or '')[:400],
                'items': items, 'truncated': len(items) >= MAX_PLAYLIST,
            }

        def cb_analyze(url, mode='auto'):
            try:
                parsed = _parse_url(url, mode if mode in ('auto', 'video', 'playlist') else 'auto')
                result = {'ok': True, 'input': parsed['url'], 'mode': mode, 'video': None, 'playlist': None, 'playlist_error': None}
                if parsed['video_id']:
                    opts = _base_opts(); opts['noplaylist'] = True
                    with yt_dlp.YoutubeDL(opts) as ydl:
                        info = ydl.extract_info(f"https://www.youtube.com/watch?v={parsed['video_id']}", download=False)
                    if not info:
                        raise RuntimeError('video unavailable')
                    v = _summarize_video(info)
                    v['source_id'] = _register_youtube_source(v)
                    result['video'] = v
                if parsed['playlist_url']:
                    try:
                        result['playlist'] = _extract_playlist(parsed['playlist_url'], parsed['is_mix'])
                    except Exception as e:
                        if not result['video']:
                            raise
                        result['playlist_error'] = _friendly_error(str(e))   # o vídeo continua utilizável
                return JSON(result)
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        # ---------- upload de arquivo local (em pedaços base64) ----------
        def cb_upload_start(name, size):
            try:
                size = int(size or 0)
                if size <= 0:
                    raise ValueError('Arquivo vazio.')
                if size > MAX_UPLOAD_MB * 1024 * 1024:
                    raise ValueError(f'O arquivo tem {_fmt_size(size)}. O limite é {MAX_UPLOAD_MB} MB.')
                ext = os.path.splitext(name or '')[1].lower()
                if ext not in UPLOAD_EXTS:
                    raise ValueError('Formato não suportado. Envie MP3, WAV, FLAC, M4A, OGG, OPUS, AIFF ou um vídeo MP4/WEBM/MKV/MOV.')
                up_id = uuid.uuid4().hex[:10]
                path = os.path.join(WORK_DIR, f'upload_{up_id}{ext}')
                open(path, 'wb').close()
                STATE['uploads'][up_id] = {'path': path, 'name': name, 'size': size, 'received': 0}
                return JSON({'ok': True, 'upload_id': up_id})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_upload_chunk(up_id, data_b64):
            try:
                up = STATE['uploads'].get(up_id)
                if not up: raise ValueError('Envio não encontrado. Tente novamente.')
                if len(data_b64) > 4 * 1024 * 1024:
                    raise ValueError('Bloco de upload muito grande.')
                chunk = base64.b64decode(data_b64, validate=True)
                if up['received'] + len(chunk) > min(up['size'], MAX_UPLOAD_MB * 1024 * 1024):
                    raise ValueError('O bloco excede o tamanho declarado do arquivo.')
                with open(up['path'], 'ab') as f:
                    f.write(chunk)
                up['received'] += len(chunk)
                return JSON({'ok': True, 'received': up['received']})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_upload_finish(up_id):
            try:
                up = STATE['uploads'].pop(up_id, None)
                if not up: raise ValueError('Envio não encontrado. Tente novamente.')
                if up['received'] != up['size']:
                    os.remove(up['path'])
                    raise ValueError(f'Envio incompleto ({_fmt_size(up["received"])} de {_fmt_size(up["size"])}). Tente novamente.')
                dur = _ffprobe_duration(up['path'])
                if dur <= 0:
                    os.remove(up['path'])
                    raise ValueError('Não consegui ler este arquivo como áudio. Verifique se ele não está corrompido.')
                src_id = uuid.uuid4().hex[:10]
                title = os.path.splitext(os.path.basename(up['name']))[0]
                src = {'id': src_id, 'kind': 'file', 'title': title, 'channel': None, 'duration': dur, 'duration_str': _hms(dur),
                       'file_name': up['name'], 'file_size': up['size'], 'file_size_label': _fmt_size(up['size']),
                       'thumbnail': None, 'url': None, 'wav': None, 'src_path': up['path'],
                       'created': time.time(), 'too_long': dur > MAX_DURATION}
                STATE['sources'][src_id] = src
                return JSON({'ok': True, 'source': src})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- tarefas: estrutura comum ----------
        def _stage(job, key, label, status='running', percent=None):
            for s in job['stages']:
                if s['key'] == key:
                    s['status'] = status
                    if percent is not None: s['percent'] = percent
                    if status == 'running':
                        s['started'] = s.get('started') or time.time()
                    if status in ('done', 'error'):
                        s['ended'] = time.time()
                elif s['status'] == 'running' and status == 'running':
                    s['status'] = 'done'; s['percent'] = 100; s['ended'] = time.time()
            if job.get('stage') != key: job['detail'] = ''
            job['stage'] = key; job['stage_label'] = label
            if percent is not None: job['percent'] = percent
            if status == 'running': job['status'] = 'running'

        def _copy_to_drive(job, path, sub):
            if not job.get('save_drive'):
                return None
            try:
                if not _drive_on():
                    raise OSError('Google Drive desconectado')
                root = os.path.realpath(DRIVE_ROOT)
                d = os.path.realpath(os.path.join(root, sub))
                if os.path.commonpath((root, d)) != root:
                    raise ValueError('Pasta de destino inválida.')
                os.makedirs(d, exist_ok=True)
                dst = os.path.join(d, os.path.basename(path))
                # Reserva o nome sem sobrescrever arquivos de tarefas anteriores.
                base, ext = os.path.splitext(dst)
                index = 1
                while True:
                    try:
                        target = open(dst, 'xb')
                        break
                    except FileExistsError:
                        dst = f'{base} ({index}){ext}'
                        index += 1
                try:
                    with target, open(path, 'rb') as source:
                        shutil.copyfileobj(source, target)
                except Exception:
                    try: os.remove(dst)
                    except OSError: pass
                    raise
                return dst.replace(DRIVE_MYDRIVE, 'Meu Drive', 1)
            except Exception as e:
                warning = 'Arquivo disponível nesta sessão, mas a cópia no Drive falhou: ' + str(e)
                job['warning'] = warning
                job.setdefault('drive_warnings', []).append(warning)
                return None

        def _final_path(info, out_dir, before):
            rd = (info or {}).get('requested_downloads') or []
            if rd and rd[0].get('filepath') and os.path.exists(rd[0]['filepath']):
                return rd[0]['filepath']
            for k in ('filepath', '_filename', 'filename'):
                p = (info or {}).get(k)
                if p and os.path.exists(p): return p
            cands = [os.path.join(out_dir, f) for f in os.listdir(out_dir)
                     if f not in before and not f.endswith(('.part', '.ytdl', '.webp', '.jpg', '.png'))]
            return max(cands, key=os.path.getmtime) if cands else None

        # ---------- download de vídeo / áudio (um ou vários itens) ----------
        def _dl_progress(job, st):
            """Atualiza o progresso geral da tarefa a partir do item atual."""
            items = job['items']; total = len(items)
            finished = sum(1 for s in items if s['status'] in ('done', 'error', 'cancelled'))
            cur_pct = (st.get('percent') or 0) / 100 if st['status'] in ('downloading', 'processing') else 0
            job['percent'] = ((finished + cur_pct) / total * 100) if total else 0
            idx = items.index(st) + 1
            job['stage_label'] = (f'{idx}/{total} · ' if total > 1 else '') + (st.get('stage') or '')
            job['detail'] = st.get('title') or ''

        def _progress_hook(d, job, st):
            _check_cancel(job)
            if d['status'] == 'downloading':
                total = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
                done = d.get('downloaded_bytes') or 0
                info = d.get('info_dict') or {}
                stream = 'áudio' if (info.get('vcodec') in (None, 'none')) else 'vídeo'
                st.update({
                    'status': 'downloading', 'percent': (done / total * 100) if total else 0,
                    'downloaded': done, 'total': total, 'speed': d.get('speed') or 0, 'eta': d.get('eta'),
                    'stage': (f'Baixando {stream} completo para cortar…' if job.get('trim') else f'Baixando {stream}…'),
                })
            elif d['status'] == 'finished':
                st.update({'percent': 100, 'stage': 'Download concluído, processando…'})
            _dl_progress(job, st)

        def _pp_hook(d, job, st):
            _check_cancel(job)
            names = {'FFmpegExtractAudio': 'Convertendo áudio…', 'FFmpegVideoRemuxer': 'Empacotando MP4…',
                     'FFmpegMerger': 'Juntando vídeo e áudio…', 'FFmpegMetadata': 'Gravando metadados…',
                     'EmbedThumbnail': 'Inserindo capa…', 'MoveFiles': 'Finalizando…'}
            if d['status'] in ('started', 'processing'):
                st.update({'status': 'processing', 'stage': names.get(d.get('postprocessor'), 'Processando…')})
                _dl_progress(job, st)

        def _build_dl_opts(job, st, out_dir):
            mode, height = job['mode'], job['height']
            audio_format, audio_quality, embed_meta = job['audio_format'], job['audio_quality'], job['embed_meta']
            opts = _base_opts()
            opts.update({
                'outtmpl': os.path.join(out_dir, '%(title).150B [%(id)s].%(ext)s'),
                'noplaylist': True, 'overwrites': True, 'windowsfilenames': True, 'concurrent_fragment_downloads': 4,
                'progress_hooks': [lambda d: _progress_hook(d, job, st)],
                'postprocessor_hooks': [lambda d: _pp_hook(d, job, st)],
                'postprocessors': [],
            })
            if job.get('trim'):
                # Recorte por tempo: o arquivo completo é baixado (rápido, com progresso) e cortado
                # depois com ffmpeg em _cut_clip. Capa/metadados são aplicados no corte, não aqui.
                opts['outtmpl'] = os.path.join(out_dir, '_completo_%(id)s.%(ext)s')
                embed_meta = False
            if mode == 'audio':
                opts['format'] = 'ba/b'
                if audio_format != 'best' and not job.get('trim'):
                    pp = {'key': 'FFmpegExtractAudio', 'preferredcodec': audio_format}
                    if audio_format not in ('wav', 'flac') and audio_quality and audio_quality != 'best':
                        pp['preferredquality'] = str(audio_quality)
                    elif audio_format == 'mp3':
                        pp['preferredquality'] = '0'  # melhor VBR
                    opts['postprocessors'].append(pp)
                if embed_meta:
                    opts['postprocessors'].append({'key': 'FFmpegMetadata', 'add_metadata': True})
                    if audio_format in ('mp3', 'm4a', 'flac', 'opus', 'aac'):
                        opts['writethumbnail'] = True
                        opts['postprocessors'].append({'key': 'EmbedThumbnail', 'already_have_thumbnail': False})
            else:
                sort = [f'res:{height}' if height and height != 'best' else 'res', 'fps', 'codec:avc1:m4a', 'ext:mp4:m4a']
                opts['format'] = 'bv*+ba/b'
                opts['format_sort'] = sort
                opts['merge_output_format'] = 'mp4'
                opts['postprocessors'].append({'key': 'FFmpegVideoRemuxer', 'preferedformat': 'mp4'})
                if embed_meta:
                    opts['postprocessors'].append({'key': 'FFmpegMetadata', 'add_metadata': True})
            return opts

        def _cut_clip(job, st, src_path, info, out_dir):
            """Corta [start, end] do arquivo completo com ffmpeg, mostrando progresso.
            Vídeo: H.264. Áudio: recodificação para corte preciso; original mantém pacotes."""
            trim = job['trim']; start, end = float(trim['start']), float(trim['end'])
            dur_total = float((info or {}).get('duration') or 0) or _ffprobe_duration(src_path)
            if dur_total and start >= dur_total:
                raise RuntimeError(f'O início do trecho ({_hms(start)}) passa da duração do vídeo ({_hms(dur_total)}).')
            if dur_total:
                end = min(end, dur_total)
            length = end - start
            if length < 0.5:
                raise RuntimeError('O trecho ficou curto demais depois de ajustar ao tamanho do vídeo.')
            title = _safe_name((info or {}).get('title') or st.get('title') or 'video', 140)
            ext = ('mp4' if job['mode'] == 'video' else job['audio_format'])
            if ext == 'best':
                ext = os.path.splitext(src_path)[1].lstrip('.') or 'm4a'
            tag = f"(trecho {_hms(start).replace(':', '.')}-{_hms(end).replace(':', '.')})"
            dst = os.path.join(out_dir, f"{title} [{(info or {}).get('id') or st.get('id')}] {tag}.{ext}")
            meta = ['-metadata', f"title={(info or {}).get('title') or ''} {tag}",
                    '-metadata', f"artist={(info or {}).get('uploader') or (info or {}).get('channel') or ''}",
                    '-metadata', f"comment={(info or {}).get('webpage_url') or ''}"]
            if job['mode'] == 'video':
                codec = ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart']
                mapping = ['-map', '0:v:0', '-map', '0:a:0?']
            else:
                fmt = job['audio_format']
                quality = job['audio_quality']
                bitrate = quality if quality != 'best' else '320'
                codec = {
                    'mp3': ['-c:a', 'libmp3lame'] + (['-q:a', '0'] if quality == 'best' else ['-b:a', bitrate + 'k']),
                    'm4a': ['-c:a', 'aac', '-b:a', bitrate + 'k'],
                    'opus': ['-c:a', 'libopus', '-b:a', bitrate + 'k'],
                    'wav': ['-c:a', 'pcm_s24le'], 'flac': ['-c:a', 'flac'],
                    'best': ['-c:a', 'copy'],
                }[fmt]
                mapping = ['-map', '0:a:0', '-vn']
                if fmt == 'best':
                    job['warning'] = 'No formato original, o recorte segue os pacotes de áudio e pode variar alguns milissegundos. Para corte preciso, escolha outro formato.'
            if not job.get('embed_meta', True):
                meta = []
            progress = os.path.join(out_dir, f'cut_{st.get("id") or "x"}.progress')
            cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-nostats', '-progress', progress,
                   '-ss', f'{start:.3f}', '-i', src_path, '-t', f'{length:.3f}'] + mapping + ['-map_metadata', '-1'] + meta + codec + [dst]
            st.update({'status': 'processing', 'stage': 'Cortando trecho…' + (' (recodificando vídeo)' if job['mode'] == 'video' else ''), 'percent': 0})
            _dl_progress(job, st)
            def on_progress(pct):
                st['percent'] = max(st.get('percent') or 0, pct); _dl_progress(job, st)
            try:
                _run_process(job, cmd, timeout=3600, progress_file=progress, progress_total=length, on_progress=on_progress)
            finally:
                if os.path.exists(progress): os.remove(progress)
            if not os.path.exists(dst) or os.path.getsize(dst) == 0:
                raise RuntimeError('Falha ao cortar o trecho: o ffmpeg não gerou o arquivo.')
            try: os.remove(src_path)
            except OSError: pass
            st['percent'] = 100
            return dst

        def _run_download_job(job):
            out_dir = os.path.join(OUT_DIR, job['id']); os.makedirs(out_dir, exist_ok=True)
            job['status'] = 'running'
            is_video = job['mode'] == 'video'
            for st in job['items']:
                if job.get('cancel'):
                    st.update({'status': 'cancelled', 'stage': 'Cancelado'}); continue
                st.update({'status': 'downloading', 'stage': 'Preparando…', 'percent': 0})
                _dl_progress(job, st)
                try:
                    opts = _build_dl_opts(job, st, out_dir)
                    logger = _SilentLogger(); opts['logger'] = logger
                    opts['ignoreerrors'] = False  # Nunca tratar uma conversão incompleta como arquivo final.
                    before = set(os.listdir(out_dir))
                    with yt_dlp.YoutubeDL(opts) as ydl:
                        info = ydl.extract_info(st['url'], download=True)
                    _check_cancel(job)
                    path = _final_path(info, out_dir, before)
                    if not path:
                        raise RuntimeError(logger.errors[-1] if logger.errors else 'O arquivo final não foi encontrado após o download.')
                    if logger.errors:
                        st['warning'] = _friendly_error(logger.errors[-1])
                    if job.get('trim'):
                        path = _cut_clip(job, st, path, info, out_dir)
                    preview = os.path.join(out_dir, st['id'] + ('.preview.mp4' if is_video else '.preview.mp3'))
                    st.update({'status': 'processing', 'stage': 'Preparando prévia…'})
                    _dl_progress(job, st)
                    preview_ok = _make_video_preview(path, preview, job) if is_video else _make_preview(path, preview, job)
                    size = os.path.getsize(path)
                    drive_path = _copy_to_drive(job, path, os.path.join('Downloads', job.get('folder') or ''))
                    st.update({'status': 'done', 'percent': 100, 'stage': 'Concluído',
                               'file': os.path.basename(path), 'rel': f"{job['id']}/{os.path.basename(path)}",
                               'size': size, 'size_label': _fmt_size(size), 'drive_path': drive_path,
                               'preview': f"{job['id']}/{os.path.basename(preview)}" if preview_ok else None,
                               'preview_kind': 'video' if is_video else 'audio',
                               'title': (info or {}).get('title') or st.get('title')})
                except yt_dlp.utils.DownloadCancelled:
                    st.update({'status': 'cancelled', 'stage': 'Cancelado'})
                except Exception as e:
                    st.update({'status': 'error', 'stage': 'Erro', 'error': _friendly_error(str(e))})
                _dl_progress(job, st)
            done = [s for s in job['items'] if s['status'] == 'done']
            errs = [s for s in job['items'] if s['status'] == 'error']
            job['files'] = [{'stem': 'Video' if is_video else 'Audio', 'label': s.get('title') or s['file'],
                             'icon': '🎬' if is_video else '🎵', 'color': '#ff5c7a' if is_video else '#38bdf8',
                             'file': s['file'], 'rel': s['rel'], 'size': s['size'], 'size_label': s['size_label'],
                             'preview': s.get('preview'), 'preview_kind': s.get('preview_kind'), 'drive_path': s.get('drive_path'), 'warning': s.get('warning')} for s in done]
            job['done_items'], job['error_items'] = len(done), len(errs)
            if errs and done:
                job['warning'] = f"{len(errs)} de {len(job['items'])} {'item falhou' if len(errs) == 1 else 'itens falharam'}. Use \"Tentar novamente\" para repetir só os que deram erro."
            if job.get('cancel'):
                raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')
            if errs and not done:
                raise RuntimeError(errs[0].get('error') or 'Nenhum arquivo foi baixado.')
            job['stage_label'] = 'Concluído'; job['detail'] = ''

        def _parse_trim(trim, items):
            """Valida o recorte {start, end} em segundos. Só para um vídeo por vez."""
            if not trim:
                return None
            if len(items) != 1:
                raise ValueError('O recorte por tempo só funciona com um vídeo por vez.')
            try:
                start = float(trim.get('start') or 0)
                end = float(trim.get('end'))
            except Exception:
                raise ValueError('Tempos do recorte inválidos. Use o formato mm:ss ou hh:mm:ss.')
            if not math.isfinite(start) or not math.isfinite(end):
                raise ValueError('Tempos do recorte inválidos.')
            start = max(0.0, start)
            if end <= start:
                raise ValueError('O fim do trecho precisa ser maior que o início.')
            if end - start < 1:
                raise ValueError('O trecho precisa ter pelo menos 1 segundo.')
            return {'start': start, 'end': end}

        def _new_download_job(payload):
            items = payload.get('items') or []
            if not isinstance(items, list) or not items:
                raise ValueError('Nenhum vídeo selecionado.')
            if len(items) > MAX_PLAYLIST:
                raise ValueError(f'No máximo {MAX_PLAYLIST} vídeos por tarefa.')
            mode = payload.get('mode', 'video')
            if mode not in ('video', 'audio'):
                raise ValueError('Tipo de download inválido.')
            height = payload.get('height', 'best')
            if height != 'best':
                try: height = int(height)
                except Exception: raise ValueError('Resolução inválida.')
                if not 100 <= height <= 8640: raise ValueError('Resolução inválida.')
            audio_format = payload.get('audio_format', 'mp3')
            if audio_format not in AUDIO_FORMATS:
                raise ValueError('Formato de áudio inválido.')
            audio_quality = str(payload.get('audio_quality', 'best'))
            if audio_quality not in AUDIO_QUALITIES:
                raise ValueError('Qualidade de áudio inválida.')
            clean_items = []
            seen = set()
            for it in items:
                vid = str(it.get('id') or '')
                url = str(it.get('url') or '')
                if not re.fullmatch(r'[A-Za-z0-9_-]{11}', vid):
                    raise ValueError('Identificador de vídeo inválido na seleção.')
                if not url.startswith('http'):
                    url = f'https://www.youtube.com/watch?v={vid}'
                parsed = _parse_url(url, 'video')
                if parsed.get('video_id') != vid:
                    raise ValueError('O link não corresponde ao identificador do vídeo.')
                if vid in seen:
                    continue
                seen.add(vid)
                url = f'https://www.youtube.com/watch?v={vid}'
                clean_items.append({'id': vid, 'url': url, 'title': str(it.get('title') or vid)[:200],
                                    'thumbnail': str(it.get('thumbnail') or '')[:500] or f'https://i.ytimg.com/vi/{vid}/mqdefault.jpg',
                                    'status': 'queued', 'percent': 0, 'stage': 'Na fila'})
            trim = _parse_trim(payload.get('trim'), clean_items)
            folder = re.sub(r'[^\w\s.-]', '', str(payload.get('folder') or ''))[:80].strip().strip('.')
            first = clean_items[0]
            n = len(clean_items)
            if mode == 'video':
                qual = 'melhor resolução' if height == 'best' else f'{height}p'
            else:
                qual = 'áudio original' if audio_format == 'best' else audio_format.upper() + ('' if audio_quality == 'best' or audio_format in ('wav', 'flac') else f' {audio_quality} kbps')
            label = (f'{n} vídeos · ' if n > 1 else '') + ('Vídeo MP4 · ' if mode == 'video' else 'Áudio · ') + qual
            if trim: label += f" · trecho {_hms(trim['start'])}–{_hms(trim['end'])}"
            return {
                'kind': 'download', 'mode': mode, 'height': height, 'audio_format': audio_format, 'audio_quality': audio_quality,
                'embed_meta': bool(payload.get('embed_meta', True)), 'folder': folder, 'trim': trim,
                'title': str(payload.get('title') or (first['title'] if n == 1 else f"{first['title']} e mais {n - 1}"))[:200],
                'thumbnail': first['thumbnail'], 'label': label, 'items': clean_items, 'duration': 0,
                'stages': [{'key': 'fetch', 'label': 'Baixar do YouTube', 'status': 'pending', 'percent': 0},
                           {'key': 'export', 'label': 'Converter e finalizar', 'status': 'pending', 'percent': 0}],
            }

        # ---------- separação de faixas (IA) ----------
        def _download_youtube_audio(job, src, out_dir, tag='src'):
            """Baixa o áudio original (m4a/webm) de um vídeo, sem conversão."""
            opts = _base_opts()
            logger = _SilentLogger(); opts['logger'] = logger
            def hook(d):
                _check_cancel(job)
                if d['status'] == 'downloading':
                    total = d.get('total_bytes') or d.get('total_bytes_estimate') or 0
                    done = d.get('downloaded_bytes') or 0
                    job['percent'] = (done / total * 100) if total else 0
                    job['detail'] = f"{_fmt_size(done) or '0 B'}{' de ' + _fmt_size(total) if total else ''}" + (f" · {_fmt_size(d['speed'])}/s" if d.get('speed') else '')
                elif d['status'] == 'finished':
                    job['percent'] = 100; job['detail'] = ''
            opts.update({
                'outtmpl': os.path.join(out_dir, f'{tag}_%(id)s.%(ext)s'),
                'noplaylist': True, 'overwrites': True, 'windowsfilenames': True, 'concurrent_fragment_downloads': 4,
                'format': 'ba/b', 'progress_hooks': [hook], 'postprocessors': [], 'ignoreerrors': False,
            })
            before = set(os.listdir(out_dir))
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(src['url'], download=True)
            _check_cancel(job)
            path = _final_path(info, out_dir, before)
            if not path:
                raise RuntimeError(logger.errors[-1] if logger.errors else 'O arquivo não foi encontrado após o download.')
            if logger.errors:
                job['warning'] = _friendly_error(logger.errors[-1])
            return path

        def _ensure_source_wav(job, src):
            """Garante um WAV 44.1 kHz estéreo da fonte (baixando do YouTube se preciso). Fica em cache por fonte."""
            if src.get('wav') and os.path.exists(src['wav']):
                _stage(job, 'fetch', 'Áudio já disponível', 'done', 100)
                return src['wav']
            src_dir = os.path.join(WORK_DIR, src['id']); os.makedirs(src_dir, exist_ok=True)
            if src['kind'] == 'youtube':
                _stage(job, 'fetch', 'Baixando áudio do YouTube…', 'running', 0)
                raw = _download_youtube_audio(job, src, src_dir, tag='src')
            else:
                raw = src['src_path']
            _stage(job, 'fetch', 'Preparando o áudio…', 'running', 100)
            job['detail'] = 'Convertendo para WAV 44.1 kHz'
            wav = os.path.join(src_dir, 'source.wav')
            _run_process(job, ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', raw,
                               '-vn', '-ac', '2', '-ar', '44100', '-c:a', 'pcm_s24le', wav])
            src['wav'] = wav
            if not src.get('duration'):
                src['duration'] = _ffprobe_duration(wav); src['duration_str'] = _hms(src['duration'])
            job['detail'] = ''
            return wav

        def _sep_cmd():
            exe = shutil.which('audio-separator')
            return [exe] if exe else [sys.executable, '-m', 'audio_separator.utils.cli']

        def _iter_output(stream, job, proc, timeout=7200):
            lines = queue.Queue()
            def read():
                buf = ''
                try:
                    while True:
                        ch = stream.read(1)
                        if not ch: break
                        if ch in ('\r', '\n'):
                            if buf.strip(): lines.put(buf)
                            buf = ''
                        else:
                            buf = (buf + ch)[-4000:]
                    if buf: lines.put(buf)
                finally:
                    lines.put(None)
            threading.Thread(target=read, daemon=True).start()
            started = time.monotonic()
            while True:
                _check_cancel(job)
                if time.monotonic() - started > timeout:
                    raise TimeoutError('A separação excedeu duas horas. Tente um áudio mais curto.')
                try: line = lines.get(timeout=0.2)
                except queue.Empty: continue
                if line is None: return
                yield line

        def _run_separator(job, wav, mode, out_format, out_dir, names):
            cfg = MODES[mode]
            cmd = _sep_cmd() + [wav, '--model_filename', cfg['model'],
                   '--model_file_dir', MODELS_DIR, '--output_dir', out_dir, '--output_format', OUT_FORMATS[out_format],
                   '--custom_output_names', json.dumps(names), '--log_level', 'info']
            if out_format == 'mp3':
                cmd += ['--output_bitrate', '320k']
            if cfg['model'].endswith('.yaml'):
                cmd += ['--demucs_shifts', '1']
            if _gpu_info()['available']:
                cmd += ['--use_autocast']   # precisão mista na GPU: bem mais rápido, qualidade praticamente igual
            env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8', TQDM_MININTERVAL='0.5')
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='ignore', env=env, bufsize=0)
            job['proc'] = proc
            tail, err_hint = [], None
            started = time.time(); phase_started = None
            est = cfg['base'] + (job.get('duration') or 180) * cfg['factor']
            onde = 'na GPU' if (_gpu_info()['available']) else 'no processador'
            if _model_ready(cfg['model']):
                _stage(job, 'model', f'Carregando o modelo {onde}…', 'running', 0)
            else:
                _stage(job, 'model', 'Baixando o modelo de IA (uma vez por sessão)…', 'running', 0)
            try:
                for line in _iter_output(proc.stdout, job, proc):
                    if job.get('cancel'):
                        proc.kill(); raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')
                    tail.append(line[:300]); tail[:] = tail[-40:]
                    low = line.lower()
                    m = re.search(r'(\d{1,3})%\|', line)  # barras do tqdm (download do modelo / separação)
                    if 'starting separation' in low:
                        _stage(job, 'separate', f'Separando as faixas {onde}…', 'running', 0); phase_started = time.time(); job['detail'] = ''
                    elif 'loading model' in low and job['stage'] == 'model':
                        job['detail'] = 'Carregando pesos…'
                    elif ' saving ' in low or 'writing output' in low:
                        if job['stage'] == 'separate': job['detail'] = 'Gravando arquivos…'
                    elif m and job['stage'] == 'model':
                        job['percent'] = int(m.group(1)); job['detail'] = f'Baixando modelo… {m.group(1)}%'
                    elif m and job['stage'] == 'separate':
                        job['percent'] = max(job['percent'], min(99, int(m.group(1)))); job['detail'] = f'{m.group(1)}%'
                    if ('error' in low or 'traceback' in low) and 'errorlevel' not in low:
                        err_hint = line
                    if job['stage'] == 'separate' and phase_started and not m:
                        job['percent'] = max(job['percent'], min(95, (time.time() - phase_started) / max(est, 1) * 100))
                proc.wait()
            finally:
                if proc.poll() is None: proc.kill()
                proc.wait()
                job['proc'] = None
                try: proc.stdout.close()
                except Exception: pass
            if job.get('cancel'):
                raise yt_dlp.utils.DownloadCancelled('Cancelado pelo usuário')
            if proc.returncode != 0:
                msg = err_hint or (tail[-1] if tail else f'o separador retornou código {proc.returncode}')
                job['log_tail'] = tail[-15:]
                raise RuntimeError(_friendly_error(msg))
            job['sep_seconds'] = round(time.time() - started)

        def _run_separator_mock(job, wav, mode, out_format, out_dir, names):
            """Modo de teste local: gera 'faixas' com filtros do ffmpeg, sem IA."""
            filters = {'Vocals': 'highpass=f=1200', 'Instrumental': 'lowpass=f=1200', 'Drums': 'lowpass=f=200',
                       'Bass': 'lowpass=f=120', 'Guitar': 'bandpass=f=800:w=400', 'Piano': 'bandpass=f=1500:w=600', 'Other': 'highpass=f=3000'}
            _stage(job, 'model', 'Carregando o modelo (simulação)…', 'running', 0); time.sleep(1.0)
            _stage(job, 'separate', 'Separando as faixas (simulação)…', 'running', 0)
            stems = MODES[mode]['stems']
            for i, stem in enumerate(stems):
                _check_cancel(job)
                ext = out_format
                dst = os.path.join(out_dir, f'{names[stem]}.{ext}')
                codec = ['-c:a', 'libmp3lame', '-b:a', '320k'] if ext == 'mp3' else (['-c:a', 'flac'] if ext == 'flac' else ['-c:a', 'pcm_s16le'])
                subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', wav, '-af', filters[stem]] + codec + [dst], check=True)
                job['percent'] = (i + 1) / len(stems) * 100; time.sleep(0.4)
            job['sep_seconds'] = 3

        def _make_video_preview(src_path, dst_path, job):
            """Prévia completa H.264/AAC em até 480p, sem alterar o vídeo para download."""
            try:
                _run_process(job, ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', src_path,
                                  '-map', '0:v:0', '-map', '0:a:0?', '-vf',
                                  "scale=w='min(854,iw)':h='min(480,ih)':force_original_aspect_ratio=decrease:force_divisible_by=2,setsar=1",
                                  '-r', '24', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '30',
                                  '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '96k',
                                  '-ac', '2', '-movflags', '+faststart', dst_path], timeout=3600)
                return os.path.isfile(dst_path) and os.path.getsize(dst_path) > 0
            except yt_dlp.utils.DownloadCancelled:
                raise
            except Exception:
                if os.path.exists(dst_path): os.remove(dst_path)
                job['warning'] = 'O vídeo foi gerado, mas sua prévia não pôde ser criada. Use Baixar para abrir o arquivo.'
                return False

        def _make_preview(src_path, dst_path, job):
            try:
                _run_process(job, ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', src_path,
                                  '-vn', '-ac', '2', '-ar', '44100', '-c:a', 'libmp3lame', '-b:a', '96k', dst_path])
                return os.path.isfile(dst_path) and os.path.getsize(dst_path) > 0
            except yt_dlp.utils.DownloadCancelled:
                raise
            except Exception:
                if os.path.exists(dst_path): os.remove(dst_path)
                job['warning'] = 'Os arquivos foram gerados, mas uma prévia não pôde ser criada. Baixe as faixas para ouvir.'
                return False

        def _run_separate_job(job):
            src = STATE['sources'][job['source_id']]
            job['duration'] = src.get('duration') or 0
            out_dir = os.path.join(OUT_DIR, job['id']); os.makedirs(out_dir, exist_ok=True)
            wav = _ensure_source_wav(job, src)
            if not job['duration']:
                job['duration'] = _ffprobe_duration(wav)
            if job['duration'] > MAX_DURATION:
                raise RuntimeError(f'Este áudio tem {_hms(job["duration"])}. O limite é {MAX_DURATION // 60} minutos para não estourar a memória da GPU.')
            mode, fmt = job['mode'], job['out_format']
            base = _safe_name(src['title'])
            names = {stem: f'stemtube_{stem.lower()}' for stem in MODES[mode]['stems']}   # nomes internos simples
            _stage(job, 'model', 'Aguardando disponibilidade do modelo…', 'running', 0)
            while not MODEL_LOCK.acquire(timeout=0.2):
                _check_cancel(job)
            try:
                _check_cancel(job)
                (_run_separator_mock if MOCK else _run_separator)(job, wav, mode, fmt, out_dir, names)
                if not MOCK: STATE.setdefault('ready_models', set()).add(MODES[mode]['model'])
            finally:
                MODEL_LOCK.release()
            _stage(job, 'export', 'Preparando prévias e arquivos…', 'running', 0)
            files = []
            stems = MODES[mode]['stems']
            for i, stem in enumerate(stems):
                _check_cancel(job)
                cand = [f for f in os.listdir(out_dir) if f.lower().startswith(names[stem]) and not f.endswith('.preview.mp3')]
                if not cand:
                    raise RuntimeError(f'A faixa "{STEM_PT[stem]}" não foi gerada pelo separador.' + (' Detalhes: ' + ' | '.join(job.get('log_tail') or [])[-300:] if job.get('log_tail') else ''))
                path = os.path.join(out_dir, f'{base} - {STEM_PT[stem]}{os.path.splitext(cand[0])[1]}')
                os.replace(os.path.join(out_dir, cand[0]), path)
                prev = os.path.join(out_dir, f'{stem.lower()}.preview.mp3')
                _make_preview(path, prev, job)
                size = os.path.getsize(path)
                files.append({'stem': stem, 'label': STEM_PT[stem], 'icon': STEM_ICON[stem], 'color': STEM_COLOR[stem],
                              'file': os.path.basename(path), 'rel': f"{job['id']}/{os.path.basename(path)}",
                              'preview': f"{job['id']}/{os.path.basename(prev)}" if os.path.exists(prev) else None,
                              'size': size, 'size_label': _fmt_size(size),
                              'drive_path': _copy_to_drive(job, path, os.path.join('Faixas', base, job['id']))})
                job['percent'] = (i + 1) / len(stems) * 100
            orig_prev = os.path.join(out_dir, 'original.preview.mp3')   # prévia do original para comparação no mixer
            if _make_preview(wav, orig_prev, job):
                job['original_preview'] = f"{job['id']}/original.preview.mp3"
            job['files'] = files
            _stage(job, 'export', 'Concluído', 'done', 100)

        def _new_separate_job(payload):
            src = STATE['sources'].get(payload.get('source_id'))
            if not src:
                raise ValueError('Fonte não encontrada. Analise o link ou envie o arquivo novamente.')
            if not SEP_AVAILABLE:
                raise ValueError('O separador de faixas não está instalado. Rode a célula 1 com a opção "instalar_separador_ia" marcada.')
            mode = payload.get('mode'); fmt = payload.get('out_format', 'wav')
            if mode not in MODES: raise ValueError('Modo de separação inválido.')
            if fmt not in OUT_FORMATS: raise ValueError('Formato de saída inválido.')
            if src.get('too_long'):
                raise ValueError(f'Este áudio tem {src["duration_str"]}. O limite para separação é {MAX_DURATION // 60} minutos.')
            job = {'kind': 'separate', 'source_id': src['id'], 'title': src['title'], 'thumbnail': src.get('thumbnail'),
                   'duration': src.get('duration') or 0, 'mode': mode, 'mode_label': MODES[mode]['label'], 'out_format': fmt,
                   'stems': MODES[mode]['stems'], 'label': f"{MODES[mode]['label']} · {fmt.upper()}",
                   'stages': [{'key': 'fetch', 'label': 'Obter o áudio', 'status': 'pending', 'percent': 0},
                              {'key': 'model', 'label': 'Modelo de IA', 'status': 'pending', 'percent': 0},
                              {'key': 'separate', 'label': 'Separar na GPU', 'status': 'pending', 'percent': 0},
                              {'key': 'export', 'label': 'Exportar faixas', 'status': 'pending', 'percent': 0}]}
            if not MOCK and not _gpu_info()['available']:
                job['warning'] = 'Sem GPU nesta sessão: a separação vai rodar no processador e pode demorar muitos minutos.'
            return job

        # ---------- mixagem ----------
        def cb_mix_job(parent_id, tracks, out_format='wav', save_drive=False):
            """Cria uma mixagem independente sem alterar as faixas da separação."""
            try:
                parent = STATE['jobs'].get(parent_id)
                if not parent or parent['status'] != 'done' or parent['kind'] != 'separate':
                    raise ValueError('Conclua uma separação antes de renderizar a mixagem.')
                if out_format not in OUT_FORMATS:
                    raise ValueError('Escolha WAV, FLAC ou MP3.')
                if not isinstance(tracks, list) or not 1 <= len(tracks) <= len(parent['files']):
                    raise ValueError('Selecione ao menos uma faixa audível para renderizar.')
                available = {f['stem']: f for f in parent['files']}
                selected, seen = [], set()
                for item in tracks:
                    stem = item.get('stem'); gain = float(item.get('gain', 1))
                    if stem not in available or stem in seen or not math.isfinite(gain) or not 0 < gain <= 1:
                        raise ValueError('Seleção de faixas ou volume inválido.')
                    seen.add(stem)
                    f = available[stem]
                    _output_path(f['rel'])
                    selected.append({'stem': stem, 'gain': gain, 'rel': f['rel'], 'label': f['label']})
                label = ' + '.join(f['label'] for f in selected)
                job = _enqueue({'kind': 'mix', 'parent_id': parent_id, 'source_id': parent['source_id'],
                                'title': parent['title'], 'thumbnail': parent.get('thumbnail'), 'duration': parent.get('duration') or 0,
                                'save_drive': bool(save_drive), 'out_format': out_format, 'mix_tracks': selected,
                                'label': f'Mixagem: {label} · {out_format.upper()}',
                                'stages': [{'key': 'export', 'label': 'Renderizar mixagem', 'status': 'pending', 'percent': 0}]})
                return JSON({'ok': True, 'job_id': job['id'], 'ahead': job['ahead']})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        def _run_mix_job(job):
            out_dir = os.path.join(OUT_DIR, job['id']); os.makedirs(out_dir, exist_ok=True)
            fmt = job['out_format']
            filename = _safe_name(job['title'], 65) + ' - Mixagem.' + fmt
            path = os.path.join(out_dir, filename)
            cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error']
            filters, labels = [], []
            for i, track in enumerate(job['mix_tracks']):
                cmd += ['-i', _output_path(track['rel'])]
                filters.append(f'[{i}:a]aresample=44100,aformat=channel_layouts=stereo,asetpts=PTS-STARTPTS,volume={track["gain"]:.8f}[s{i}]')
                labels.append(f'[s{i}]')
            # Em 44100 Hz, padding/corte de 219 amostras compensam attack=5 sem exigir latency.
            filters.append(''.join(labels) + f'amix=inputs={len(labels)}:duration=longest:dropout_transition=0:normalize=0,apad=pad_len=219,alimiter=limit=0.98:level=false:attack=5,atrim=start_sample=219,asetpts=PTS-STARTPTS[mix]')
            codec = {'wav': ['-c:a', 'pcm_s24le'], 'flac': ['-c:a', 'flac'], 'mp3': ['-c:a', 'libmp3lame', '-b:a', '320k']}[fmt]
            progress = os.path.join(out_dir, 'render.progress')
            cmd += ['-filter_complex', ';'.join(filters), '-map', '[mix]', '-vn', '-ar', '44100'] + codec
            cmd += ['-progress', progress, '-nostats', path]
            _stage(job, 'export', 'Renderizando mixagem…', 'running', 0)
            job['detail'] = ' + '.join(t['label'] for t in job['mix_tracks'])
            try:
                _run_process(job, cmd, progress_file=progress)
                if not os.path.isfile(path) or os.path.getsize(path) == 0:
                    raise RuntimeError('A mixagem não gerou um arquivo de áudio válido.')
                _check_cancel(job)
                drive = _copy_to_drive(job, path, os.path.join('Mixagens', job['id']))
                size = os.path.getsize(path)
                prev = os.path.join(out_dir, 'mix.preview.mp3')
                job['files'] = [{'stem': 'Mix', 'label': 'Mixagem', 'icon': '🎛', 'color': '#34d399',
                                 'file': filename, 'rel': f"{job['id']}/{filename}", 'size': size, 'size_label': _fmt_size(size),
                                 'preview': f"{job['id']}/mix.preview.mp3" if _make_preview(path, prev, job) else None, 'drive_path': drive}]
                _stage(job, 'export', 'Mixagem concluída', 'done', 100)
            except BaseException:
                if os.path.exists(path): os.remove(path)
                raise
            finally:
                if os.path.exists(progress): os.remove(progress)

        # ---------- fila: duas pistas (downloads e processamento na GPU rodam em paralelo) ----------
        LANES = {'download': 'download', 'separate': 'process', 'mix': 'process'}
        JOB_QUEUES = STATE.setdefault('queues', {'download': queue.Queue(), 'process': queue.Queue()})

        def _run_job(job_id):
            job = STATE['jobs'][job_id]
            job['started'] = time.time()
            try:
                {'separate': _run_separate_job, 'mix': _run_mix_job, 'download': _run_download_job}[job['kind']](job)
                _check_cancel(job)
                job['status'] = 'done'; job['percent'] = 100
                for s in job['stages']:
                    if s['status'] in ('pending', 'running'): s['status'] = 'done'; s['percent'] = 100
            except yt_dlp.utils.DownloadCancelled:
                job['status'] = 'cancelled'; job['stage_label'] = 'Cancelado'
                for s in job['stages']:
                    if s['status'] == 'running': s['status'] = 'cancelled'
            except Exception as e:
                job['status'] = 'error'; job['error'] = _friendly_error(str(e)); job['stage_label'] = 'Erro'
                for s in job['stages']:
                    if s['status'] == 'running': s['status'] = 'error'
            job['finished'] = time.time()
            job['elapsed'] = round(job['finished'] - job['started'])

        def _worker(lane):
            q = JOB_QUEUES[lane]
            while True:
                job_id = q.get()
                try:
                    job = STATE['jobs'].get(job_id)
                    if job and job.get('cancel'):
                        job['status'] = 'cancelled'; job['stage_label'] = 'Cancelado'; job['finished'] = time.time()
                        for it in job.get('items') or []: it.update({'status': 'cancelled', 'stage': 'Cancelado'})
                    elif job:
                        _run_job(job_id)
                except Exception as e:
                    job = STATE['jobs'].get(job_id)
                    if job:
                        job['status'] = 'error'; job['error'] = _friendly_error(str(e)); job['finished'] = time.time()
                finally:
                    q.task_done()

        def _ensure_workers():
            workers = STATE.setdefault('workers', {})
            for lane in JOB_QUEUES:
                w = workers.get(lane)
                if not w or not w.is_alive():
                    w = threading.Thread(target=_worker, args=(lane,), daemon=True); w.start()
                    workers[lane] = w

        def _ahead_of(job):
            lane = LANES[job['kind']]
            return sum(1 for j in STATE['jobs'].values() if j['id'] != job['id'] and LANES[j['kind']] == lane
                       and j['status'] in ('queued', 'running') and j['created'] < job['created'])

        def _enqueue(job):
            job_id = uuid.uuid4().hex[:10]
            job.update({'id': job_id, 'status': 'queued', 'cancel': False, 'created': time.time(), 'percent': 0, 'detail': '',
                        'stage': None, 'stage_label': 'Na fila', 'error': None})
            job.setdefault('files', []); job.setdefault('items', []); job.setdefault('warning', None); job.setdefault('save_drive', False)
            STATE['jobs'][job_id] = job
            job['ahead'] = _ahead_of(job)
            JOB_QUEUES[LANES[job['kind']]].put(job_id); _ensure_workers()
            return job

        def cb_start_job(payload):
            try:
                kind = payload.get('kind')
                if kind == 'download':
                    job = _new_download_job(payload)
                elif kind == 'separate':
                    job = _new_separate_job(payload)
                else:
                    return JSON({'ok': False, 'error': 'Tipo de tarefa inválido.'})
                job['save_drive'] = bool(payload.get('save_drive'))
                _enqueue(job)
                return JSON({'ok': True, 'job_id': job['id'], 'ahead': job['ahead']})
            except Exception as e:
                return JSON({'ok': False, 'error': _friendly_error(str(e))})

        def _job_view(job):
            v = {k: val for k, val in job.items() if k not in ('cancel', 'proc', 'mix_tracks')}
            v['ok'] = True
            if job['status'] == 'running' and job.get('started'):
                v['elapsed'] = round(time.time() - job['started'])
            v['ahead'] = _ahead_of(job) if job['status'] == 'queued' else 0
            return v

        def cb_job_status(job_id):
            job = STATE['jobs'].get(job_id)
            if not job:
                return JSON({'ok': False, 'error': 'Tarefa não encontrada.'})
            return JSON(_job_view(job))

        def cb_jobs_list():
            try:
                out = []
                for j in sorted(STATE['jobs'].values(), key=lambda x: x['created']):
                    out.append({'id': j['id'], 'kind': j['kind'], 'status': j['status'], 'label': j.get('label'), 'title': j.get('title'),
                                'thumbnail': j.get('thumbnail'), 'percent': round(j.get('percent') or 0), 'stage_label': j.get('stage_label'),
                                'detail': j.get('detail'), 'error': j.get('error'), 'created': j['created'], 'files': len(j.get('files') or []),
                                'items': len(j.get('items') or []), 'errors': j.get('error_items') or 0})
                return JSON({'ok': True, 'jobs': out, 'active': any(j['status'] in ('queued', 'running') for j in STATE['jobs'].values())})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_cancel_job(job_id):
            job = STATE['jobs'].get(job_id)
            if job and job.get('status') in ('queued', 'running'):
                job['cancel'] = True
                p = job.get('proc')
                if p:
                    try: p.kill()
                    except Exception: pass
            return JSON({'ok': True})

        def cb_zip_job(job_id):
            try:
                job = STATE['jobs'].get(job_id)
                if not job or job.get('status') not in ('done', 'cancelled') or not job.get('files'):
                    return JSON({'ok': False, 'error': 'Nenhum arquivo para compactar.'})
                name = _safe_name(job.get('folder') or job.get('title') or 'arquivos', 60)
                slug = re.sub(r'[^\w.-]+', '_', name)
                zp = os.path.join(OUT_DIR, f'{job_id}_{slug}.zip')
                # Escreve uma vez e publica só o ZIP completo, inclusive em tentativas simultâneas.
                with tempfile.TemporaryDirectory(dir=OUT_DIR) as staging:
                    partial = os.path.join(staging, 'result.zip')
                    with zipfile.ZipFile(partial, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
                        for item in job['files']:
                            archive.write(_output_path(item['rel']), arcname=item['file'])
                    os.replace(partial, zp)
                sub = {'download': 'Downloads', 'separate': 'Faixas', 'mix': 'Mixagens'}[job['kind']]
                drive = _copy_to_drive(job, zp, sub) if job.get('save_drive') else None
                return JSON({'ok': True, 'rel': os.path.basename(zp), 'size_label': _fmt_size(os.path.getsize(zp)), 'drive_path': drive})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_colab_download(rel):
            """Download pelo mecanismo nativo do Colab (vai direto para a pasta Downloads do navegador)."""
            try:
                path = _output_path(rel)
                from google.colab import files as colab_files
                colab_files.download(path)
                return JSON({'ok': True})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_preview_b64(rel):
            """Plano B do mixer: entrega a prévia em base64 pela ponte do Colab quando o proxy de portas não responde."""
            try:
                path = _output_path(rel, preview=True)
                if os.path.getsize(path) > 32 * 1024 * 1024:
                    raise ValueError('Use o carregamento em blocos para esta prévia.')
                with open(path, 'rb') as f:
                    data = base64.b64encode(f.read()).decode('ascii')
                return JSON({'ok': True, 'b64': data, 'mime': 'video/mp4' if path.endswith('.mp4') else 'audio/mpeg'})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_preview_chunk(rel, offset=0):
            """Transfere prévias em blocos pequenos quando o proxy do Colab falha."""
            try:
                path = _output_path(rel, preview=True)
                total = os.path.getsize(path)
                if isinstance(offset, bool) or not isinstance(offset, int) or not 0 <= offset <= total:
                    raise ValueError('Posição da prévia inválida.')
                with open(path, 'rb') as stream:
                    stream.seek(offset)
                    chunk = stream.read(512 * 1024)
                return JSON({'ok': True, 'b64': base64.b64encode(chunk).decode('ascii'),
                             'offset': offset, 'next_offset': offset + len(chunk), 'total': total,
                             'mime': 'video/mp4' if path.endswith('.mp4') else 'audio/mpeg'})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        def cb_cleanup():
            try:
                for j in list(STATE['jobs'].values()):
                    if j['status'] in ('queued', 'running'):
                        return JSON({'ok': False, 'error': 'Há tarefas em andamento. Aguarde ou cancele antes de limpar.'})
                shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR, exist_ok=True)
                shutil.rmtree(WORK_DIR, ignore_errors=True); os.makedirs(WORK_DIR, exist_ok=True)
                STATE['jobs'].clear(); STATE['sources'].clear(); STATE['uploads'].clear()
                return JSON({'ok': True})
            except Exception as e:
                return JSON({'ok': False, 'error': str(e)})

        # ---------- pré-download do modelo padrão (em segundo plano) ----------
        def _prefetch_model(model):
            if MOCK or not SEP_AVAILABLE or _model_ready(model) or STATE['prefetch'].get(model):
                return
            STATE['prefetch'][model] = 'running'
            def run():
                with MODEL_LOCK:
                    try:
                        if not _model_ready(model):
                            subprocess.run(_sep_cmd() + ['--download_model_only', '--model_filename', model,
                                           '--model_file_dir', MODELS_DIR, '--log_level', 'warning'],
                                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=1800, check=True)
                            STATE.setdefault('ready_models', set()).add(model)
                        STATE['prefetch'][model] = 'done'
                    except Exception:
                        STATE['prefetch'][model] = 'failed'
            threading.Thread(target=run, daemon=True).start()

        def cb_models_status():
            return JSON({'ok': True, 'separator': SEP_AVAILABLE,
                         'models': {k: {'ready': _model_ready(v['model']), 'prefetch': STATE['prefetch'].get(v['model'])} for k, v in MODES.items()}})

        # ---------- servidor de arquivos (prévias do mixer, com suporte a Range) ----------
        class _FileHandler(http.server.SimpleHTTPRequestHandler):
            def __init__(self, *a, **k):
                super().__init__(*a, directory=OUT_DIR, **k)
            def log_message(self, *a): pass
            def list_directory(self, path):
                self.send_error(403, 'Listagem desabilitada'); return None
            def do_HEAD(self):
                try:
                    rel = urllib.parse.unquote(urllib.parse.urlsplit(self.path).path).lstrip('/')
                    _output_path(rel)
                except (ValueError, OSError):
                    self.send_error(404); return
                super().do_HEAD()
            def do_GET(self):
                try:
                    rel = urllib.parse.unquote(urllib.parse.urlsplit(self.path).path).lstrip('/')
                    path = _output_path(rel)
                except (ValueError, OSError):
                    self.send_error(404); return
                size = os.path.getsize(path)
                rng = self.headers.get('Range')
                start, end = 0, size - 1
                if rng:
                    match = re.fullmatch(r'bytes=(\d*)-(\d*)', rng)
                    try:
                        if not match or not any(match.groups()) or size == 0: raise ValueError()
                        a, b = match.groups()
                        if a:
                            start = int(a); end = min(int(b), size - 1) if b else size - 1
                        else:
                            suffix = int(b)
                            if suffix <= 0: raise ValueError()
                            start = max(0, size - suffix)
                        if start >= size or end < start: raise ValueError()
                    except ValueError:
                        self.send_response(416)
                        self.send_header('Content-Range', f'bytes */{size}')
                        self.send_header('Content-Length', '0'); self.end_headers(); return
                ctype = self.guess_type(path)
                self.send_response(206 if rng else 200)
                self.send_header('Content-Type', ctype)
                self.send_header('Accept-Ranges', 'bytes')
                self.send_header('Content-Length', str(end - start + 1))
                if rng: self.send_header('Content-Range', f'bytes {start}-{end}/{size}')
                self.send_header('Cache-Control', 'no-store')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                with open(path, 'rb') as f:
                    f.seek(start); remaining = end - start + 1
                    while remaining > 0:
                        chunk = f.read(min(65536, remaining))
                        if not chunk: break
                        try: self.wfile.write(chunk)
                        except (BrokenPipeError, ConnectionResetError): break
                        remaining -= len(chunk)

        def _start_file_server():
            if STATE.get('server'): return
            class _Srv(socketserver.ThreadingTCPServer):
                allow_reuse_address = True; daemon_threads = True
            try:
                srv = _Srv(('127.0.0.1', FILE_PORT), _FileHandler)
            except OSError as e:
                raise RuntimeError('A porta de prévias está ocupada. Reinicie a sessão e execute as células 1 e 2.') from e
            STATE['server'] = srv
            threading.Thread(target=srv.serve_forever, daemon=True).start()

        _start_file_server()
        _ensure_workers()
        _prefetch_model(MODES['2stem']['model'])

        # ---------- registro das funções para a interface ----------
        def _register(name, fn):
            try:
                from google.colab import output as _out
                _out.register_callback(f'stemtube.{name}', fn)
            except Exception:
                pass

        for _n, _f in {
            'bootstrap': cb_bootstrap, 'save_cookies': cb_save_cookies, 'clear_cookies': cb_clear_cookies, 'test_cookies': cb_test_cookies,
            'analyze': cb_analyze, 'upload_start': cb_upload_start, 'upload_chunk': cb_upload_chunk, 'upload_finish': cb_upload_finish,
            'start_job': cb_start_job, 'job_status': cb_job_status, 'jobs_list': cb_jobs_list, 'cancel_job': cb_cancel_job,
            'zip_job': cb_zip_job, 'colab_download': cb_colab_download, 'cleanup': cb_cleanup, 'models_status': cb_models_status,
            'preview_b64': cb_preview_b64, 'preview_chunk': cb_preview_chunk, 'mix_job': cb_mix_job,
        }.items():
            _register(_n, _f)

        APP_HTML = r"""<style>
          #stemtube, #stemtube * { box-sizing: border-box; }
          #stemtube {
            --bg: #0a0c14; --bg-2: #0f1220; --panel: rgba(21, 24, 38, .78); --panel-solid: #151826; --panel-2: #1b1f30; --line: rgba(255,255,255,.08); --line-2: rgba(255,255,255,.17);
            --text: #f2f4fb; --muted: #a2a9c0; --muted-2: #6f7691;
            --a1: #ff4f7b; --a2: #7c5cff; --a3: #22d3ee;
            --accent: var(--a2); --accent-soft: rgba(124,92,255,.16); --rose-soft: rgba(255,79,123,.14);
            --ok: #34d399; --ok-soft: rgba(52,211,153,.14); --warn: #fbbf24; --warn-soft: rgba(251,191,36,.14);
            --err: #f87171; --err-soft: rgba(248,113,113,.14); --info: #60a5fa; --info-soft: rgba(96,165,250,.14);
            --grad: linear-gradient(135deg, var(--a1), var(--a2) 60%, var(--a3));
            --grad-2: linear-gradient(135deg, var(--a1), var(--a2));
            --radius: 18px; --radius-sm: 12px; --shadow: 0 18px 50px rgba(0,0,0,.45); --glow: 0 0 0 1px rgba(255,255,255,.04), 0 12px 40px rgba(124,92,255,.2);
            font-family: 'Inter', system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif;
            color: var(--text); background: var(--bg); border-radius: 26px; overflow: hidden;
            max-width: 1140px; margin: 0 auto; min-height: 620px; position: relative; line-height: 1.5;
            -webkit-font-smoothing: antialiased; isolation: isolate;
          }
          #stemtube[data-theme="light"] {
            --bg: #f4f5fb; --bg-2: #eceef8; --panel: rgba(255,255,255,.84); --panel-solid: #ffffff; --panel-2: #f1f2fa; --line: rgba(20,24,40,.08); --line-2: rgba(20,24,40,.18);
            --text: #12152a; --muted: #5a6280; --muted-2: #8b93ad; --shadow: 0 18px 50px rgba(20,24,60,.10); --glow: 0 0 0 1px rgba(20,24,60,.04), 0 12px 40px rgba(124,92,255,.12);
          }
          #stemtube h1, #stemtube h2, #stemtube .display { font-family: 'Sora', 'Inter', system-ui, sans-serif; letter-spacing: -.02em; }

          /* ---- fundo animado ---- */
          #stemtube .aurora { position:absolute; inset:0; pointer-events:none; overflow:hidden; z-index:-1; }
          #stemtube .aurora i { position:absolute; border-radius:50%; filter: blur(60px); opacity:.5; animation: stFloat 18s ease-in-out infinite; }
          #stemtube .aurora i:nth-child(1) { width:560px; height:560px; left:-180px; top:-260px; background: radial-gradient(circle, rgba(255,79,123,.5), transparent 65%); }
          #stemtube .aurora i:nth-child(2) { width:560px; height:560px; right:-220px; top:-120px; background: radial-gradient(circle, rgba(124,92,255,.5), transparent 65%); animation-delay:-6s; }
          #stemtube .aurora i:nth-child(3) { width:480px; height:480px; left:35%; bottom:-300px; background: radial-gradient(circle, rgba(34,211,238,.32), transparent 65%); animation-delay:-11s; }
          #stemtube[data-theme="light"] .aurora i { opacity:.32; }
          @keyframes stFloat { 0%,100% { transform: translate(0,0) scale(1);} 50% { transform: translate(40px,30px) scale(1.08);} }
          #stemtube .grain { position:absolute; inset:0; pointer-events:none; z-index:-1; opacity:.06; background-image: radial-gradient(rgba(255,255,255,.6) .6px, transparent .6px); background-size: 3px 3px; }

          /* ---- topo ---- */
          #stemtube .topbar { display:flex; align-items:center; gap:12px; flex-wrap:wrap; padding:16px 22px; border-bottom:1px solid var(--line); background: var(--panel); backdrop-filter: blur(14px); }
          #stemtube .brand { display:flex; align-items:center; gap:12px; margin-right:auto; }
          #stemtube .logo { width:46px; height:46px; border-radius:14px; display:grid; place-items:center; background: var(--grad); box-shadow: 0 8px 22px rgba(124,92,255,.4); position:relative; overflow:hidden; }
          #stemtube .logo::after { content:''; position:absolute; inset:0; background: linear-gradient(120deg, transparent 30%, rgba(255,255,255,.35) 50%, transparent 70%); transform: translateX(-100%); animation: stShine 4.5s ease-in-out infinite; }
          @keyframes stShine { 0%,60% { transform: translateX(-100%);} 100% { transform: translateX(100%);} }
          #stemtube .logo svg { width:26px; height:26px; }
          #stemtube .brand h1 { font-size:19px; margin:0; font-weight:800; display:flex; align-items:center; gap:8px; }
          #stemtube .brand h1 span { background: var(--grad); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemtube .brand h1 .ver { font-family:'Inter',sans-serif; font-size:10.5px; font-weight:700; letter-spacing:.06em; padding:2px 7px; border-radius:999px; border:1px solid var(--line-2); color:var(--muted); }
          #stemtube .brand p { margin:0; font-size:12px; color:var(--muted); }
          #stemtube .status { display:flex; gap:8px; flex-wrap:wrap; align-items:center; }
          #stemtube .chip { display:inline-flex; align-items:center; gap:7px; padding:6px 11px; border-radius:999px; font-size:12px; font-weight:600; border:1px solid var(--line); background:var(--panel-2); color:var(--muted); white-space:nowrap; transition: all .2s; }
          #stemtube .chip .dot { width:8px; height:8px; border-radius:50%; background:var(--muted-2); flex:none; }
          #stemtube .chip.ok { color:var(--ok); border-color: rgba(52,211,153,.35); background:var(--ok-soft);} #stemtube .chip.ok .dot{background:var(--ok); box-shadow:0 0 8px var(--ok)}
          #stemtube .chip.warn { color:var(--warn); border-color: rgba(251,191,36,.35); background:var(--warn-soft);} #stemtube .chip.warn .dot{background:var(--warn)}
          #stemtube .chip.err { color:var(--err); background:var(--err-soft);} #stemtube .chip.err .dot{background:var(--err)}
          #stemtube .chip.info { color:var(--info); background:var(--info-soft);} #stemtube .chip.info .dot{background:var(--info)}
          #stemtube .chip.clickable { cursor:pointer; } #stemtube .chip.clickable:hover { border-color: var(--line-2); transform: translateY(-1px); }
          #stemtube .user { display:flex; align-items:center; gap:9px; padding:5px 12px 5px 5px; border-radius:999px; border:1px solid var(--line); background:var(--panel-2); font-size:12.5px; font-weight:600; max-width:100%; }
          #stemtube .avatar { width:28px; height:28px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:800; color:#fff; background: var(--grad-2); flex:none; }
          #stemtube .user span { overflow:hidden; text-overflow:ellipsis; white-space:nowrap; max-width:200px; }
          #stemtube .icon-btn { width:36px; height:36px; border-radius:10px; border:1px solid var(--line); background:var(--panel-2); color:var(--text); cursor:pointer; display:grid; place-items:center; font-size:16px; transition: all .15s; }
          #stemtube .icon-btn:hover { border-color:var(--line-2); transform: rotate(15deg); }

          /* ---- etapas ---- */
          #stemtube .stepper { display:flex; gap:6px; padding:16px 22px 0; overflow-x:auto; scrollbar-width:none; }
          #stemtube .stepper::-webkit-scrollbar{display:none}
          #stemtube .step { flex:1; min-width:118px; display:flex; align-items:center; gap:10px; padding:10px 12px; border-radius:12px; border:1px solid transparent; color:var(--muted-2); font-size:12.5px; font-weight:600; cursor:default; transition: all .25s; position:relative; }
          #stemtube .step .n { width:26px; height:26px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:800; background:var(--panel-2); border:1px solid var(--line); flex:none; transition: all .25s; }
          #stemtube .step.done { color:var(--muted); cursor:pointer; } #stemtube .step.done .n { background:var(--ok-soft); border-color:transparent; color:var(--ok); }
          #stemtube .step.active { color:var(--text); background:var(--panel); border-color:var(--line); box-shadow:var(--shadow); backdrop-filter: blur(10px); }
          #stemtube .step.active .n { background: var(--grad-2); border-color:transparent; color:#fff; box-shadow: 0 0 14px rgba(124,92,255,.6); }
          #stemtube .step.done:hover { background:var(--panel); }

          /* ---- fila ---- */
          #stemtube .queuebar { margin:16px 22px 0; padding:12px 14px; border-radius:14px; border:1px solid var(--line); background: var(--panel); box-shadow: var(--shadow); backdrop-filter: blur(10px); }
          #stemtube .queuebar .qhead { display:flex; align-items:center; gap:10px; flex-wrap:wrap; font-weight:800; font-size:13.5px; margin-bottom:8px; }
          #stemtube .qlist { display:grid; gap:6px; }
          #stemtube .qjob { display:grid; grid-template-columns: 26px 1fr 130px auto; gap:10px; align-items:center; padding:8px 10px; border-radius:10px; background:var(--panel-2); border:1px solid var(--line); font-size:12.5px; }
          #stemtube .qjob > div { min-width:0; }
          #stemtube .qjob.viewing { border-color:var(--accent); }
          #stemtube .qjob .ql { font-weight:700; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
          #stemtube .qjob .qs { color:var(--muted); font-size:11.5px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; margin-top:1px; }
          #stemtube .qjob .bar { height:6px; }
          #stemtube .st { width:24px; height:24px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:900; background:var(--panel-solid); border:1px solid var(--line); color:var(--muted-2); }
          #stemtube .st.done { background:var(--ok-soft); color:var(--ok); border-color:transparent; }
          #stemtube .st.warn { background:var(--warn-soft); color:var(--warn); border-color:transparent; }
          #stemtube .st.err { background:var(--err-soft); color:var(--err); border-color:transparent; }
          #stemtube .st.run { border-color:transparent; background:var(--accent-soft); }
          #stemtube .st.run::after { content:''; width:11px; height:11px; border:2px solid var(--accent-soft); border-top-color:var(--accent); border-radius:50%; animation: stSpin .8s linear infinite; }
          @media (max-width: 600px) { #stemtube .queuebar { margin:12px 14px 0; } #stemtube .qjob { grid-template-columns: 24px 1fr auto; } #stemtube .qjob .bar { display:none; } }

          /* ---- conteúdo ---- */
          #stemtube main { padding:20px 22px 26px; }
          #stemtube section[data-step] { display:none; animation: stFade .3s ease; }
          #stemtube section[data-step].on { display:block; }
          @keyframes stFade { from { opacity:0; transform: translateY(8px);} to { opacity:1; transform:none;} }
          #stemtube .card { background:var(--panel); border:1px solid var(--line); border-radius:var(--radius); padding:22px; box-shadow:var(--shadow); backdrop-filter: blur(12px); }
          #stemtube .card + .card, #stemtube .card + details.card, #stemtube details.card + .card, #stemtube .feats + .card, #stemtube .feats + details.card { margin-top:16px; }
          #stemtube h2 { font-size:21px; margin:0 0 6px; font-weight:800; }
          #stemtube h3 { font-size:14px; margin:0 0 10px; font-weight:700; color:var(--text); }
          #stemtube .sub { margin:0 0 16px; color:var(--muted); font-size:13.5px; }
          #stemtube .sub b, #stemtube .note b, #stemtube .steps-help b, #stemtube .feat b { color:var(--text); font-weight:600; }
          #stemtube .hero { text-align:center; padding:30px 20px 26px; position:relative; overflow:hidden; }
          #stemtube .hero h2 { font-size:31px; line-height:1.15; margin-bottom:10px; }
          #stemtube .hero h2 em { font-style:normal; background: var(--grad); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemtube .hero .sub { font-size:14.5px; max-width:640px; margin:0 auto 20px; }
          #stemtube .eq { display:flex; gap:4px; align-items:flex-end; justify-content:center; height:54px; margin:0 auto 18px; }
          #stemtube .eq i { width:6px; border-radius:4px; background: linear-gradient(180deg, var(--a3), var(--a2), var(--a1)); animation: stEq 1.4s ease-in-out infinite; opacity:.85; }
          @keyframes stEq { 0%,100% { height:14%; } 50% { height:100%; } }
          #stemtube .tabs { display:inline-flex; gap:4px; padding:4px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); margin-bottom:18px; }
          #stemtube .tab { padding:9px 16px; border-radius:10px; border:0; background:transparent; color:var(--muted); font:inherit; font-weight:700; font-size:13px; cursor:pointer; transition: all .2s; display:inline-flex; gap:8px; align-items:center; }
          #stemtube .tab.sel { background:var(--panel-solid); color:var(--text); box-shadow: 0 4px 14px rgba(0,0,0,.25); }
          #stemtube .src-pane { display:none; } #stemtube .src-pane.on { display:block; animation: stFade .25s ease; }

          #stemtube .input-row { display:flex; gap:10px; align-items:stretch; max-width:780px; margin:0 auto; }
          #stemtube .field { position:relative; flex:1; display:flex; align-items:center; }
          #stemtube .field svg { position:absolute; left:14px; width:18px; height:18px; fill:var(--muted-2); pointer-events:none; }
          #stemtube input[type=text], #stemtube textarea, #stemtube select { width:100%; background:var(--panel-2); border:1px solid var(--line); color:var(--text); border-radius:12px; padding:14px 14px 14px 42px; font:inherit; font-size:14.5px; outline:none; transition: border .15s, box-shadow .15s; }
          #stemtube select { padding:9px 12px; width:auto; font-size:13px; font-weight:600; }
          #stemtube textarea { padding:12px 14px; min-height:110px; resize:vertical; font-family: ui-monospace, Menlo, Consolas, monospace; font-size:12px; }
          #stemtube input[type=text].plain { padding-left:14px; }
          #stemtube input[type=text]:focus, #stemtube textarea:focus, #stemtube select:focus { border-color:var(--accent); box-shadow: 0 0 0 3px var(--accent-soft); }
          #stemtube input::placeholder { color:var(--muted-2); }
          #stemtube .linkmode { display:flex; flex-direction:column; align-items:center; gap:10px; margin:18px auto 0; max-width:780px; }
          #stemtube .linkmode .lm-label { font-size:13px; font-weight:700; color:var(--muted); }
          #stemtube .drop { max-width:780px; margin:0 auto; padding:34px 20px; border-radius:16px; border:2px dashed var(--line-2); background:var(--panel-2); cursor:pointer; transition: all .2s; position:relative; }
          #stemtube .drop:hover, #stemtube .drop.over { border-color:var(--accent); background: color-mix(in srgb, var(--accent-soft) 50%, var(--panel-2)); transform: scale(1.005); }
          #stemtube .drop .big { font-size:34px; margin-bottom:8px; }
          #stemtube .drop b { display:block; font-size:15px; }
          #stemtube .drop span { color:var(--muted); font-size:12.5px; }
          #stemtube .drop input { position:absolute; inset:0; opacity:0; cursor:pointer; }
          #stemtube .upbar { max-width:780px; margin:12px auto 0; display:none; }
          #stemtube .upbar.on { display:block; }
          #stemtube .feats { display:grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap:12px; margin-top:16px; }
          #stemtube .feat { display:flex; gap:12px; align-items:flex-start; padding:14px 16px; border-radius:14px; border:1px solid var(--line); background:var(--panel); backdrop-filter: blur(10px); font-size:12.5px; color:var(--muted); }
          #stemtube .feat .fi { width:38px; height:38px; border-radius:11px; display:grid; place-items:center; font-size:19px; flex:none; background: color-mix(in srgb, var(--c, var(--a2)) 18%, transparent); }
          #stemtube .feat b { display:block; font-size:13.5px; margin-bottom:2px; font-family:'Sora',sans-serif; }

          #stemtube .btn { display:inline-flex; align-items:center; justify-content:center; gap:8px; padding:12px 18px; border-radius:12px; border:1px solid var(--line); background:var(--panel-2); color:var(--text); font:inherit; font-size:14px; font-weight:700; cursor:pointer; transition: transform .1s, background .15s, border-color .15s, opacity .15s, box-shadow .2s; white-space:nowrap; text-decoration:none; }
          #stemtube .btn:hover { border-color:var(--line-2); transform: translateY(-1px); } #stemtube .btn:active { transform: translateY(1px) scale(.99); }
          #stemtube .btn.primary { background: linear-gradient(135deg, var(--a1), var(--a2) 55%, #5b8cff); background-size: 200% 200%; border-color:transparent; color:#fff; box-shadow: 0 10px 26px rgba(124,92,255,.35); animation: stGrad 6s ease infinite; }
          @keyframes stGrad { 0%,100% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } }
          #stemtube .btn.primary:hover { filter: brightness(1.08); box-shadow: 0 14px 32px rgba(124,92,255,.45); }
          #stemtube .btn.ghost { background:transparent; }
          #stemtube .btn.danger { color:var(--err); border-color: rgba(248,113,113,.4); background:var(--err-soft); }
          #stemtube .btn.sm { padding:8px 12px; font-size:12.5px; border-radius:10px; }
          #stemtube .btn.lg { padding:15px 28px; font-size:15px; border-radius:14px; }
          #stemtube .btn:disabled { opacity:.45; cursor:not-allowed; transform:none; filter:none; animation:none; }
          #stemtube .btn .spin, #stemtube .spin { width:16px; height:16px; border:2px solid rgba(255,255,255,.35); border-top-color:#fff; border-radius:50%; animation: stSpin .7s linear infinite; display:inline-block; }
          @keyframes stSpin { to { transform: rotate(360deg);} }
          #stemtube .hint { font-size:12.5px; color:var(--muted-2); margin-top:12px; }
          #stemtube .hint b { color:var(--muted); font-weight:700; }
          #stemtube .actions { display:flex; gap:10px; justify-content:space-between; align-items:center; flex-wrap:wrap; margin-top:20px; }
          #stemtube .actions .right { display:flex; gap:10px; margin-left:auto; flex-wrap:wrap; }
          #stemtube .row { display:flex; gap:10px; flex-wrap:wrap; align-items:center; }
          #stemtube .grid2 { display:grid; grid-template-columns: 1fr 1fr; gap:16px; }
          #stemtube .small { font-size:12.5px; } #stemtube .muted { color:var(--muted); }
          #stemtube .badge { display:inline-flex; align-items:center; gap:5px; padding:3px 9px; border-radius:999px; font-size:11.5px; font-weight:700; background:var(--panel-2); border:1px solid var(--line); color:var(--muted); }
          #stemtube .badge.ok { color:var(--ok); background:var(--ok-soft); border-color:transparent; }
          #stemtube .badge.warn { color:var(--warn); background:var(--warn-soft); border-color:transparent; }
          #stemtube .badge.err { color:var(--err); background:var(--err-soft); border-color:transparent; }
          #stemtube .badge.info { color:var(--info); background:var(--info-soft); border-color:transparent; }
          #stemtube .badge.acc { color:#fff; background: var(--grad-2); border-color:transparent; }
          #stemtube .linkbtn { background:none; border:0; color:var(--info); font:inherit; font-size:12.5px; font-weight:700; cursor:pointer; padding:6px 0; }
          #stemtube .sr { position:absolute; width:1px; height:1px; overflow:hidden; clip:rect(0 0 0 0); }

          /* ---- colapsável ---- */
          #stemtube details.card { padding:0; }
          #stemtube details.card > summary { list-style:none; cursor:pointer; padding:16px 20px; display:flex; align-items:center; gap:12px; font-weight:700; font-size:14px; }
          #stemtube details.card > summary::-webkit-details-marker { display:none; }
          #stemtube details.card > summary .chev { margin-left:auto; transition: transform .2s; color:var(--muted); }
          #stemtube details.card[open] > summary .chev { transform: rotate(180deg); }
          #stemtube details.card > .body { padding:0 20px 20px; border-top:1px solid var(--line); padding-top:16px; }
          #stemtube .steps-help { counter-reset: s; margin:0; padding:0; list-style:none; display:grid; gap:10px; }
          #stemtube .steps-help li { display:flex; gap:12px; font-size:13px; color:var(--muted); align-items:flex-start; }
          #stemtube .steps-help li::before { counter-increment:s; content: counter(s); width:22px; height:22px; border-radius:50%; background:var(--accent-soft); color:var(--a2); font-weight:800; font-size:11.5px; display:grid; place-items:center; flex:none; margin-top:1px; }
          #stemtube .steps-help a, #stemtube .note a, #stemtube .sub a { color:var(--info); text-decoration:none; font-weight:600; }
          #stemtube .note { font-size:12.5px; color:var(--muted); padding:12px 14px; border-radius:12px; background:var(--panel-2); border:1px dashed var(--line-2); margin-top:14px; }
          #stemtube .note.warn { border-color: rgba(251,191,36,.5); background:var(--warn-soft); color: var(--text); border-style:solid; }
          #stemtube .note.info { border-color: rgba(96,165,250,.5); background:var(--info-soft); color: var(--text); border-style:solid; }
          #stemtube .note.err { border-color: rgba(248,113,113,.5); background:var(--err-soft); color: var(--text); border-style:solid; }
          #stemtube .note.ok { border-color: rgba(52,211,153,.5); background:var(--ok-soft); color: var(--text); border-style:solid; }

          /* ---- prévia ---- */
          #stemtube .preview { display:grid; grid-template-columns: minmax(0, 1.1fr) minmax(0, 1fr); gap:20px; align-items:start; }
          #stemtube .player { position:relative; border-radius:14px; overflow:hidden; background:#000; aspect-ratio:16/9; cursor:pointer; box-shadow: var(--shadow); border:1px solid var(--line); }
          #stemtube .player img { width:100%; height:100%; object-fit:cover; display:block; transition: transform .4s; }
          #stemtube .player:hover img { transform: scale(1.04); }
          #stemtube .player iframe { position:absolute; inset:0; width:100%; height:100%; border:0; }
          #stemtube .player .play { position:absolute; inset:0; display:grid; place-items:center; background: linear-gradient(to top, rgba(0,0,0,.5), transparent 60%); }
          #stemtube .player .play span { width:66px; height:48px; border-radius:15px; background:var(--a1); display:grid; place-items:center; box-shadow: 0 10px 25px rgba(0,0,0,.5); transition: transform .15s; }
          #stemtube .player:hover .play span { transform: scale(1.07); }
          #stemtube .player .play svg { width:22px; height:22px; fill:#fff; margin-left:3px; }
          #stemtube .player .dur { position:absolute; right:10px; bottom:10px; background:rgba(0,0,0,.8); color:#fff; padding:3px 7px; border-radius:6px; font-size:12px; font-weight:700; }
          #stemtube .filecard { border-radius:14px; background: linear-gradient(135deg, rgba(255,79,123,.25), rgba(124,92,255,.25), rgba(34,211,238,.18)); aspect-ratio:16/9; display:grid; place-items:center; font-size:56px; position:relative; border:1px solid var(--line); }
          #stemtube .vtitle { font-size:19px; font-weight:800; margin:0 0 6px; line-height:1.3; font-family:'Sora',sans-serif; letter-spacing:-.01em; }
          #stemtube .vchan { color:var(--muted); font-size:13.5px; margin-bottom:12px; display:flex; align-items:center; gap:8px; }
          #stemtube .vchan a { color:var(--text); font-weight:600; text-decoration:none; }
          #stemtube .meta { display:flex; gap:8px; flex-wrap:wrap; margin-bottom:14px; }
          #stemtube .desc { font-size:13px; color:var(--muted); white-space:pre-wrap; max-height:96px; overflow:hidden; position:relative; }
          #stemtube .desc.open { max-height:none; }
          #stemtube .desc:not(.open)::after { content:''; position:absolute; left:0; right:0; bottom:0; height:40px; background: linear-gradient(to bottom, transparent, var(--panel-solid)); }
          @media (max-width: 760px) { #stemtube .preview, #stemtube .grid2 { grid-template-columns: 1fr; } }

          /* ---- playlist ---- */
          #stemtube .plhead { display:flex; gap:16px; align-items:center; }
          #stemtube .plhead img { width:160px; aspect-ratio:16/9; object-fit:cover; border-radius:12px; border:1px solid var(--line); flex:none; background:#000; }
          #stemtube .toolbar { display:flex; gap:10px; align-items:center; flex-wrap:wrap; margin:16px 0 10px; }
          #stemtube .toolbar .field { max-width:320px; min-width:180px; }
          #stemtube .toolbar input[type=text] { padding:10px 12px 10px 38px; font-size:13.5px; }
          #stemtube .toolbar .field svg { left:12px; width:16px; height:16px; }
          #stemtube .counter { margin-left:auto; font-size:13px; color:var(--muted); font-weight:600; }
          #stemtube .list { border:1px solid var(--line); border-radius:14px; overflow:hidden; max-height:440px; overflow-y:auto; background:var(--panel-2); }
          #stemtube .item { display:grid; grid-template-columns: 34px 30px 96px 1fr auto; gap:12px; align-items:center; padding:10px 12px; border-bottom:1px solid var(--line); cursor:pointer; transition: background .12s; }
          #stemtube .item:last-child { border-bottom:0; } #stemtube .item:hover { background: color-mix(in srgb, var(--panel-solid) 60%, var(--panel-2)); }
          #stemtube .item.off { opacity:.45; cursor:not-allowed; }
          #stemtube .item .idx { color:var(--muted-2); font-size:12px; font-weight:700; text-align:right; }
          #stemtube .item img { width:100%; aspect-ratio:16/9; object-fit:cover; border-radius:8px; background:#000; }
          #stemtube .item .it { font-size:13.5px; font-weight:600; overflow:hidden; text-overflow:ellipsis; display:-webkit-box; -webkit-line-clamp:2; -webkit-box-orient:vertical; line-height:1.3; }
          #stemtube .item .ic { font-size:12px; color:var(--muted-2); margin-top:3px; }
          #stemtube .item .dur { font-size:12px; font-weight:700; color:var(--muted); background:var(--panel-solid); border:1px solid var(--line); padding:3px 7px; border-radius:6px; }
          #stemtube .cb { width:20px; height:20px; border-radius:6px; border:2px solid var(--line-2); display:grid; place-items:center; transition: all .12s; background:var(--panel-solid); }
          #stemtube .cb.on { background:var(--a2); border-color:var(--a2); } #stemtube .cb.on::after { content:'✓'; color:#fff; font-size:12px; font-weight:900; }
          @media (max-width: 820px) { #stemtube .item { grid-template-columns: 30px 26px 80px 1fr; } #stemtube .item .dur { display:none; } }
          @media (max-width: 600px) { #stemtube .item { grid-template-columns: 28px 1fr; } #stemtube .item .idx, #stemtube .item img { display:none; } #stemtube .plhead { flex-direction:column; align-items:flex-start; } #stemtube .plhead img { width:100%; } }

          /* ---- cartões de escolha ---- */
          #stemtube .choices { display:grid; grid-template-columns: repeat(auto-fit, minmax(230px, 1fr)); gap:12px; }
          #stemtube .choice { position:relative; text-align:left; padding:18px; border-radius:16px; border:1px solid var(--line); background:var(--panel-2); cursor:pointer; transition: all .2s; color:var(--text); font:inherit; overflow:hidden; }
          #stemtube .choice::before { content:''; position:absolute; inset:0; background: linear-gradient(135deg, rgba(255,79,123,.14), rgba(124,92,255,.18)); opacity:0; transition: opacity .2s; }
          #stemtube .choice:hover { border-color:var(--line-2); transform: translateY(-2px); box-shadow: var(--shadow); }
          #stemtube .choice:hover::before { opacity:.6; }
          #stemtube .choice.sel { border-color:var(--a2); box-shadow: 0 0 0 3px var(--accent-soft), var(--glow); }
          #stemtube .choice.sel::before { opacity:1; }
          #stemtube .choice > * { position:relative; }
          #stemtube .choice .ic { font-size:26px; margin-bottom:8px; display:block; }
          #stemtube .choice .t { font-weight:800; font-size:15px; display:block; font-family:'Sora',sans-serif; }
          #stemtube .choice .d { display:block; font-size:12.5px; color:var(--muted); margin-top:4px; line-height:1.45; }
          #stemtube .choice .tag { position:absolute; top:12px; right:12px; }
          #stemtube .choice.big { padding:24px; } #stemtube .choice.big .ic { font-size:34px; } #stemtube .choice.big .t { font-size:18px; }
          #stemtube .choice.sel .check { position:absolute; top:12px; right:12px; width:22px; height:22px; border-radius:50%; background: var(--grad-2); color:#fff; display:grid; place-items:center; font-size:12px; font-weight:900; }
          #stemtube .choice:not(.sel) .check { display:none; }
          #stemtube .choice.off { opacity:.5; cursor:not-allowed; } #stemtube .choice.off:hover { transform:none; box-shadow:none; }
          #stemtube .quality { display:grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap:10px; }
          #stemtube .quality .choice { padding:14px; } #stemtube .quality .choice .t { font-size:17px; } #stemtube .quality .choice .check { display:none !important; }
          #stemtube .pills { display:flex; gap:6px; flex-wrap:wrap; margin-top:10px; }
          #stemtube .pill { font-size:11.5px; font-weight:700; padding:3px 9px; border-radius:999px; color:#0b0d14; }
          #stemtube .chips { display:flex; gap:8px; flex-wrap:wrap; }
          #stemtube .chipbtn { padding:9px 14px; border-radius:999px; border:1px solid var(--line); background:var(--panel-2); color:var(--muted); font:inherit; font-size:13px; font-weight:700; cursor:pointer; transition: all .15s; }
          #stemtube .chipbtn:hover { border-color:var(--line-2); color:var(--text); }
          #stemtube .chipbtn.sel { border-color:transparent; color:#fff; background: var(--grad-2); box-shadow: 0 6px 16px rgba(124,92,255,.35); }
          #stemtube .chipbtn small { display:block; font-weight:500; font-size:11px; opacity:.85; }
          #stemtube .toggle { display:flex; align-items:center; justify-content:space-between; gap:14px; padding:14px 0; border-top:1px solid var(--line); cursor:pointer; }
          #stemtube .toggle:first-of-type { border-top:0; padding-top:0; }
          #stemtube .toggle .tt { font-weight:700; font-size:14px; } #stemtube .toggle .td { font-size:12.5px; color:var(--muted); margin-top:2px; }
          #stemtube .sw { width:46px; height:26px; border-radius:999px; background:var(--line-2); position:relative; flex:none; transition: background .2s; }
          #stemtube .sw::after { content:''; position:absolute; top:3px; left:3px; width:20px; height:20px; border-radius:50%; background:#fff; transition: transform .2s; box-shadow: 0 2px 6px rgba(0,0,0,.3); }
          #stemtube .sw.on { background: var(--grad-2); } #stemtube .sw.on::after { transform: translateX(20px); }
          #stemtube .toggle.off { cursor:default; opacity:.7; }
          #stemtube .summary { display:grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap:10px; }
          #stemtube .summary div { padding:12px 14px; border-radius:12px; background:var(--panel-2); border:1px solid var(--line); }
          #stemtube .summary .k { font-size:11px; text-transform:uppercase; letter-spacing:.06em; color:var(--muted-2); font-weight:700; }
          #stemtube .summary .v { font-weight:700; font-size:14px; margin-top:3px; word-break:break-word; }
          #stemtube .summary .v small { color:var(--muted); font-weight:600; font-size:12px; }
          #stemtube .mini-list { margin-top:14px; display:grid; gap:6px; max-height:200px; overflow:auto; }
          #stemtube .mini-list div { display:flex; gap:10px; align-items:center; font-size:13px; color:var(--muted); }
          #stemtube .mini-list img { width:56px; aspect-ratio:16/9; object-fit:cover; border-radius:6px; background:#000; }
          #stemtube .mini-list span { overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }

          /* ---- processamento ---- */
          #stemtube .proc { display:grid; grid-template-columns: 200px 1fr; gap:26px; align-items:center; }
          #stemtube .ringbox { display:flex; flex-direction:column; align-items:center; gap:12px; }
          #stemtube .ringbox .stage { font-size:13px; font-weight:700; color:var(--muted); text-align:center; max-width:220px; line-height:1.35; }
          #stemtube .ring { width:200px; height:200px; position:relative; display:grid; place-items:center; margin:0 auto; flex:none; }
          #stemtube .ring svg { position:absolute; inset:0; transform: rotate(-90deg); }
          #stemtube .ring circle { fill:none; stroke-width:10; stroke-linecap:round; }
          #stemtube .ring .track { stroke: var(--line-2); }
          #stemtube .ring .prog { stroke: url(#stGrad); transition: stroke-dashoffset .6s ease; filter: drop-shadow(0 0 8px rgba(124,92,255,.6)); }
          #stemtube .ring .pct { font-family:'Sora',sans-serif; font-size:40px; font-weight:800; line-height:1; text-align:center; position:relative; }
          #stemtube .ring .pct small { display:block; font-size:11px; color:var(--muted-2); font-weight:600; margin-top:6px; font-family:'Inter',sans-serif; letter-spacing:.04em; text-transform:uppercase; }
          #stemtube .ring.busy svg { transform-origin: 50% 50%; animation: stRingBusy 1.6s linear infinite; }
          #stemtube .ring.busy .prog { stroke-dasharray: 132 396; stroke-dashoffset: 0; transition: none; }
          @keyframes stRingBusy { from { transform: rotate(-90deg); } to { transform: rotate(270deg); } }
          #stemtube .timeline { display:grid; gap:10px; }
          #stemtube .tl { display:grid; grid-template-columns: 30px 1fr auto; gap:12px; align-items:center; padding:12px 14px; border-radius:12px; border:1px solid var(--line); background:var(--panel-2); transition: all .25s; }
          #stemtube .tl.running { border-color: rgba(124,92,255,.5); background: color-mix(in srgb, var(--accent-soft) 40%, var(--panel-2)); box-shadow: 0 0 0 3px var(--accent-soft); }
          #stemtube .tl.done { opacity:.75; }
          #stemtube .tl .lab { font-weight:700; font-size:13.5px; } #stemtube .tl .det { font-size:12px; color:var(--muted); margin-top:1px; min-height:14px; }
          #stemtube .tl .tm { font-size:12px; color:var(--muted-2); font-variant-numeric: tabular-nums; }
          #stemtube .bar { height:8px; border-radius:999px; background:var(--line); overflow:hidden; position:relative; }
          #stemtube .bar i { display:block; height:100%; width:0; border-radius:999px; background: linear-gradient(90deg, var(--a1), var(--a2)); transition: width .4s ease; }
          #stemtube .bar.ok i { background: linear-gradient(90deg, var(--ok), #6ee7b7); }
          #stemtube .bar.striped i { background-image: linear-gradient(90deg, var(--a1), var(--a2)), repeating-linear-gradient(45deg, rgba(255,255,255,.18) 0 8px, transparent 8px 16px); background-blend-mode: overlay; animation: stStripe 1s linear infinite; }
          @keyframes stStripe { to { background-position: 0 0, 32px 0; } }
          #stemtube .cur { display:grid; grid-template-columns: 160px 1fr; gap:16px; align-items:center; margin:18px 0; }
          #stemtube .cur img { width:160px; aspect-ratio:16/9; object-fit:cover; border-radius:12px; border:1px solid var(--line); background:#000; }
          #stemtube .cur .ct { font-weight:800; font-size:15px; margin-bottom:4px; }
          #stemtube .cur .cs { color:var(--muted); font-size:13px; margin-bottom:10px; }
          #stemtube .stats { display:flex; gap:14px; flex-wrap:wrap; font-size:12.5px; color:var(--muted); margin-top:8px; font-weight:600; }
          #stemtube .stats b { color:var(--text); }
          #stemtube .jobs { display:grid; gap:8px; margin-top:16px; }
          #stemtube .job { display:grid; grid-template-columns: 28px 1fr auto; gap:12px; align-items:center; padding:10px 14px; border-radius:12px; background:var(--panel-2); border:1px solid var(--line); }
          #stemtube .job .jt { font-size:13.5px; font-weight:600; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
          #stemtube .job .js { font-size:12px; color:var(--muted); margin-top:2px; }
          #stemtube .job .js.err { color:var(--err); } #stemtube .job .js.warn { color:var(--warn); }
          #stemtube .job .jr { display:flex; gap:6px; align-items:center; flex-wrap:wrap; justify-content:flex-end; }
          @media (max-width: 720px) { #stemtube .proc { grid-template-columns: 1fr; } #stemtube .cur { grid-template-columns: 1fr; } #stemtube .cur img { width:100%; max-height:200px; } }

          /* ---- mixer ---- */
          #stemtube .transport { display:grid; grid-template-columns: auto 1fr auto; gap:14px; align-items:center; padding:14px 16px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); }
          #stemtube .playbtn { width:52px; height:52px; border-radius:50%; border:0; cursor:pointer; color:#fff; font-size:20px; display:grid; place-items:center; background: var(--grad-2); box-shadow: 0 10px 24px rgba(124,92,255,.4); transition: transform .15s; }
          #stemtube .playbtn:hover { transform: scale(1.06); } #stemtube .playbtn:disabled { opacity:.5; }
          #stemtube .seek { -webkit-appearance:none; appearance:none; width:100%; height:8px; border-radius:999px; background: linear-gradient(90deg, var(--a2) var(--p, 0%), var(--line) var(--p, 0%)); outline:none; cursor:pointer; }
          #stemtube .seek::-webkit-slider-thumb { -webkit-appearance:none; width:18px; height:18px; border-radius:50%; background:#fff; box-shadow: 0 2px 8px rgba(0,0,0,.4); border:3px solid var(--a2); }
          #stemtube .seek::-moz-range-thumb { width:18px; height:18px; border-radius:50%; background:#fff; border:3px solid var(--a2); }
          #stemtube .time { font-variant-numeric: tabular-nums; font-size:12.5px; color:var(--muted); font-weight:600; white-space:nowrap; }
          #stemtube .stems { display:grid; gap:10px; margin-top:14px; }
          #stemtube .stem { display:grid; grid-template-columns: 44px 1fr auto; gap:12px; align-items:center; padding:12px 14px; border-radius:14px; background:var(--panel-2); border:1px solid var(--line); border-left:4px solid var(--c, var(--a2)); transition: all .2s; }
          #stemtube .stem.muted { opacity:.55; }
          #stemtube .stem .ico { width:44px; height:44px; border-radius:12px; display:grid; place-items:center; font-size:22px; background: color-mix(in srgb, var(--c) 22%, transparent); }
          #stemtube .stem .nm { font-weight:800; font-size:14.5px; font-family:'Sora',sans-serif; display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
          #stemtube .stem .fi { font-size:12px; color:var(--muted); margin-top:2px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
          #stemtube .stem .ctl { display:flex; align-items:center; gap:8px; flex-wrap:wrap; justify-content:flex-end; }
          #stemtube .mbtn { width:34px; height:34px; border-radius:9px; border:1px solid var(--line); background:var(--panel-solid); color:var(--muted); font-weight:800; font-size:12px; cursor:pointer; transition: all .15s; }
          #stemtube .mbtn.on.m { background:var(--err-soft); color:var(--err); border-color:transparent; }
          #stemtube .mbtn.on.s { background:var(--warn-soft); color:var(--warn); border-color:transparent; }
          #stemtube .vol { -webkit-appearance:none; appearance:none; width:110px; height:6px; border-radius:999px; background: linear-gradient(90deg, var(--c) var(--p, 100%), var(--line) var(--p, 100%)); outline:none; cursor:pointer; }
          #stemtube .vol::-webkit-slider-thumb { -webkit-appearance:none; width:14px; height:14px; border-radius:50%; background:#fff; border:2px solid var(--c); }
          #stemtube .vol::-moz-range-thumb { width:14px; height:14px; border-radius:50%; background:#fff; border:2px solid var(--c); }
          #stemtube .meter { display:flex; gap:2px; align-items:flex-end; height:18px; width:36px; }
          #stemtube .meter i { flex:1; background: var(--c); border-radius:2px; height:20%; opacity:.5; transition: height .1s; }
          #stemtube .stem.playing .meter i { animation: stMeter .9s ease-in-out infinite; opacity:.9; }
          #stemtube .stem.playing .meter i:nth-child(2) { animation-delay:-.3s; } #stemtube .stem.playing .meter i:nth-child(3) { animation-delay:-.6s; } #stemtube .stem.playing .meter i:nth-child(4) { animation-delay:-.15s; }
          @keyframes stMeter { 0%,100% { height:25%; } 50% { height:95%; } }
          @media (max-width: 640px) { #stemtube .stem { grid-template-columns: 40px 1fr; } #stemtube .stem .ctl { grid-column: 1 / -1; justify-content:flex-start; } #stemtube .vol { width:100%; flex:1; } }
          #stemtube .result-head { display:flex; align-items:center; gap:16px; flex-wrap:wrap; }
          #stemtube .result-head .big { width:62px; height:62px; border-radius:18px; display:grid; place-items:center; font-size:30px; background: var(--grad-2); box-shadow: 0 10px 26px rgba(124,92,255,.4); animation: stPop .5s cubic-bezier(.2,1.4,.4,1); }
          #stemtube .result-head .big.err { background: linear-gradient(135deg, var(--err), #fb923c); } #stemtube .result-head .big.warn { background: linear-gradient(135deg, var(--warn), #fb923c); }
          @keyframes stPop { from { transform: scale(.4); opacity:0; } to { transform: scale(1); opacity:1; } }
          #stemtube .confetti { position:absolute; width:8px; height:8px; border-radius:2px; top:-10px; animation: stConf 1.8s ease-out forwards; pointer-events:none; }
          @keyframes stConf { to { transform: translateY(420px) rotate(540deg); opacity:0; } }
          #stemtube audio.mixprev { width:100%; margin-top:14px; border-radius:12px; }

          /* ---- rodapé, toasts, modal ---- */
          #stemtube footer { display:flex; gap:12px; flex-wrap:wrap; justify-content:space-between; align-items:center; padding:14px 22px; border-top:1px solid var(--line); font-size:12px; color:var(--muted-2); }
          #stemtube footer .signature { font-weight:700; color:var(--muted); }
          #stemtube footer .signature b { background: var(--grad); -webkit-background-clip:text; background-clip:text; color:transparent; }
          #stemtube .toasts { position:fixed; right:18px; bottom:18px; display:grid; gap:8px; z-index:1000; max-width:min(420px, calc(100vw - 36px)); }
          #stemtube .toast { display:flex; gap:10px; align-items:flex-start; padding:12px 14px; border-radius:12px; background:var(--panel-solid); border:1px solid var(--line); color:var(--text); box-shadow: var(--shadow); font-size:13px; animation: stToast .25s ease; }
          @keyframes stToast { from { transform: translateY(10px); opacity:0; } to { transform:none; opacity:1; } }
          #stemtube .toast.ok { border-left:4px solid var(--ok);} #stemtube .toast.err { border-left:4px solid var(--err);} #stemtube .toast.warn { border-left:4px solid var(--warn);} #stemtube .toast.info { border-left:4px solid var(--info);}
          #stemtube .overlay { position:fixed; inset:0; background: rgba(0,0,0,.55); display:none; place-items:center; z-index:900; padding:20px; backdrop-filter: blur(4px); }
          #stemtube .overlay.on { display:grid; }
          #stemtube .modal { width:min(480px, 100%); background:var(--panel-solid); border:1px solid var(--line); border-radius:18px; padding:22px; box-shadow: var(--shadow); animation: stPop .3s cubic-bezier(.2,1.2,.4,1); }
          #stemtube .modal h3 { font-size:17px; margin-bottom:8px; } #stemtube .modal p { color:var(--muted); font-size:13.5px; margin:0 0 16px; }
          #stemtube .modal .actions { margin-top:0; justify-content:flex-end; }
          #stemtube .loading { text-align:center; padding:36px 20px; }
          #stemtube .loading .lring { width:46px; height:46px; margin:0 auto 14px; border-radius:50%; border:4px solid var(--accent-soft); border-top-color:var(--a2); animation: stSpin .8s linear infinite; }
          #stemtube .loading p { margin:0; color:var(--muted); font-size:14px; }
          @media (max-width: 600px) { #stemtube { border-radius:0; } #stemtube .brand p { display:none; } #stemtube .status { width:100%; order:5; flex-wrap:nowrap; overflow-x:auto; scrollbar-width:none; padding-bottom:2px; } #stemtube .status::-webkit-scrollbar { display:none; } #stemtube .user { order:3; } #stemtube .icon-btn { order:4; } #stemtube .brand { order:1; } #stemtube main { padding:16px 14px 22px; } #stemtube .topbar, #stemtube .stepper, #stemtube footer { padding-left:14px; padding-right:14px; } #stemtube .hero h2 { font-size:24px; } #stemtube .input-row { flex-direction:column; } #stemtube .card { padding:16px; } #stemtube .user span { max-width:120px; } #stemtube .step span.lbl { display:none; } #stemtube .step { min-width:0; justify-content:center; padding:8px; } #stemtube .step.active span.lbl { display:inline; } }

          #stemtube .studio-preview { margin:16px 0; border:1px solid #8471ba55; border-radius:20px; overflow:hidden; background:linear-gradient(135deg,#201d36,#141827); color:#f4f1ff; box-shadow:0 8px 26px #0002; }
          #stemtube .studio-preview summary { display:flex; align-items:center; gap:14px; padding:18px 22px; cursor:pointer; list-style:none; }
          #stemtube .studio-preview summary::-webkit-details-marker { display:none; }
          #stemtube .studio-preview summary>span:nth-child(2) { flex:1; min-width:0; }
          #stemtube .studio-preview summary small { display:block; color:#b9b1cd; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; margin-top:4px; }
          #stemtube .preview-icon { display:grid; place-items:center; width:44px; height:44px; border-radius:14px; background:#9d7eff22; color:#c7b5ff; font-size:26px; }
          #stemtube .preview-body { padding:0 22px 18px; }
          #stemtube .preview-body audio { display:none; }
          #stemtube .preview-body video { display:block; width:100%; max-height:480px; object-fit:contain; background:#080910; border-radius:12px; margin-bottom:16px; }
          #stemtube .preview-transport { display:flex; align-items:center; gap:18px; padding:14px 0; }
          #stemtube .preview-play { width:54px; height:54px; flex-shrink:0; border:0; border-radius:50%; background:linear-gradient(140deg,#df72c0,#8c64ff); color:white; font-size:22px; cursor:pointer; box-shadow:0 4px 18px #9564ff44; }
          #stemtube .preview-play:disabled { opacity:.45; cursor:wait; }
          #stemtube .preview-timeline { flex:1; min-width:0; }
          #stemtube .preview-time { display:flex; justify-content:space-between; color:#c3bdd5; font-variant-numeric:tabular-nums; font-size:12px; margin-bottom:8px; }
          #stemtube .preview-timeline input { width:100%; accent-color:#bb97ff; cursor:pointer; }
          #stemtube .preview-options { display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:16px; border-top:1px solid #ffffff14; padding-top:14px; font-size:12px; color:#c3bdd5; }
          #stemtube .preview-options label { display:flex; align-items:center; gap:10px; }
          #stemtube .preview-options input { width:90px; accent-color:#bb97ff; }
          #stemtube .preview-options select { background:#29233d; color:#eee8ff; border:1px solid #77658f; border-radius:8px; padding:5px 8px; }
          #stemtube .preview-status { font-size:12px; color:#bdb3d5; margin:14px 0 8px; }
          #stemtube .preview-body .small { color:#9993ac; font-size:11px; margin:10px 0 0; }
          #stemtube .studio-preview.is-playing { border-color:#b393ff; }
          #stemtube .preview-retry[hidden] { display:none; }


          body.studio-window { margin:0; }
          .studio-window #stemtube { width:100%; max-width:none; min-height:100vh; min-height:100dvh; border-radius:0; display:flex; flex-direction:column; }
          .studio-window #stemtube main { flex:1; width:min(100%, 1440px); margin-inline:auto; min-width:0; }
          #stemtube .modal { max-height:calc(100dvh - 40px); overflow:auto; }
          @media (max-width:600px) {
            #stemtube .status { flex-wrap:wrap; }
            #stemtube .tabs { max-width:100%; }
            #stemtube .tab { flex:1; padding:10px; justify-content:center; }
            #stemtube .btn, #stemtube .tab, #stemtube .icon-btn, #stemtube .mbtn { min-height:44px; }
            #stemtube .icon-btn, #stemtube .mbtn { min-width:44px; }
            #stemtube .stem, #stemtube .item { grid-template-columns:auto minmax(0,1fr); }
            #stemtube .job { grid-template-columns:auto minmax(0,1fr); }
            #stemtube .job > :last-child { grid-column:2; }
            #stemtube input[type=text], #stemtube textarea, #stemtube select { font-size:16px; }
            #stemtube .actions .btn { max-width:100%; white-space:normal; }
          }
          @media (prefers-reduced-motion:reduce) {
            #stemtube *, #stemtube *::after { animation-duration:0.01ms !important; animation-iteration-count:1 !important; transition-duration:0.01ms !important; }
          }
        </style>

        <div id="stemtube" data-theme="dark">
          <div class="aurora"><i></i><i></i><i></i></div><div class="grain"></div>
          <svg width="0" height="0" style="position:absolute"><defs><linearGradient id="stGrad" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#ff4f7b"/><stop offset=".6" stop-color="#7c5cff"/><stop offset="1" stop-color="#22d3ee"/></linearGradient></defs></svg>

          <header class="topbar">
            <div class="brand">
              <div class="logo"><svg viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="M3 10v4M6.5 7v10M10 4v16"/><path d="M14 7.5v9l7-4.5z" fill="#fff" stroke="none"/></svg></div>
              <div><h1>Stem<span>Tube</span> <span class="ver" id="verBadge">Studio</span></h1><p>Baixe, recorte, separe e mixe áudio e vídeo do YouTube</p></div>
            </div>
            <div class="status">
              <div class="chip" id="chipGpu"><i class="dot"></i><span>Verificando GPU…</span></div>
              <div class="chip clickable" id="chipCookies" title="Cookies da sua conta"><i class="dot"></i><span>Cookies</span></div>
              <div class="chip" id="chipDrive"><i class="dot"></i><span>Drive</span></div>
            </div>
            <div class="user"><div class="avatar" id="avatar">?</div><span id="userName">Conta Google</span></div>
            <button class="icon-btn" id="themeBtn" title="Alternar tema claro/escuro" aria-label="Alternar tema">☾</button>
          </header>

          <nav class="stepper" aria-label="Etapas">
            <div class="step active" data-go="1"><span class="n">1</span><span class="lbl">Fonte</span></div>
            <div class="step" data-go="2"><span class="n">2</span><span class="lbl">Prévia</span></div>
            <div class="step" data-go="3"><span class="n">3</span><span class="lbl">Opções</span></div>
            <div class="step" data-go="4"><span class="n">4</span><span class="lbl">Processar</span></div>
            <div class="step" data-go="5"><span class="n">5</span><span class="lbl">Resultado</span></div>
          </nav>

          <div class="queuebar" id="queueBar" hidden>
            <div class="qhead">🎼 Suas tarefas <span class="badge" id="queueCount"></span><span class="small muted" id="queueHint" style="font-weight:500"></span></div>
            <div class="qlist" id="queueList"></div>
          </div>

          <main>
            <!-- ===== 1: FONTE ===== -->
            <section data-step="1" class="on">
              <div class="card hero">
                <div class="eq" id="eq"></div>
                <h2>Baixe, recorte, <em>separe e mixe</em><br>tudo em um só lugar</h2>
                <p class="sub">Cole um link do YouTube (vídeo, Shorts, playlist ou canal) ou envie um arquivo do computador. Baixe em MP4 ou áudio, recorte um trecho ou deixe a inteligência artificial separar voz e instrumentos para ouvir e mixar aqui mesmo.</p>
                <div class="tabs"><button class="tab sel" data-src="yt">▶ Link do YouTube</button><button class="tab" data-src="file">📁 Arquivo do computador</button></div>
                <div class="src-pane on" data-pane="yt">
                  <div class="input-row">
                    <div class="field"><svg viewBox="0 0 24 24"><path d="M3.9 12a5 5 0 0 1 5-5h4v2h-4a3 3 0 0 0 0 6h4v2h-4a5 5 0 0 1-5-5zm7.1-1h2v2h-2v-2zm4-4h4a5 5 0 1 1 0 10h-4v-2h4a3 3 0 1 0 0-6h-4V7z"/></svg><input type="text" id="urlInput" aria-label="Link do YouTube" placeholder="https://www.youtube.com/watch?v=…  ou  …/playlist?list=…" autocomplete="off" spellcheck="false"></div>
                    <button class="btn" id="pasteBtn" title="Colar da área de transferência">📋 Colar</button>
                    <button class="btn primary" id="analyzeBtn">Analisar →</button>
                  </div>
                  <div class="linkmode">
                    <span class="lm-label">O que você quer usar deste link?</span>
                    <div class="chips">
                      <button class="chipbtn sel" data-lm="auto" title="Eu analiso o link: se tiver vídeo e playlist, pergunto qual você quer">✨ Detectar automaticamente</button>
                      <button class="chipbtn" data-lm="video" title="Ignora a playlist e usa só o vídeo do link">🎬 Somente o vídeo</button>
                      <button class="chipbtn" data-lm="playlist" title="Usa a playlist (list=) ou o canal do link e ignora o vídeo">📃 Playlist completa</button>
                    </div>
                  </div>
                  <p class="hint">Funciona com links de vídeo, <b>YouTube Music</b>, Shorts, youtu.be, playlists (<b>list=</b> no endereço) e canais (aba Vídeos). Mixes automáticos (list=RD…) só permitem baixar o vídeo.</p>
                </div>
                <div class="src-pane" data-pane="file">
                  <div class="drop" id="drop">
                    <div class="big">🎧</div><b>Arraste um arquivo de áudio ou vídeo aqui, ou clique para escolher</b>
                    <span>MP3, WAV, FLAC, M4A, OGG, OPUS, AIFF ou vídeo MP4/WEBM/MKV/MOV · até <b id="maxUp">1 GB</b> · para separar faixas com IA</span>
                    <input type="file" id="fileInput" accept="audio/*,video/mp4,video/webm,.mp3,.wav,.flac,.m4a,.aac,.ogg,.opus,.aiff,.aif,.mp4,.webm,.mkv,.mov">
                  </div>
                  <div class="upbar" id="upBar"><div class="row" style="justify-content:space-between;margin-bottom:6px"><span class="small" id="upLabel">Enviando…</span><span class="small muted" id="upPct">0%</span></div><div class="bar striped"><i id="upFill"></i></div></div>
                </div>
              </div>

              <div class="feats">
                <div class="feat" style="--c:#ff4f7b"><div class="fi">🎬</div><div><b>Vídeo em MP4 ou só o áudio</b>Qualquer resolução disponível (até 8K) ou MP3, M4A, OPUS, FLAC, WAV e original. Playlists inteiras com seleção item a item.</div></div>
                <div class="feat" style="--c:#7c5cff"><div class="fi">🎧</div><div><b>Separação de faixas com IA</b>Voz + playback, banda (4 faixas) ou completo (6 faixas). Ouça no mixer, silencie, isole e ajuste volumes antes de baixar.</div></div>
                <div class="feat" style="--c:#22d3ee"><div class="fi">✂️</div><div><b>Recorte, mixagem e Drive</b>Baixe só um trecho, renderize sua própria mixagem, receba tudo em ZIP e guarde cópias no Google Drive.</div></div>
              </div>

              <details class="card" id="cookiesCard">
                <summary>🔐 Cookies da sua conta <span class="badge" id="cookiesBadge">verificando…</span><span class="chev">▾</span></summary>
                <div class="body">
                  <div class="grid2">
                    <div>
                      <p class="sub" style="margin-bottom:12px">Só é preciso para vídeos <b>privados</b>, <b>+18</b>, <b>só para membros</b> ou quando o YouTube pedir “confirme que você não é um robô” (comum em servidores). Vídeos públicos funcionam sem isso.</p>
                      <div class="row">
                        <label class="btn primary" for="cookiesFile">📂 Carregar cookies.txt</label><input type="file" id="cookiesFile" accept=".txt,.json,text/plain,application/json" class="sr">
                        <button class="btn" id="cookiesPasteBtn">📋 Colar conteúdo</button>
                        <button class="btn" id="cookiesTestBtn">🧪 Testar</button>
                        <button class="btn ghost" id="cookiesClearBtn">🗑 Remover</button>
                      </div>
                      <div id="cookiesPasteBox" style="display:none;margin-top:12px"><textarea id="cookiesText" placeholder="Cole aqui o conteúdo do cookies.txt (formato Netscape), o JSON exportado pela extensão ou o cabeçalho Cookie: …"></textarea><div class="row" style="margin-top:8px"><button class="btn sm primary" id="cookiesSaveTextBtn">Salvar</button><button class="btn sm ghost" id="cookiesCancelTextBtn">Cancelar</button></div></div>
                      <div class="note" id="cookiesStatus">Nenhum cookie carregado.</div>
                      <div class="note" id="cookiesDriveNote" style="display:none"></div>
                    </div>
                    <div>
                      <h3>Como exportar (leva 1 minuto)</h3>
                      <ol class="steps-help">
                        <li><span>Instale a extensão <a href="https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc" target="_blank" rel="noopener">Get cookies.txt LOCALLY</a> (Chrome/Edge) ou <a href="https://addons.mozilla.org/firefox/addon/cookies-txt/" target="_blank" rel="noopener">cookies.txt</a> (Firefox).</span></li>
                        <li><span>Abra <b>youtube.com</b> logado na conta que tem acesso aos vídeos.</span></li>
                        <li><span>Clique na extensão → <b>Export</b> (formato Netscape) e salve o arquivo.</span></li>
                        <li><span>Volte aqui e use <b>Carregar cookies.txt</b>. Com o Drive conectado ele fica salvo para as próximas vezes.</span></li>
                      </ol>
                      <div class="note warn">O Colab roda em um servidor do Google e não consegue ler os cookies do seu navegador sozinho, por isso a exportação é feita uma única vez. Cookies dão acesso à sua conta: nunca compartilhe o arquivo.</div>
                    </div>
                  </div>
                </div>
              </details>
            </section>

            <!-- ===== 2: PRÉVIA ===== -->
            <section data-step="2">
              <div id="previewArea"></div>
              <div class="actions"><button class="btn ghost" data-back="1">← Trocar fonte</button><div class="right"><button class="btn primary" id="toOptionsBtn" disabled>Continuar →</button></div></div>
            </section>

            <!-- ===== 3: OPÇÕES ===== -->
            <section data-step="3">
              <div class="card">
                <h2>O que você quer fazer?</h2>
                <p class="sub" id="actionSub">Escolha uma opção. Você ajusta os detalhes logo abaixo.</p>
                <div class="choices" id="actionChoices"></div>
                <div class="note warn" id="sepUnavailableNote" style="display:none"></div>
              </div>

              <div id="optVideo" style="display:none"><div class="card" id="videoQualityCard"></div></div>
              <div id="optAudio" style="display:none"><div class="card" id="audioFormatCard"></div></div>
              <div id="optSep" style="display:none">
                <div class="card">
                  <h2>Como separar?</h2>
                  <p class="sub">Quanto mais faixas, mais tempo leva. O tempo estimado considera a duração da música e a GPU do Colab.</p>
                  <div class="choices" id="modeChoices"></div>
                  <div class="note warn" id="gpuNote" style="display:none"></div>
                  <div class="note info" id="modelNote" style="display:none"></div>
                </div>
                <div class="card">
                  <h3>Formato das faixas</h3>
                  <div class="chips" id="fmtChips">
                    <button class="chipbtn sel" data-fmt="wav">WAV<small>sem perda · maior</small></button>
                    <button class="chipbtn" data-fmt="flac">FLAC<small>sem perda · compacto</small></button>
                    <button class="chipbtn" data-fmt="mp3">MP3 320<small>menor · universal</small></button>
                  </div>
                </div>
              </div>

              <div class="card" id="extrasCard">
                <h3>Extras</h3>
                <div class="toggle" id="metaToggle"><div><div class="tt">Incluir capa e metadados</div><div class="td">Grava título, canal e a miniatura como capa no arquivo (áudio) e metadados no MP4.</div></div><div class="sw on"></div></div>
                <div class="toggle" id="trimToggle"><div><div class="tt">✂️ Baixar só um trecho</div><div class="td">Escolha início e fim. O arquivo completo é baixado e cortado em seguida, com recodificação para corte preciso. No formato de áudio original, o corte é aproximado por pacotes. Em vídeo, o trecho é recodificado (trechos longos em 4K demoram mais). O trecho não recebe capa embutida.</div></div><div class="sw"></div></div>
                <div id="trimBox" style="display:none;margin-top:4px">
                  <div class="row" style="align-items:flex-end">
                    <div><label class="small muted" for="trimStart">Início</label><input type="text" class="plain" id="trimStart" placeholder="0:00" inputmode="numeric" style="width:130px;margin-top:6px"></div>
                    <div><label class="small muted" for="trimEnd">Fim</label><input type="text" class="plain" id="trimEnd" placeholder="mm:ss" inputmode="numeric" style="width:130px;margin-top:6px"></div>
                    <div class="small" id="trimInfo" style="padding-bottom:12px"></div>
                  </div>
                  <p class="hint" style="margin-top:8px">Formatos aceitos: <b>1:23</b> (min:seg), <b>1:02:03</b> (h:min:seg) ou só segundos. Deixe o fim vazio para ir até o final. Duração do vídeo: <b id="trimTotal">—</b>.</p>
                </div>
              </div>

              <div class="card">
                <h3>Destino</h3>
                <div class="toggle off"><div><div class="tt">Download pelo navegador</div><div class="td">Sempre ativo: ao terminar, você recebe um botão “Baixar” para cada arquivo e um ZIP com tudo.</div></div><div class="sw on"></div></div>
                <div class="toggle" id="driveToggle"><div><div class="tt">Também salvar no Google Drive</div><div class="td" id="driveToggleDesc">Cópia em Meu Drive / StemTube.</div></div><div class="sw"></div></div>
                <div id="folderBox" style="display:none;margin-top:4px">
                  <label class="small muted" for="folderInput">Nome da subpasta no Drive (opcional)</label>
                  <input type="text" class="plain" id="folderInput" placeholder="ex.: Aulas de violão" maxlength="80" style="margin-top:6px">
                </div>
              </div>

              <div class="card">
                <h3>Resumo</h3>
                <div class="summary" id="summaryGrid"></div>
                <div class="mini-list" id="summaryList"></div>
              </div>
              <div class="actions"><button class="btn ghost" data-back="2">← Voltar</button><div class="right"><button class="btn primary lg" id="startBtn">✨ Começar</button></div></div>
            </section>

            <!-- ===== 4: PROCESSAR ===== -->
            <section data-step="4">
              <div class="card" id="procCard"></div>
              <div class="actions" id="procActions"></div>
            </section>

            <!-- ===== 5: RESULTADO ===== -->
            <section data-step="5">
              <div class="card" id="resultCard" style="position:relative;overflow:hidden"></div>
              <div class="actions" id="resultActions"></div>
            </section>
          </main>

          <footer>
            <span>Uso pessoal · respeite os termos do YouTube e os direitos dos criadores.</span>
            <span class="signature">Desenvolvido por <b>@emersonms</b> - 2026</span>
            <span id="footInfo"></span>
          </footer>

          <div class="toasts" id="toasts"></div>
          <div class="overlay" id="overlay"><div class="modal" id="modal"></div></div>
        </div>

        <script>
        (function(){
          'use strict';
          const root = document.getElementById('stemtube');
          const $ = (sel, el=root) => el.querySelector(sel);
          const $$ = (sel, el=root) => Array.from(el.querySelectorAll(sel));
          const session = window.stemTubeSession;
          const IS_COLAB = !!session || !!(window.google && google.colab && google.colab.kernel);

          function showFatal(msg){
            let box = document.getElementById('stFatal');
            if(!box){
              box = document.createElement('div'); box.id = 'stFatal';
              box.style.cssText = 'position:relative;z-index:50;margin:16px 22px 0;padding:14px 16px;border-radius:12px;border-left:4px solid #ef4444;background:rgba(239,68,68,.12);font-size:13px;white-space:pre-wrap;word-break:break-word;font-family:ui-monospace,Consolas,monospace';
              root.insertBefore(box, root.querySelector('nav'));
            }
            box.textContent = '⚠️ Erro na interface (copie e envie esta mensagem):\n' + msg;
          }
          window.addEventListener('error', e => showFatal((e.message || 'erro') + (e.filename ? `\n${e.filename}:${e.lineno}` : '')));
          window.addEventListener('unhandledrejection', e => showFatal(String(e.reason && (e.reason.stack || e.reason.message) || e.reason)));

          // ---------- estado ----------
          const S = { cfg:{}, srcTab:'yt', linkMode:'auto', analysis:null, choice:null, selected:new Set(), source:null, playerOn:false,
                      action:null, height:null, audioFormat:'mp3', audioQuality:'best', embedMeta:true, trim:{on:false},
                      mode:'2stem', fmt:'wav', saveDrive:false, jobId:null, timer:null, step:1, maxStep:1, mixer:null, knownJobs:{}, queueTimer:null, modelTimer:null };
          const STD_RES = [2160, 1440, 1080, 720, 480, 360];
          const RES_NAMES = {4320:'8K', 2160:'4K UHD', 1440:'2K QHD', 1080:'Full HD', 720:'HD', 480:'SD', 360:'Baixa', 240:'Muito baixa', 144:'Mínima'};
          const MODE_INFO = {
            '2stem': {ic:'🎤', t:'Voz + Playback', d:'Ideal para karaokê, estudar canto ou tirar a voz. Usa o BS-Roformer, o modelo mais preciso para voz.', tag:'Mais usado'},
            '4stem': {ic:'🥁', t:'Banda (4 faixas)', d:'Voz, bateria, baixo e o resto dos instrumentos. Usa o Demucs afinado, excelente em bateria e baixo.', tag:'Mais lento'},
            '6stem': {ic:'🎸', t:'Completo (6 faixas)', d:'Voz, bateria, baixo, guitarra, piano e outros. Guitarra e piano podem vazar um pouco entre si.', tag:''},
          };
          const AUDIO_FORMATS = [
            {id:'mp3',  ic:'🎵', t:'MP3',  d:'Universal. Toca em qualquer lugar.', q:true},
            {id:'m4a',  ic:'🍎', t:'M4A / AAC', d:'Ótima qualidade, ideal para Apple e celulares.', q:true},
            {id:'opus', ic:'🪶', t:'OPUS', d:'Melhor compressão. Arquivos menores.', q:true},
            {id:'flac', ic:'💎', t:'FLAC', d:'Sem perdas (a partir da fonte). Arquivo grande.', q:false},
            {id:'wav',  ic:'🎛', t:'WAV',  d:'Sem compressão. Para edição.', q:false},
            {id:'best', ic:'⚡', t:'Original', d:'Sem conversão: o áudio exatamente como o YouTube entrega (m4a/webm). Mais rápido.', q:false},
          ];
          const AUDIO_Q = [{id:'best', t:'Melhor', d:'VBR máxima'}, {id:'320', t:'320 kbps'}, {id:'256', t:'256 kbps'}, {id:'192', t:'192 kbps'}, {id:'128', t:'128 kbps'}, {id:'96', t:'96 kbps'}];
          const ACTIONS = {
            video:    {ic:'🎬', t:'Baixar vídeo em MP4', d:'Vídeo + áudio, compatível com qualquer aparelho. Você escolhe a resolução.'},
            audio:    {ic:'🎵', t:'Baixar só o áudio', d:'MP3, M4A, OPUS, FLAC, WAV ou o áudio original, na qualidade que preferir.'},
            separate: {ic:'🎧', t:'Separar faixas com IA', d:'Voz, playback, bateria, baixo e mais. Ouça no mixer, renderize mixagens e baixe em WAV, FLAC ou MP3.'},
          };

          // ---------- ponte com o Python ----------
          async function api(name, ...args){
            if(!IS_COLAB){
              if(window.__mockApi) return window.__mockApi(name, ...args);
              throw new Error('Este app precisa ser executado dentro do Google Colab.');
            }
            if(session && !session.connected()) throw new Error('A conexão com a célula foi encerrada. Reabra o Studio pela célula 2.');
            const r = session ? await session.invoke(name, args) : await google.colab.kernel.invokeFunction('stemtube.' + name, args, {});
            let d = r && r.data && (r.data['application/json'] ?? r.data['text/plain']);
            if(typeof d === 'string'){ try { d = JSON.parse(d); } catch(e){ d = {ok:false, error:d}; } }
            return d || {ok:false, error:'Sem resposta do Python. Execute a célula do app novamente.'};
          }
          let _rzT = null;
          function resize(){
            if(!IS_COLAB || session) return;
            clearTimeout(_rzT);
            _rzT = setTimeout(() => { try { const o = google.colab.output; if(o && typeof o.resizeIframeToContent === 'function') o.resizeIframeToContent(); } catch(e){} }, 60);
          }
          async function fileUrl(rel){
            const path = rel.split('/').map(encodeURIComponent).join('/');
            if(IS_COLAB){ const base = session ? await session.mediaBase(S.cfg.port) : await google.colab.kernel.proxyPort(S.cfg.port, {cache:true}); return base.replace(/\/?$/, '/') + path; }
            return 'http://127.0.0.1:' + S.cfg.port + '/' + path;
          }
          async function downloadRel(rel, btn){
            if(!IS_COLAB){ window.open(await fileUrl(rel), '_blank'); return; }
            try{
              if(btn) busy(btn, true, 'Enviando…');
              toast('Preparando o download… o navegador vai salvar o arquivo em alguns segundos.', 'info', 6000);
              const r = await api('colab_download', rel);
              if(!r.ok) throw new Error(r.error);
              toast('Download enviado ao navegador.', 'ok', 4000);
            }catch(e){ toast('Não consegui iniciar o download: ' + e.message, 'err', 7000); }
            finally{ if(btn) busy(btn, false); }
          }

          // ---------- utilidades ----------
          function esc(s){ return String(s ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c])); }
          function fmtDur(s){ if(s==null) return '—'; s=Math.round(s); const h=Math.floor(s/3600), m=Math.floor(s%3600/60), x=s%60; return (h? h+':'+String(m).padStart(2,'0') : m) + ':' + String(x).padStart(2,'0'); }
          function fmtLong(s){ if(!s) return '—'; const h=Math.floor(s/3600), m=Math.round(s%3600/60); return h ? `${h} h ${m} min` : `${m} min`; }
          function fmtNum(n){ if(n==null) return '—'; return new Intl.NumberFormat('pt-BR', {notation:'compact', maximumFractionDigits:1}).format(n); }
          function fmtDate(d){ if(!d || d.length!==8) return '—'; return `${d.slice(6,8)}/${d.slice(4,6)}/${d.slice(0,4)}`; }
          function fmtBytes(b){ if(!b) return '—'; const u=['B','KB','MB','GB']; let i=0; while(b>=1024 && i<u.length-1){ b/=1024; i++; } return b.toFixed(i?1:0)+' '+u[i]; }
          function fmtEta(sec){ sec = Math.round(sec); if(sec < 60) return `${sec} s`; const m = Math.floor(sec/60), s = sec%60; return s ? `${m} min ${s} s` : `${m} min`; }
          function toast(msg, type='info', ms=4200){
            const t = document.createElement('div'); t.className = 'toast '+type;
            t.innerHTML = `<span>${{ok:'✅', err:'⚠️', info:'ℹ️', warn:'⚠️'}[type]||''}</span><span>${esc(msg)}</span>`;
            $('#toasts').appendChild(t); setTimeout(()=> t.remove(), ms);
          }
          function confirmModal({title, text, ok='Confirmar', cancel='Cancelar', danger=false}){
            return new Promise(res => {
              const m = $('#modal'); m.innerHTML = `<h3>${esc(title)}</h3><p>${text}</p><div class="actions"><button class="btn ghost" data-r="0">${esc(cancel)}</button><button class="btn ${danger?'danger':'primary'}" data-r="1">${esc(ok)}</button></div>`;
              $('#overlay').classList.add('on');
              m.onclick = e => { const b = e.target.closest('[data-r]'); if(!b) return; $('#overlay').classList.remove('on'); res(b.dataset.r === '1'); };
            });
          }
          function busy(btn, on, label){
            if(!btn) return; if(on){ btn.dataset.label = btn.innerHTML; btn.innerHTML = `<i class="spin"></i> ${esc(label||'Aguarde…')}`; btn.disabled = true; }
            else { btn.innerHTML = btn.dataset.label || btn.innerHTML; btn.disabled = false; }
          }
          function goto(n){
            S.step = n; S.maxStep = Math.max(S.maxStep, n);
            $$('section[data-step]').forEach(s => s.classList.toggle('on', +s.dataset.step === n));
            $$('.step').forEach(st => { const k = +st.dataset.go; st.classList.toggle('active', k===n); st.classList.toggle('done', k<n); });
            if(n !== 5) (S.resultPlayers || []).forEach(p => p.pause());
            if(n !== 5 && S.mixer) S.mixer.pause();
            if(n !== 3) stopWatchModels();
            root.scrollIntoView({behavior:'smooth', block:'start'}); resize();
          }
          $$('.step').forEach(st => st.addEventListener('click', () => { const k=+st.dataset.go; if(k < S.step && k <= 3 && (k === 1 || S.source || S.analysis)) goto(k); }));
          $$('[data-back]').forEach(b => b.addEventListener('click', () => goto(+b.dataset.back)));
          new ResizeObserver(() => resize()).observe(root);

          // equalizador decorativo
          (function(){ const eq = $('#eq'); for(let i=0;i<28;i++){ const b=document.createElement('i'); b.style.animationDelay = (-(i*0.13)%1.4)+'s'; b.style.animationDuration = (1.1 + (i%5)*0.18)+'s'; eq.appendChild(b);} })();

          // tema
          const themeBtn = $('#themeBtn');
          function setTheme(t){ root.dataset.theme = t; themeBtn.textContent = t==='dark' ? '☾' : '☀'; try{ localStorage.setItem('stemtube-theme', t);}catch(e){} }
          themeBtn.addEventListener('click', () => setTheme(root.dataset.theme==='dark' ? 'light' : 'dark'));
          try{ const t = localStorage.getItem('stemtube-theme'); if(t) setTheme(t); }catch(e){}

          // ---------- bootstrap ----------
          function renderCookies(c){
            const badge = $('#cookiesBadge'), chip = $('#chipCookies'), st = $('#cookiesStatus');
            if(!c || !c.found){
              badge.textContent = 'não carregados'; badge.className = 'badge';
              chip.className = 'chip clickable'; chip.querySelector('span').textContent = 'Cookies: não carregados';
              st.className = 'note'; st.innerHTML = 'Nenhum cookie carregado. Vídeos públicos funcionam normalmente. Se o YouTube pedir “confirme que você não é um robô”, carregue os cookies da sua conta.';
            } else {
              const ok = c.logged_in;
              badge.textContent = ok ? 'conta reconhecida' : 'carregados'; badge.className = 'badge ' + (ok?'ok':'warn');
              chip.className = 'chip ' + (ok?'ok':'warn') + ' clickable'; chip.querySelector('span').textContent = ok ? 'Cookies: conta reconhecida' : 'Cookies: sem login';
              st.className = 'note'; st.innerHTML = `<b>${c.total}</b> cookies do YouTube carregados ${c.source==='drive' ? 'automaticamente do seu <b>Drive</b>' : 'nesta sessão'} (atualizados em ${esc(c.updated)}). ` + (ok ? 'Os cookies de login da sua conta estão presentes.' : '<span style="color:var(--warn)">Não encontrei cookies de login: exporte novamente com o YouTube logado.</span>') + (c.saved_drive ? ' Salvo no Drive para as próximas vezes.' : '');
            }
          }
          function renderGpu(g){
            const chip = $('#chipGpu');
            if(g && g.available){ chip.className = 'chip ok'; chip.querySelector('span').textContent = `GPU ${g.name.replace('Tesla ','').replace('NVIDIA ','')} · ${g.memory_gb} GB`; chip.title = 'GPU pronta para separar faixas'; }
            else if(g && g.mock){ chip.className = 'chip info'; chip.querySelector('span').textContent = 'Modo de teste (sem IA)'; }
            else { chip.className = 'chip warn clickable'; chip.querySelector('span').textContent = 'Sem GPU'; chip.title = 'Clique para saber como ativar a GPU'; chip.onclick = () => confirmModal({title:'Ativar a GPU no Colab', text:'A separação de faixas precisa de GPU. No menu do Colab, vá em <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução</b>, escolha <b>T4 GPU</b> e salve. Depois rode as células 1 e 2 novamente.<br><br>Sem GPU o app continua baixando vídeos e áudio normalmente; só a separação fica lenta (roda no processador).', ok:'Entendi', cancel:'Fechar'}); }
          }
          async function bootstrap(){
            const r = await api('bootstrap');
            if(!r.ok){ toast('Falha ao iniciar: ' + r.error, 'err'); return; }
            S.cfg = r;
            const name = r.user || 'Conta Google'; $('#userName').textContent = name; $('#avatar').textContent = (name[0]||'?').toUpperCase();
            $('#verBadge').textContent = 'v' + (r.version || '1.0');
            $('#chipDrive').className = 'chip ' + (r.drive ? 'ok' : ''); $('#chipDrive span').textContent = r.drive ? 'Drive conectado' : 'Drive não conectado';
            $('#footInfo').textContent = `${r.app || 'StemTube'} ${r.version || ''} · yt-dlp ${r.ytdlp}` + (r.js_runtime ? ` · ${r.js_runtime}` : '');
            $('#maxUp').textContent = r.max_upload_mb >= 1024 ? `${r.max_upload_mb / 1024} GB` : `${r.max_upload_mb} MB`;
            if(!r.drive){ const n = $('#cookiesDriveNote'); n.style.display='block'; n.className = 'note warn'; n.innerHTML = 'O Google Drive não foi conectado na célula 1, então os cookies valem só para esta sessão. Conecte o Drive para que eles sejam carregados automaticamente nas próximas vezes.'; }
            renderCookies(r.cookies); renderGpu(r.gpu); renderModeChoices();
            if(r.cookies_migrated_from) toast(`Cookies recuperados da pasta ${r.cookies_migrated_from} do seu Drive e copiados para Meu Drive / StemTube.`, 'ok', 7000);
            if(!r.js_runtime) toast('Nenhum runtime JavaScript (deno) detectado: alguns vídeos podem falhar. Rode a célula 1 novamente.', 'warn', 7000);
            if(!r.separator) toast('Separador de IA não instalado: apenas downloads estão disponíveis. Para separar faixas, rode a célula 1 com "instalar_separador_ia" marcado.', 'warn', 9000);
            resize();
            try { startQueuePolling(); } catch(e){}
          }

          // ---------- cookies ----------
          $('#chipCookies').addEventListener('click', () => { if(S.step !== 1) goto(1); $('#cookiesCard').open = true; $('#cookiesCard').scrollIntoView({behavior:'smooth'}); });
          $('#cookiesFile').addEventListener('change', async e => { const f = e.target.files[0]; if(!f) return; const text = await f.text(); e.target.value = ''; await saveCookies(text); });
          $('#cookiesPasteBtn').addEventListener('click', () => { $('#cookiesPasteBox').style.display = 'block'; $('#cookiesText').focus(); resize(); });
          $('#cookiesCancelTextBtn').addEventListener('click', () => { $('#cookiesPasteBox').style.display = 'none'; resize(); });
          $('#cookiesSaveTextBtn').addEventListener('click', async e => { const t = $('#cookiesText').value; if(!t.trim()) return toast('Cole o conteúdo dos cookies primeiro.', 'warn'); busy(e.target, true, 'Salvando'); await saveCookies(t); busy(e.target, false); $('#cookiesPasteBox').style.display='none'; $('#cookiesText').value=''; });
          async function saveCookies(text){
            const r = await api('save_cookies', text, true);
            if(!r.ok) return toast(r.error, 'err', 7000);
            renderCookies(r.cookies); toast(r.cookies.saved_drive ? 'Cookies salvos e guardados no Drive.' : 'Cookies salvos para esta sessão.', 'ok');
          }
          $('#cookiesTestBtn').addEventListener('click', async e => { busy(e.target, true, 'Testando'); const r = await api('test_cookies'); busy(e.target, false); if(!r.ok) return toast(r.error, 'err'); toast(r.message, r.valid ? 'ok' : 'warn', 7000); });
          $('#cookiesClearBtn').addEventListener('click', async () => {
            if(!await confirmModal({title:'Remover cookies?', text:'Os cookies serão apagados desta sessão e do seu Drive. Você pode carregá-los novamente depois.', ok:'Remover', danger:true})) return;
            const r = await api('clear_cookies', true); if(r.ok){ renderCookies(r.cookies); toast('Cookies removidos.', 'ok'); }
          });

          // ---------- etapa 1: fonte ----------
          $$('.tab[data-src]').forEach(t => t.addEventListener('click', () => { S.srcTab = t.dataset.src; $$('.tab[data-src]').forEach(x => x.classList.toggle('sel', x === t)); $$('.src-pane').forEach(p => p.classList.toggle('on', p.dataset.pane === S.srcTab)); resize(); }));
          $$('[data-lm]').forEach(b => b.addEventListener('click', () => { S.linkMode = b.dataset.lm; $$('[data-lm]').forEach(x => x.classList.toggle('sel', x === b)); }));
          $('#pasteBtn').addEventListener('click', async () => {
            try { const t = await navigator.clipboard.readText(); if(t){ $('#urlInput').value = t.trim(); toast('Link colado.', 'ok', 1800); } }
            catch(e){ toast('O navegador não permitiu ler a área de transferência aqui. Use Ctrl+V no campo.', 'warn'); $('#urlInput').focus(); }
          });
          $('#urlInput').addEventListener('keydown', e => { if(e.key==='Enter') $('#analyzeBtn').click(); });
          $('#analyzeBtn').addEventListener('click', async () => {
            const url = $('#urlInput').value.trim();
            if(!url) return toast('Cole um link do YouTube primeiro.', 'warn');
            if(!/youtu\.?be/i.test(url)) return toast('Isso não parece um link do YouTube.', 'warn');
            if(S.linkMode === 'playlist' && !/[?&]list=/i.test(url) && !/youtube\.com\/(@|channel\/|c\/|user\/)/i.test(url)) return toast('Este link não tem uma playlist (falta "list=" no endereço). Abra a playlist no YouTube e copie o link dela, ou escolha "Somente o vídeo".', 'warn', 8000);
            const btn = $('#analyzeBtn'); busy(btn, true, 'Analisando…');
            $('#previewArea').innerHTML = `<div class="card loading"><div class="lring"></div><p>Buscando informações no YouTube…<br><span class="small muted">${S.linkMode==='video' ? 'Analisando o vídeo.' : 'Playlists grandes podem levar alguns segundos.'}</span></p></div>`;
            goto(2); $('#toOptionsBtn').disabled = true;
            let r; try { r = await api('analyze', url, S.linkMode); } catch(e){ r = {ok:false, error:e.message}; }
            busy(btn, false);
            if(!r.ok){
              $('#previewArea').innerHTML = `<div class="card"><div class="result-head"><div class="big err">⚠️</div><div><h2 style="margin:0">Não consegui analisar esse link</h2><p class="sub" style="margin:4px 0 0">${esc(r.error)}</p></div></div>${/cookie/i.test(r.error) ? '<div class="note warn" style="margin-top:14px">Abra a seção <b>Cookies da sua conta</b> na etapa anterior, carregue o arquivo e tente de novo.</div>' : ''}</div>`;
              resize(); return;
            }
            resetSelection();
            S.analysis = r; S.source = null;
            S.choice = (r.video && r.playlist) ? null : (r.video ? 'video' : 'playlist');
            if(S.choice === 'playlist') r.playlist.items.filter(i => !i.unavailable).forEach(i => S.selected.add(i.id));
            renderPreview();
          });
          function resetSelection(){
            S.selected = new Set(); S.playerOn = false; S.action = null; S.height = null; S.trim = {on:false};
            $('#trimStart').value = ''; $('#trimEnd').value = ''; $('#folderInput').value = '';
          }

          // upload em pedaços
          const drop = $('#drop');
          ['dragenter','dragover'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.add('over'); }));
          ['dragleave','drop'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.remove('over'); }));
          drop.addEventListener('drop', e => { const f = e.dataTransfer.files && e.dataTransfer.files[0]; if(f) uploadFile(f); });
          $('#fileInput').addEventListener('change', e => { const f = e.target.files[0]; e.target.value=''; if(f) uploadFile(f); });
          async function uploadFile(file){
            if(file.size > (S.cfg.max_upload_mb||1024) * 1024 * 1024) return toast(`O arquivo tem ${(file.size/1048576).toFixed(0)} MB. O limite é ${S.cfg.max_upload_mb||1024} MB.`, 'err', 7000);
            if(!S.cfg.separator) return toast('O separador de IA não está instalado nesta sessão, então arquivos locais não podem ser processados. Rode a célula 1 com "instalar_separador_ia" marcado.', 'warn', 9000);
            const bar = $('#upBar'), fill = $('#upFill'), pct = $('#upPct'), lab = $('#upLabel');
            bar.classList.add('on'); fill.style.width = '0%'; pct.textContent = '0%'; lab.textContent = `Enviando ${file.name}…`; resize();
            try {
              const st = await api('upload_start', file.name, file.size);
              if(!st.ok) throw new Error(st.error);
              const CH = 768 * 1024; let sent = 0;
              for(let off = 0; off < file.size; off += CH){
                const buf = await file.slice(off, off + CH).arrayBuffer();
                let bin = ''; const bytes = new Uint8Array(buf); for(let i=0;i<bytes.length;i+=0x8000) bin += String.fromCharCode.apply(null, bytes.subarray(i, i+0x8000));
                const r = await api('upload_chunk', st.upload_id, btoa(bin));
                if(!r.ok) throw new Error(r.error);
                sent += bytes.length; const p = Math.round(sent / file.size * 100); fill.style.width = p + '%'; pct.textContent = p + '%';
              }
              lab.textContent = 'Verificando o arquivo…';
              const fin = await api('upload_finish', st.upload_id);
              if(!fin.ok) throw new Error(fin.error);
              toast('Arquivo recebido.', 'ok');
              resetSelection();
              S.analysis = null; S.source = fin.source; S.choice = 'file';
              renderPreview(); goto(2);
            } catch(e){ toast('Falha no envio: ' + e.message, 'err', 8000); }
            finally { bar.classList.remove('on'); resize(); }
          }

          // ---------- etapa 2: prévia ----------
          function renderPreview(){
            const area = $('#previewArea'); let html = '';
            if(S.choice === 'file'){
              const s = S.source;
              html = `<div class="card"><div class="preview"><div class="filecard">🎧<span class="dur" style="position:absolute;right:10px;bottom:10px;background:rgba(0,0,0,.6);color:#fff;font-size:12px;font-weight:700;padding:2px 8px;border-radius:6px">${esc(s.duration_str)}</span></div>
                <div><h3 class="vtitle">${esc(s.title)}</h3><div class="vchan">📁 Arquivo do seu computador</div><div class="meta"><span class="badge">⏱ ${esc(s.duration_str)}</span><span class="badge">📦 ${esc(s.file_size_label)}</span><span class="badge">📄 ${esc((s.file_name||'').split('.').pop().toUpperCase())}</span></div>
                ${s.too_long ? `<div class="note warn" style="margin-top:0">Áudio longo demais para separar (limite de ${Math.round((S.cfg.max_duration||1500)/60)} min). Envie um arquivo mais curto.</div>` : '<p class="small muted">Confira se é o arquivo certo e continue para escolher como separar as faixas.</p>'}</div></div></div>`;
              area.innerHTML = html; $('#toOptionsBtn').disabled = !!s.too_long; resize(); return;
            }
            const a = S.analysis;
            if(a.video && a.playlist){
              html += `<div class="card"><h2>Esse link tem um vídeo e uma playlist</h2><p class="sub">O que você quer usar?</p><div class="choices">
                <button class="choice ${S.choice==='video'?'sel':''}" data-choice="video"><span class="check">✓</span><span class="ic">🎬</span><span class="t">Somente este vídeo</span><span class="d">${esc(a.video.title)}</span></button>
                <button class="choice ${S.choice==='playlist'?'sel':''}" data-choice="playlist"><span class="check">✓</span><span class="ic">📃</span><span class="t">A playlist completa</span><span class="d">${esc(a.playlist.title)} · ${a.playlist.count} vídeos</span></button></div></div>`;
            }
            if(a.playlist_error) html += `<div class="card" style="border-color: rgba(251,191,36,.5)"><div class="row"><span style="font-size:22px">⚠️</span><div><b>A playlist deste link não pôde ser carregada</b><div class="small muted" style="margin-top:4px">${esc(a.playlist_error)} Você ainda pode usar o vídeo abaixo.</div></div></div></div>`;
            if(S.choice === 'video') html += videoCard(a.video);
            else if(S.choice === 'playlist') html += playlistCard(a.playlist);
            area.innerHTML = html;
            $$('[data-choice]', area).forEach(b => b.addEventListener('click', () => { S.choice = b.dataset.choice; S.playerOn = false; S.action = null; if(S.choice==='playlist' && !S.selected.size) a.playlist.items.filter(i=>!i.unavailable).forEach(i=>S.selected.add(i.id)); renderPreview(); }));
            const player = $('#player', area);
            if(player) player.addEventListener('click', () => { if(S.playerOn) return; S.playerOn = true; player.innerHTML = `<iframe src="https://www.youtube.com/embed/${esc(a.video.id)}?autoplay=1&rel=0" allow="autoplay; encrypted-media; picture-in-picture" allowfullscreen title="Prévia"></iframe>`; });
            const desc = $('#desc', area); if(desc){ const b = $('#descBtn', area); b.addEventListener('click', () => { desc.classList.toggle('open'); b.textContent = desc.classList.contains('open') ? 'Mostrar menos' : 'Mostrar mais'; resize(); }); if(desc.scrollHeight <= 100) b.style.display='none'; }
            if(S.choice === 'playlist') bindPlaylist();
            updateContinue(); resize();
          }
          function videoCard(v){
            const flags = [];
            if(v.is_live) flags.push('<span class="badge err">AO VIVO</span>');
            if(v.age_limit >= 18) flags.push('<span class="badge warn">+18 · precisa de cookies</span>');
            if(v.availability && v.availability !== 'public') flags.push(`<span class="badge info">${esc(v.availability)}</span>`);
            if(v.too_long) flags.push(`<span class="badge warn">longo demais para separar (limite ${Math.round((S.cfg.max_duration||1500)/60)} min)</span>`);
            const best = v.resolutions[0];
            return `<div class="card"><div class="preview">
              <div class="player" id="player" title="Clique para assistir a prévia"><img src="${esc(v.thumbnail)}" alt=""><div class="play"><span><svg viewBox="0 0 24 24"><path d="M8 5v14l11-7z"/></svg></span></div><div class="dur">${fmtDur(v.duration)}</div></div>
              <div>
                <h3 class="vtitle">${esc(v.title)}</h3>
                <div class="vchan">📺 <a href="${esc(v.channel_url||'#')}" target="_blank" rel="noopener">${esc(v.channel||'—')}</a></div>
                <div class="meta">
                  <span class="badge">👁 ${fmtNum(v.views)} views</span><span class="badge">👍 ${fmtNum(v.likes)}</span><span class="badge">📅 ${fmtDate(v.upload_date)}</span>
                  ${best ? `<span class="badge acc">até ${best.height}p${best.fps>30?best.fps:''}</span>` : ''}
                  ${v.audio_formats && v.audio_formats[0] ? `<span class="badge">🎧 até ${v.audio_formats[0].abr} kbps</span>` : ''}
                  ${flags.join('')}
                </div>
                <div class="desc" id="desc">${esc(v.description || 'Sem descrição.')}</div>
                <button class="linkbtn" id="descBtn">Mostrar mais</button>
              </div></div></div>`;
          }
          function playlistCard(p){
            const avail = p.items.filter(i=>!i.unavailable).length;
            return `<div class="card">
              <div class="plhead"><img src="${esc(p.thumbnail||'')}" alt=""><div>
                <span class="badge acc">${p.is_mix ? 'MIX AUTOMÁTICO' : 'PLAYLIST'}</span>
                <h3 class="vtitle" style="margin-top:8px">${esc(p.title)}</h3>
                ${p.is_mix ? '<div class="small muted" style="margin-bottom:6px">Mixes são gerados pelo YouTube e mudam a cada acesso. Estes são os vídeos que ele montou agora.</div>' : ''}
                <div class="vchan">📺 ${esc(p.channel||'—')}</div>
                <div class="meta" style="margin:0"><span class="badge">🎞 ${p.count} vídeos</span><span class="badge">⏱ ${fmtLong(p.total_duration)} no total</span>${avail<p.count ? `<span class="badge warn">${p.count-avail} indisponíveis</span>`:''}</div>
              </div></div>
              <div class="toolbar">
                <div class="field"><svg viewBox="0 0 24 24"><path d="M15.5 14h-.8l-.3-.3A6.5 6.5 0 1 0 14 15.5l.3.3v.8l5 5 1.5-1.5-5-5zm-6 0a4.5 4.5 0 1 1 0-9 4.5 4.5 0 0 1 0 9z"/></svg><input type="text" id="plFilter" placeholder="Filtrar por título…"></div>
                <button class="btn sm" id="selAll">Selecionar todos</button><button class="btn sm" id="selNone">Nenhum</button><button class="btn sm" id="selInv">Inverter</button>
                <span class="counter" id="selCount"></span>
              </div>
              <div class="list" id="plList">${p.items.map(itemRow).join('')}</div>
              ${p.truncated ? '<div class="note warn">Mostrando os primeiros 1000 vídeos.</div>' : ''}
            </div>`;
          }
          function itemRow(i){
            return `<div class="item ${i.unavailable?'off':''}" data-id="${esc(i.id)}" data-t="${esc((i.title||'').toLowerCase())}">
              <div class="cb ${S.selected.has(i.id)?'on':''}"></div><div class="idx">${i.index}</div><img loading="lazy" src="${esc(i.thumbnail)}" alt="">
              <div><div class="it">${esc(i.title)}</div><div class="ic">${esc(i.channel||'')}${i.unavailable?' · <span style="color:var(--warn)">indisponível</span>':''}</div></div>
              <div class="dur">${fmtDur(i.duration)}</div></div>`;
          }
          function bindPlaylist(){
            const p = S.analysis.playlist, list = $('#plList');
            const refresh = () => { $$('.item', list).forEach(el => { const on = S.selected.has(el.dataset.id); el.querySelector('.cb').classList.toggle('on', on); }); $('#selCount').textContent = `${S.selected.size} de ${p.count} selecionados`; updateContinue(); };
            list.addEventListener('click', e => { const el = e.target.closest('.item'); if(!el || el.classList.contains('off')) return; const id = el.dataset.id; S.selected.has(id) ? S.selected.delete(id) : S.selected.add(id); refresh(); });
            const visible = () => $$('.item', list).filter(el => el.style.display !== 'none' && !el.classList.contains('off'));
            $('#selAll').addEventListener('click', () => { visible().forEach(el => S.selected.add(el.dataset.id)); refresh(); });
            $('#selNone').addEventListener('click', () => { visible().forEach(el => S.selected.delete(el.dataset.id)); refresh(); });
            $('#selInv').addEventListener('click', () => { visible().forEach(el => S.selected.has(el.dataset.id) ? S.selected.delete(el.dataset.id) : S.selected.add(el.dataset.id)); refresh(); });
            $('#plFilter').addEventListener('input', e => { const q = e.target.value.toLowerCase().trim(); $$('.item', list).forEach(el => el.style.display = (!q || el.dataset.t.includes(q)) ? '' : 'none'); });
            refresh();
          }
          function updateContinue(){
            const ok = S.choice === 'video' ? !!(S.analysis && S.analysis.video && !S.analysis.video.is_live) : (S.choice === 'playlist' ? S.selected.size > 0 : (S.choice === 'file' && S.source && !S.source.too_long));
            $('#toOptionsBtn').disabled = !ok;
          }
          $('#toOptionsBtn').addEventListener('click', () => { renderActions(); goto(3); });

          // ---------- etapa 3: opções ----------
          function availableActions(){
            if(S.choice === 'file') return ['separate'];
            if(S.choice === 'playlist') return ['video', 'audio'];
            return ['video', 'audio', 'separate'];
          }
          function separateBlocked(){
            if(!S.cfg.separator) return 'O separador de IA não foi instalado nesta sessão. Rode a célula 1 com "instalar_separador_ia" marcado para habilitar.';
            const src = currentSource(); if(src && src.too_long) return `Esta música tem ${src.duration_str}. A separação aceita até ${Math.round((S.cfg.max_duration||1500)/60)} min; o download continua disponível.`;
            return null;
          }
          function currentSource(){ return S.choice === 'file' ? S.source : (S.analysis && S.analysis.video ? S.analysis.video : null); }
          function renderActions(){
            const acts = availableActions(); const blocked = separateBlocked();
            if(!S.action || !acts.includes(S.action) || (S.action === 'separate' && blocked)) S.action = acts.find(a => a !== 'separate' || !blocked) || acts[0];
            $('#actionChoices').innerHTML = acts.map(k => { const a = ACTIONS[k]; const off = k === 'separate' && blocked; const pills = k === 'separate' ? `<span class="pills"><span class="pill" style="background:#f472b6">Voz</span><span class="pill" style="background:#38bdf8">Playback</span><span class="pill" style="background:#fb923c">Bateria</span><span class="pill" style="background:#a78bfa">Baixo</span><span class="pill" style="background:#94a3b8">+</span></span>` : '';
              return `<button class="choice big ${k===S.action?'sel':''} ${off?'off':''}" data-action="${k}" ${off?'disabled':''}><span class="check">✓</span><span class="ic">${a.ic}</span><span class="t">${esc(a.t)}</span><span class="d">${esc(a.d)}</span>${pills}</button>`; }).join('');
            const n = $('#sepUnavailableNote'); if(blocked && S.choice !== 'playlist'){ n.style.display = 'block'; n.textContent = blocked; } else n.style.display = 'none';
            $('#actionSub').textContent = S.choice === 'playlist' ? `${S.selected.size} vídeos selecionados. A separação de faixas trabalha com uma música por vez.` : (S.choice === 'file' ? 'Arquivos do computador são usados para separar faixas com IA.' : 'Escolha uma opção. Você ajusta os detalhes logo abaixo.');
            renderOptions();
          }
          $('#actionChoices').addEventListener('click', e => { const c = e.target.closest('.choice'); if(!c || c.disabled) return; S.action = c.dataset.action; $$('#actionChoices .choice').forEach(x => x.classList.toggle('sel', x === c)); renderOptions(); });

          function renderOptions(){
            const act = S.action, dl = act === 'video' || act === 'audio';
            $('#optVideo').style.display = act === 'video' ? 'block' : 'none';
            $('#optAudio').style.display = act === 'audio' ? 'block' : 'none';
            $('#optSep').style.display = act === 'separate' ? 'block' : 'none';
            $('#extrasCard').style.display = dl ? 'block' : 'none';
            if(act === 'video') renderVideoQuality();
            if(act === 'audio') renderAudioFormat();
            if(act === 'separate'){ renderSepOptions(); watchModels(); } else stopWatchModels();
            // extras
            const single = S.choice === 'video';
            $('#trimToggle').style.display = single && dl ? '' : 'none'; $('#trimBox').style.display = single && dl && S.trim.on ? 'block' : 'none';
            if(single && dl){ $('#trimToggle .sw').classList.toggle('on', !!S.trim.on); $('#trimTotal').textContent = fmtDur(S.analysis.video.duration); $('#trimEnd').placeholder = fmtDur(S.analysis.video.duration); refreshTrim(); }
            $('#metaToggle .sw').classList.toggle('on', S.embedMeta);
            // destino
            const sub = act === 'separate' ? 'Faixas' : (act ? 'Downloads' : '');
            $('#driveToggleDesc').textContent = S.cfg.drive ? `Cópia em Meu Drive / StemTube / ${sub}.` : 'Indisponível: o Drive não foi conectado na célula 1.';
            if(!S.cfg.drive) S.saveDrive = false;
            $('#driveToggle .sw').classList.toggle('on', S.saveDrive); $('#folderBox').style.display = (S.saveDrive && dl) ? 'block' : 'none';
            if(S.choice === 'playlist' && !$('#folderInput').value) $('#folderInput').value = (S.analysis.playlist.title||'').slice(0,80);
            $('#startBtn').innerHTML = act === 'separate' ? '✨ Separar faixas' : (act === 'video' ? '⬇ Baixar vídeo' : '⬇ Baixar áudio');
            renderSummary(); resize();
          }
          function renderVideoQuality(){
            const single = S.choice === 'video', v = S.analysis.video, qc = $('#videoQualityCard');
            let opts;
            if(single) opts = v.resolutions.map(r => ({ id: r.height, t: `${r.height}p` + (r.fps>30 ? `<small style="font-size:12px;color:var(--muted)"> ${r.fps}fps</small>` : ''), d: [RES_NAMES[r.height], r.vcodec, r.size_label ? '~'+r.size_label : null].filter(Boolean).join(' · '), badge: r.hdr ? 'HDR' : (r.height>=2160 ? '4K' : null) }));
            else opts = [{id:'best', t:'Melhor', d:'A maior resolução de cada vídeo'}].concat(STD_RES.map(h => ({id:h, t:`${h}p`, d: RES_NAMES[h] + ' ou a mais próxima abaixo'})));
            if(!opts.length) opts = [{id:'best', t:'Melhor', d:'A maior resolução disponível'}];
            if(S.height == null || !opts.some(o => String(o.id) === String(S.height))) S.height = opts[0].id;
            qc.innerHTML = `<h2>Resolução</h2><p class="sub">${single ? 'Estas são as resoluções disponíveis para este vídeo. O tamanho é estimado.' : 'Na playlist cada vídeo pode ter resoluções diferentes: eu uso a que você escolher ou a mais próxima abaixo dela.'}</p>
              <div class="quality">${opts.map(o => `<button class="choice ${String(o.id)===String(S.height)?'sel':''}" data-q="${o.id}"><span class="t">${o.t}${o.badge?` <span class="badge acc" style="vertical-align:middle">${o.badge}</span>`:''}</span><span class="d">${esc(o.d)}</span></button>`).join('')}</div>
              <div class="note">Saída sempre em <b>.mp4</b>. Quando o YouTube só oferece a resolução em VP9/AV1 (comum em 1440p e 4K), o arquivo é empacotado em MP4 sem perda de qualidade.</div>`;
            $$('[data-q]', qc).forEach(b => b.addEventListener('click', () => { S.height = isNaN(+b.dataset.q) ? b.dataset.q : +b.dataset.q; renderVideoQuality(); renderSummary(); }));
          }
          function renderAudioFormat(){
            const f = AUDIO_FORMATS.find(x => x.id === S.audioFormat) || AUDIO_FORMATS[0], qc = $('#audioFormatCard');
            const src = S.choice === 'video' ? S.analysis.video.audio_formats : [];
            qc.innerHTML = `<h2>Formato do áudio</h2><p class="sub">Escolha o formato do arquivo final.</p>
              <div class="quality">${AUDIO_FORMATS.map(o => `<button class="choice ${o.id===S.audioFormat?'sel':''}" data-af="${o.id}"><span class="t">${o.ic} ${o.t}</span><span class="d">${esc(o.d)}</span></button>`).join('')}</div>
              ${f.q ? `<h3 style="margin-top:18px">Qualidade</h3><div class="chips">${AUDIO_Q.map(q => `<button class="chipbtn ${q.id===S.audioQuality?'sel':''}" data-aq="${q.id}">${q.t}${q.d?`<small>${esc(q.d)}</small>`:''}</button>`).join('')}</div><p class="hint">O YouTube entrega áudio de até ~130–160 kbps (ou ~256 kbps em vídeos musicais com Premium). Escolher mais que isso não melhora o som, só aumenta o arquivo.</p>` : ''}
              ${src.length ? `<h3 style="margin-top:18px">Faixas de áudio disponíveis na fonte</h3><div class="chips">${src.map(a => `<span class="badge">${esc(a.ext)} · ${esc(a.acodec)} · ${a.abr} kbps${a.size_label?' · '+a.size_label:''}${a.lang?' · '+esc(a.lang):''}</span>`).join('')}</div>` : ''}`;
            $$('[data-af]', qc).forEach(b => b.addEventListener('click', () => { S.audioFormat = b.dataset.af; renderAudioFormat(); renderSummary(); resize(); }));
            $$('[data-aq]', qc).forEach(b => b.addEventListener('click', () => { S.audioQuality = b.dataset.aq; renderAudioFormat(); renderSummary(); }));
          }
          function estSeconds(mode, dur){ const m = S.cfg.modes && S.cfg.modes[mode]; if(!m) return 0; return m.base + (dur||180) * m.factor; }
          function renderModeChoices(){
            const box = $('#modeChoices'); const modes = S.cfg.modes || {};
            box.innerHTML = Object.keys(modes).map(k => { const mi = MODE_INFO[k] || {}; const pills = modes[k].stems.map(st => `<span class="pill" style="background:${S.cfg.stem_color[st]}">${esc(S.cfg.stem_pt[st])}</span>`).join('');
              return `<button class="choice ${k===S.mode?'sel':''}" data-mode="${k}"><span class="check">✓</span>${mi.tag ? `<span class="badge tag ${k==='2stem'?'acc':''}">${esc(mi.tag)}</span>` : ''}<span class="ic">${mi.ic}</span><span class="t">${esc(mi.t)}</span><span class="d">${esc(mi.d)}</span><span class="pills">${pills}</span><span class="d" style="margin-top:8px;color:var(--muted-2)">⏱ estimado: <b data-est="${k}">—</b></span></button>`; }).join('');
          }
          $('#modeChoices').addEventListener('click', e => { const c = e.target.closest('.choice'); if(!c) return; S.mode = c.dataset.mode; $$('#modeChoices .choice').forEach(x => x.classList.toggle('sel', x === c)); renderSummary(); watchModels(); });
          $('#fmtChips').addEventListener('click', e => { const c = e.target.closest('.chipbtn'); if(!c) return; S.fmt = c.dataset.fmt; $$('#fmtChips .chipbtn').forEach(x => x.classList.toggle('sel', x === c)); renderSummary(); });
          function renderSepOptions(){
            const src = currentSource();
            $$('[data-est]').forEach(el => { el.textContent = fmtEta(estSeconds(el.dataset.est, src && src.duration)); });
            const g = S.cfg.gpu || {}; const gn = $('#gpuNote');
            if(!g.available && !g.mock){ gn.style.display = 'block'; gn.innerHTML = '<b>Sem GPU nesta sessão.</b> A separação vai rodar no processador e pode levar dezenas de minutos. Para ativar: <b>Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU</b>, depois rode as células 1 e 2 de novo.'; } else gn.style.display = 'none';
          }
          async function watchModels(){
            stopWatchModels();
            if(S.action !== 'separate') return;
            const tick = async () => {
              let r; try { r = await api('models_status'); } catch(e){ return; }
              if(!r.ok) return;
              const m = r.models[S.mode]; const note = $('#modelNote');
              if(S.step !== 3 || S.action !== 'separate'){ note.style.display = 'none'; return; }
              if(m && m.ready){ note.style.display = 'block'; note.className = 'note ok'; note.innerHTML = '✅ <b>Modelo de IA pronto</b> nesta sessão. Ele já foi baixado; só precisa ser carregado na memória.'; stopWatchModels(); }
              else { note.style.display = 'block'; note.className = 'note info'; note.innerHTML = (m && m.prefetch === 'running') ? '⏳ O modelo de IA está sendo baixado em segundo plano (uma vez por sessão). Você pode começar agora mesmo: a separação espera o download terminar.' : 'ℹ️ O modelo de IA deste modo será baixado na primeira vez (uma vez por sessão, cerca de 1 a 3 minutos).'; }
              resize();
            };
            await tick(); if(!S.modelTimer && S.step === 3 && S.action === 'separate') S.modelTimer = setInterval(tick, 5000);
          }
          function stopWatchModels(){ if(S.modelTimer){ clearInterval(S.modelTimer); S.modelTimer = null; } }

          // extras: metadados, recorte, destino
          $('#metaToggle').addEventListener('click', () => { S.embedMeta = !S.embedMeta; $('#metaToggle .sw').classList.toggle('on', S.embedMeta); renderSummary(); });
          function parseTime(s){
            s = String(s||'').trim(); if(!s) return null;
            if(/^\d+([.,]\d+)?$/.test(s)) return parseFloat(s.replace(',', '.'));
            const p = s.split(':'); if(p.length > 3 || p.some(x => !/^\d{1,2}([.,]\d+)?$/.test(x))) return null;
            return p.reduce((a, x) => a*60 + parseFloat(x.replace(',', '.')), 0);
          }
          function trimState(){
            const dur = (S.analysis && S.analysis.video && S.analysis.video.duration) || 0;
            const sRaw = $('#trimStart').value.trim(), eRaw = $('#trimEnd').value.trim();
            const st = sRaw ? parseTime(sRaw) : 0, en = eRaw ? parseTime(eRaw) : (dur || null);
            if(st == null || en == null) return {ok:false, msg: eRaw || sRaw ? 'Tempo inválido. Use mm:ss (ex.: 1:23).' : 'Informe o fim do trecho.'};
            if(dur && st >= dur) return {ok:false, msg:`O início precisa ser menor que ${fmtDur(dur)}.`};
            if(dur && en > dur + 1) return {ok:false, msg:`O fim não pode passar de ${fmtDur(dur)}.`};
            if(en <= st) return {ok:false, msg:'O fim precisa ser maior que o início.'};
            if(en - st < 1) return {ok:false, msg:'O trecho precisa ter pelo menos 1 segundo.'};
            return {ok:true, start: st, end: dur ? Math.min(en, dur) : en};
          }
          function refreshTrim(){
            const info = $('#trimInfo');
            if(!S.trim.on){ info.innerHTML = ''; renderSummary(); return; }
            const t = trimState();
            info.innerHTML = t.ok ? `<span class="muted">Trecho de <b>${fmtDur(t.start)}</b> a <b>${fmtDur(t.end)}</b> · duração <b>${fmtDur(t.end - t.start)}</b></span>` : `<span style="color:var(--err)">${esc(t.msg)}</span>`;
            renderSummary();
          }
          $('#trimToggle').addEventListener('click', () => { S.trim.on = !S.trim.on; $('#trimToggle .sw').classList.toggle('on', S.trim.on); $('#trimBox').style.display = S.trim.on ? 'block' : 'none'; if(S.trim.on) $('#trimEnd').focus(); refreshTrim(); resize(); });
          $('#trimStart').addEventListener('input', refreshTrim); $('#trimEnd').addEventListener('input', refreshTrim);
          function currentTrim(){ if(S.choice !== 'video' || !S.trim.on || S.action === 'separate') return null; const t = trimState(); return t.ok ? {start: t.start, end: t.end} : {invalid: t.msg}; }
          $('#driveToggle').addEventListener('click', () => {
            if(!S.cfg.drive) return toast('O Google Drive não foi conectado na célula 1. Rode a célula 1 com a opção de conectar o Drive marcada.', 'warn', 7000);
            S.saveDrive = !S.saveDrive; $('#driveToggle .sw').classList.toggle('on', S.saveDrive); $('#folderBox').style.display = (S.saveDrive && S.action !== 'separate') ? 'block' : 'none'; renderSummary(); resize();
          });

          function selectedItems(){
            if(S.choice === 'video'){ const v = S.analysis.video; return [{id:v.id, url:v.url, title:v.title, thumbnail:v.thumbnail}]; }
            if(S.choice === 'playlist') return S.analysis.playlist.items.filter(i => S.selected.has(i.id)).map(i => ({id:i.id, url:i.url, title:i.title, thumbnail:i.thumbnail}));
            return [];
          }
          function qualityLabel(){
            if(S.action === 'video') return S.height === 'best' ? 'Melhor resolução disponível' : `${S.height}p ${RES_NAMES[S.height]||''}`.trim();
            const f = AUDIO_FORMATS.find(x=>x.id===S.audioFormat); return f.t + (f.q ? ` · ${AUDIO_Q.find(q=>q.id===S.audioQuality).t}` : '');
          }
          function renderSummary(){
            const act = S.action; if(!act) return;
            const rows = []; const src = currentSource();
            if(act === 'separate'){
              rows.push(['Música', esc(src.title)], ['Duração', esc(src.duration_str)], ['Modo', esc(MODE_INFO[S.mode].t)], ['Faixas', esc(S.cfg.modes[S.mode].stems.map(st => S.cfg.stem_pt[st]).join(', '))], ['Formato', S.fmt.toUpperCase()], ['Tempo estimado', '≈ ' + fmtEta(estSeconds(S.mode, src.duration))]);
              $('#summaryList').innerHTML = '';
            } else {
              const items = selectedItems(); let est = null;
              if(S.choice === 'video'){ const v = S.analysis.video; if(act==='video'){ const r = v.resolutions.find(r=>r.height===S.height); est = r && r.size_label; } else { const a = v.audio_formats[0]; est = a && a.size_label ? '~'+a.size_label : null; } }
              const trim = currentTrim();
              rows.push(['Tipo', act === 'video' ? '🎬 Vídeo MP4' : '🎵 Somente áudio'], ['Qualidade', esc(qualityLabel())], ['Itens', `${items.length} ${items.length===1?'vídeo':'vídeos'}` + (S.choice==='playlist' ? ` <small>de ${S.analysis.playlist.count}</small>` : '')]);
              if(trim) rows.push(['Trecho', trim.invalid ? `<span style="color:var(--err)">inválido</span>` : `${fmtDur(trim.start)} → ${fmtDur(trim.end)} <small>(${fmtDur(trim.end - trim.start)})</small>`]);
              rows.push(['Tamanho estimado', trim ? '<small>proporcional ao trecho</small>' : (est ? esc(est) : (S.choice==='playlist' ? '<small>varia por vídeo</small>' : '—'))], ['Extras', S.embedMeta ? 'Capa e metadados' : 'Sem extras']);
              $('#summaryList').innerHTML = S.choice === 'playlist' ? items.slice(0, 50).map(i => `<div><img src="${esc(i.thumbnail)}" alt=""><span>${esc(i.title)}</span></div>`).join('') + (items.length>50 ? `<div class="muted small">… e mais ${items.length-50}</div>` : '') : '';
            }
            rows.push(['Destino', 'Navegador' + (S.saveDrive ? ' + Google Drive' : '')]);
            $('#summaryGrid').innerHTML = rows.map(([k,v]) => `<div><div class="k">${esc(k)}</div><div class="v">${v}</div></div>`).join('');
            resize();
          }

          $('#startBtn').addEventListener('click', async () => {
            const act = S.action; if(!act) return;
            let payload;
            if(act === 'separate'){
              const src = currentSource();
              payload = { kind:'separate', source_id: src.source_id || src.id, mode: S.mode, out_format: S.fmt, save_drive: S.saveDrive };
            } else {
              const items = selectedItems(); const trim = currentTrim();
              if(!items.length) return toast('Nenhum vídeo selecionado.', 'warn');
              if(trim && trim.invalid) return toast('Recorte inválido: ' + trim.invalid, 'warn', 6000);
              if(items.length > 1){
                const ok = await confirmModal({ title: 'Iniciar download?', text: `<b>${items.length}</b> arquivos em <b>${esc(qualityLabel())}</b>${S.saveDrive ? ', com cópia no Google Drive' : ''}. ${items.length > 20 ? 'Playlists grandes podem levar bastante tempo: mantenha esta aba aberta.' : 'Você poderá cancelar a qualquer momento e adicionar outros links à fila enquanto baixa.'}`, ok: 'Sim, baixar' });
                if(!ok) return;
              }
              payload = { kind:'download', items, mode: act, height: S.height || 'best', audio_format: S.audioFormat, audio_quality: S.audioQuality, embed_meta: S.embedMeta,
                          save_drive: S.saveDrive, folder: $('#folderInput').value.trim(), trim: trim ? {start: trim.start, end: trim.end} : null,
                          title: S.choice === 'playlist' ? S.analysis.playlist.title : items[0].title };
            }
            const btn = $('#startBtn'); busy(btn, true, 'Enviando…');
            try {
              const r = await api('start_job', payload);
              if(!r.ok) return toast(r.error, 'err', 9000);
              S.jobId = r.job_id; stopWatchModels();
              if(r.ahead) toast(`Adicionado à fila: ${r.ahead} tarefa(s) na frente. Você pode preparar outra enquanto isso.`, 'info', 6000);
              renderProc(null); goto(4); poll(); startQueuePolling();
            } catch(e){ toast('Erro ao iniciar: ' + e.message, 'err', 8000); }
            finally { busy(btn, false); }
          });

          // ---------- etapa 4: processamento ----------
          const RING_C = 2 * Math.PI * 84;
          function renderProc(j){
            const card = $('#procCard');
            if(!j){
              card.innerHTML = `<div class="loading"><div class="lring"></div><p>Preparando a tarefa…</p></div>`;
              $('#procActions').innerHTML = ''; resize(); return;
            }
            if(j.kind === 'download') renderDownloadProc(j); else renderStageProc(j);
            $('#procActions').innerHTML = `<button class="btn ghost" id="anotherBtn">＋ Preparar outra</button><span class="small muted">Mantenha esta aba aberta até terminar.</span><div class="right"><button class="btn danger" id="cancelBtn">✕ Cancelar</button></div>`;
            $('#cancelBtn').addEventListener('click', async e => { if(!await confirmModal({title:'Cancelar esta tarefa?', text:'O processamento será interrompido. Arquivos já concluídos continuam disponíveis; o item em andamento é descartado.', ok:'Cancelar tarefa', cancel:'Continuar', danger:true})) return; busy(e.target, true, 'Cancelando'); await api('cancel_job', j.id); });
            $('#anotherBtn').addEventListener('click', () => { resetToStart(); });
            resize();
          }
          function renderStageProc(j){
            const card = $('#procCard');
            const pct = j.status === 'done' ? 100 : Math.max(0, Math.min(100, Number(j.percent) || 0));
            const indeterminate = j.status === 'running' && j.stage === 'model' && !(j.detail||'').includes('%');
            const stages = (j.stages||[]).map(s => {
              const cls = s.status === 'running' ? 'running' : (s.status === 'done' ? 'done' : (s.status === 'error' ? 'err' : ''));
              const icon = s.status === 'done' ? '<div class="st done">✓</div>' : (s.status === 'running' ? '<div class="st run"></div>' : (s.status === 'error' ? '<div class="st err">!</div>' : (s.status === 'cancelled' ? '<div class="st">–</div>' : '<div class="st">·</div>')));
              const det = s.status === 'running' ? esc(j.stage_label || '') + (j.detail ? ' · ' + esc(j.detail) : '') : (s.status === 'done' && s.started && s.ended ? `concluído em ${fmtEta(s.ended - s.started)}` : '');
              const tm = s.status === 'running' && s.started ? fmtEta(Date.now()/1000 - s.started) : '';
              return `<div class="tl ${cls}">${icon}<div><div class="lab">${esc(s.label)}</div><div class="det">${det}</div></div><div class="tm">${tm}</div></div>`;
            }).join('');
            const title = j.status === 'done' ? 'Concluído' : j.status === 'queued' ? `Na fila${j.ahead ? ` · ${j.ahead} na frente` : ''}` : (j.kind === 'mix' ? 'Renderizando mixagem' : 'Separando as faixas');
            const est = j.kind === 'separate' && j.status === 'running' ? `Tempo estimado total: ≈ ${fmtEta(estSeconds(j.mode, j.duration))}` : '';
            const stageShort = {fetch:'obtendo áudio', model:'modelo de IA', separate:'separando', export:'exportando'}[j.stage] || (j.status === 'queued' ? 'na fila' : '');
            card.innerHTML = `<div class="proc"><div class="ringbox"><div class="ring ${indeterminate || j.status==='queued' ? 'busy' : ''}"><svg viewBox="0 0 200 200" style="animation-delay:${-(performance.now()%1600)}ms"><circle class="track" cx="100" cy="100" r="84"/><circle class="prog" cx="100" cy="100" r="84" stroke-dasharray="${RING_C}" stroke-dashoffset="${indeterminate ? RING_C*0.8 : RING_C*(1-pct/100)}"/></svg><div class="pct">${indeterminate ? '…' : Math.round(pct)+'%'}<small>${esc(stageShort)}</small></div></div><div class="stage">${esc(j.stage_label||'')}</div></div>
              <div><h2>${title}</h2><p class="sub" style="margin-bottom:12px"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · ${fmtEta(j.elapsed)} decorridos` : ''}</p>${est ? `<p class="small muted" style="margin:0 0 12px">${est}. Você pode preparar outra tarefa enquanto espera: a fila continua rodando.</p>` : ''}<div class="timeline">${stages}</div>${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}</div></div>`;
          }
          function renderDownloadProc(j){
            const card = $('#procCard');
            const items = j.items || [], total = items.length, done = items.filter(i=>['done','error','cancelled'].includes(i.status)).length;
            const cur = items.find(i => i.status==='downloading' || i.status==='processing') || items.find(i=>i.status==='queued');
            const overall = total ? ((done + (cur && cur.percent ? cur.percent/100 : 0)) / total * 100) : 0;
            const waiting = j.status === 'queued';
            const headNote = waiting ? `<p class="sub" style="margin:6px 0 0">Aguardando ${j.ahead||0} ${j.ahead===1?'download terminar':'downloads terminarem'}. Começa automaticamente. Enquanto isso, você pode preparar outro link.</p>` : '';
            card.innerHTML = `<div class="row" style="justify-content:space-between"><h2 style="margin:0">${waiting ? '⏳ Na fila…' : (j.mode === 'video' ? '🎬 Baixando vídeo…' : '🎵 Baixando áudio…')}</h2><span class="badge acc">${done} de ${total} concluídos</span></div>${headNote}
              <p class="sub" style="margin:6px 0 0"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · ${fmtEta(j.elapsed)} decorridos` : ''}</p>
              ${j.trim ? `<div class="small muted" style="margin-top:6px">✂️ Trecho de ${fmtDur(j.trim.start)} a ${fmtDur(j.trim.end)}: primeiro o arquivo completo é baixado, depois cortado.</div>` : ''}
              <div class="bar ${waiting ? '' : 'striped'}" style="margin-top:14px"><i style="width:${overall.toFixed(1)}%"></i></div>
              ${cur ? `<div class="cur"><img src="${esc(cur.thumbnail||'')}" alt=""><div>
                <div class="ct">${esc(cur.title||'')}</div><div class="cs">${esc(cur.stage||'')}</div>
                <div class="bar ${cur.status==='processing'?'ok':''}"><i style="width:${(cur.percent||0).toFixed(1)}%"></i></div>
                <div class="stats"><span>${(cur.percent||0).toFixed(0)}%</span>${cur.total?`<span><b>${fmtBytes(cur.downloaded)}</b> / ${fmtBytes(cur.total)}</span>`:''}${cur.speed?`<span>⚡ <b>${fmtBytes(cur.speed)}/s</b></span>`:''}${cur.eta!=null?`<span>⏳ ${fmtDur(cur.eta)} restantes</span>`:''}</div>
              </div></div>` : ''}
              ${total > 1 ? `<div class="jobs">${items.map(itemJobRow).join('')}</div>` : ''}${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}`;
          }
          function itemJobRow(i){
            const st = {queued:'', downloading:'run', processing:'run', done:'done', error:'err', cancelled:''}[i.status] || '';
            const icon = {done:'✓', error:'!', cancelled:'–', queued:'…'}[i.status] || '';
            const sub = i.status==='error' ? i.error : (i.status==='done' ? `${i.size_label||''}${i.warning?' · aviso: '+i.warning:''}${i.drive_path?' · salvo no Drive':''}` : i.stage);
            const act = i.status==='done' && i.rel ? `<button class="btn sm primary" data-dl="${esc(i.rel)}" title="Salvar este arquivo no seu computador">⬇ Baixar</button>` : (i.status==='downloading' && i.percent ? `<span class="badge">${i.percent.toFixed(0)}%</span>` : '');
            return `<div class="job"><div class="st ${st}">${icon}</div><div><div class="jt" title="${esc(i.title)}">${esc(i.title||i.id)}</div><div class="js ${i.status==='error'?'err':(i.warning?'warn':'')}">${esc(sub||'')}</div></div><div class="jr">${act}</div></div>`;
          }
          async function poll(){
            clearTimeout(S.timer);
            const requested = S.jobId;
            if(!requested) return;
            let j; try { j = await api('job_status', requested); } catch(e){ if(S.jobId !== requested) return; S.timer = setTimeout(poll, 2500); return; }
            if(S.jobId !== requested) return;
            if(!j.ok){ toast(j.error, 'err'); return; }
            if(j.status === 'queued' || j.status === 'running'){ if(S.step === 4 && S.jobId === j.id) renderProc(j); S.timer = setTimeout(poll, 1200); return; }
            if(j.status === 'done' && S.step === 4){
              renderProc(j);
              await new Promise(resolve => setTimeout(resolve, 500));
              if(S.jobId !== j.id) return;
            }
            if(S.step === 4 || S.step === 5){ renderResult(j); if(S.step !== 5) goto(5); }
          }

          // ---------- etapa 5: resultado + mixer ----------
          function confetti(card){
            const colors = ['#ff4f7b','#7c5cff','#22d3ee','#34d399','#fbbf24'];
            for(let i=0;i<36;i++){ const c = document.createElement('i'); c.className = 'confetti'; c.style.left = Math.random()*100+'%'; c.style.background = colors[i%colors.length]; c.style.animationDelay = (Math.random()*.6)+'s'; c.style.animationDuration = (1.4+Math.random())+'s'; card.appendChild(c); setTimeout(()=>c.remove(), 3000); }
          }
          async function renderResult(j){
            clearResultPlayers();
            const card = $('#resultCard');
            if(S.mixer){ S.mixer.destroy(); S.mixer = null; }
            const files = j.files || [];
            if(j.status !== 'done' && !files.length){
              const cancelled = j.status === 'cancelled';
              card.innerHTML = `<div class="result-head"><div class="big ${cancelled?'warn':'err'}">${cancelled?'✋':'❌'}</div><div><h2 style="margin:0">${cancelled ? 'Tarefa cancelada' : 'Não deu certo'}</h2><p class="sub" style="margin:4px 0 0">${esc(j.title||'')} · ${esc(j.label||'')}</p></div></div>${j.error ? `<div class="note err">${esc(j.error)}</div>` : ''}${(j.items||[]).length > 1 ? `<div class="jobs">${j.items.map(itemJobRow).join('')}</div>` : ''}`;
              $('#resultActions').innerHTML = `<button class="btn ghost" id="newBtn">↺ Nova tarefa</button><div class="right"><button class="btn primary" id="retryBtn">↻ Tentar novamente</button></div>`;
              $('#newBtn').addEventListener('click', resetToStart);
              $('#retryBtn').addEventListener('click', () => retryJob(j));
              resize(); return;
            }
            if(j.kind === 'download') renderDownloadResult(j); else await renderStemsResult(j);
            resize();
          }
          function retryJob(j){
            if(j.kind === 'download'){ retryFailed(j, true); return; }
            if(j.kind === 'mix'){ viewJob(j.parent_id); return; }
            const src = currentSource();
            if(src && (src.source_id || src.id) === j.source_id){ S.action = 'separate'; renderActions(); goto(3); } else resetToStart();
          }
          async function retryFailed(j, all){
            const errs = (j.items||[]).filter(i => all ? i.status !== 'done' : i.status === 'error');
            if(!errs.length) return resetToStart();
            const items = errs.map(i => ({id:i.id, url:i.url, title:i.title, thumbnail:i.thumbnail}));
            const r = await api('start_job', { kind:'download', items, mode: j.mode, height: j.height, audio_format: j.audio_format, audio_quality: j.audio_quality, embed_meta: j.embed_meta, save_drive: j.save_drive, folder: j.folder, trim: items.length === 1 ? j.trim : null, title: j.title + ' (nova tentativa)' });
            if(!r.ok) return toast(r.error, 'err'); S.jobId = r.job_id; renderProc(null); goto(4); poll(); startQueuePolling();
          }
          function driveSummary(job, files){
            if(!job.save_drive || !files.length) return '';
            const saved = files.filter(file => file.drive_path).length;
            return ` · Drive: ${saved} de ${files.length} arquivo(s) salvo(s)`;
          }
          function renderDownloadResult(j){
            const card = $('#resultCard');
            const items = j.items || [], done = items.filter(i=>i.status==='done'), errs = items.filter(i=>i.status==='error'), canc = items.filter(i=>i.status==='cancelled');
            const kind = errs.length && !done.length ? 'err' : (errs.length || canc.length ? 'warn' : 'ok');
            const title = j.status==='cancelled' ? 'Download cancelado' : (kind==='ok' ? (j.mode === 'video' ? 'Vídeo pronto!' : 'Áudio pronto!') : (kind==='warn' ? 'Concluído com avisos' : 'Nenhum arquivo foi baixado'));
            card.innerHTML = `<div class="result-head"><div class="big ${kind==='ok'?'':kind}">${kind==='ok'?'🎉':(kind==='warn'?'⚠️':'❌')}</div><div><h2 style="margin:0">${title}</h2><p class="sub" style="margin:4px 0 0"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · levou ${fmtEta(j.elapsed)}` : ''}<br>${done.length} ${done.length===1?'arquivo pronto':'arquivos prontos'}${errs.length?` · ${errs.length} com erro`:''}${canc.length?` · ${canc.length} cancelados`:''}${driveSummary(j, done)}</p></div></div>
              ${done.length ? `<div class="note info" style="margin-top:14px">Clique em <b>Baixar</b> para salvar cada arquivo no seu computador${done.length>1?', ou use <b>Baixar tudo (.zip)</b>':''}. Os arquivos ficam neste servidor só enquanto a sessão do Colab estiver ativa.</div>` : ''}
              ${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}
              <div class="jobs">${items.map(itemJobRow).join('')}</div>${previewMarkup(j.files || [], true)}`;
            const single = items.length === 1 && done.length === 1 && S.analysis && S.analysis.video && S.analysis.video.id === items[0].id && S.cfg.separator && !S.analysis.video.too_long;
            $('#resultActions').innerHTML = `<button class="btn ghost" id="newBtn">↺ Nova tarefa</button><div class="right">${single ? '<button class="btn" id="sepBtn">🎧 Separar faixas desta música</button>' : ''}${done.length>1 ? '<button class="btn" id="zipBtn">🗜 Baixar tudo (.zip)</button>' : ''}${errs.length ? '<button class="btn" id="retryBtn">↻ Tentar novamente os que falharam</button>' : ''}</div>`;
            $('#newBtn').addEventListener('click', resetToStart);
            const sb = $('#sepBtn'); if(sb) sb.addEventListener('click', () => { S.choice = 'video'; S.action = 'separate'; renderActions(); goto(3); });
            const zb = $('#zipBtn'); if(zb) zb.addEventListener('click', async () => { busy(zb, true, 'Compactando…'); const r = await api('zip_job', j.id); busy(zb, false); if(!r.ok) return toast(r.error, 'err'); toast(`ZIP pronto (${r.size_label}).`, 'ok'); downloadRel(r.rel, zb); });
            const rb = $('#retryBtn'); if(rb) rb.addEventListener('click', () => retryFailed(j, false));
            mountResultPlayers(j.files || []);
            if(kind === 'ok') confetti(card);
          }
          async function renderStemsResult(j){
            const card = $('#resultCard');
            const files = j.files || [];
            const sep = j.kind === 'separate';
            card.innerHTML = `<div class="result-head"><div class="big">🎉</div><div><h2 style="margin:0">${sep ? 'Faixas prontas!' : 'Mixagem pronta!'}</h2><p class="sub" style="margin:4px 0 0"><b>${esc(j.title||'')}</b> · ${esc(j.label||'')}${j.elapsed ? ` · levou ${fmtEta(j.elapsed)}` : ''}${driveSummary(j, files)}</p></div></div>
              ${j.warning ? `<div class="note warn">${esc(j.warning)}</div>` : ''}
              ${sep ? `<div class="note info" style="margin-top:16px">Ouça o resultado no <b>mixer</b>: silencie (M) ou isole (S) cada faixa e ajuste o volume. Baixe as faixas separadas ou use <b>Renderizar mixagem</b> para juntar as faixas audíveis em um único áudio. A faixa Original serve apenas para comparação e nunca entra na mixagem.</div>
              <div class="transport" style="margin-top:14px"><button class="playbtn" id="mxPlay" title="Tocar / pausar">▶</button><div><input type="range" class="seek" id="mxSeek" min="0" max="1000" value="0"><div class="row" style="justify-content:space-between;margin-top:4px"><span class="time" id="mxCur">0:00</span><span class="time" id="mxDur">${fmtDur(j.duration)}</span></div></div><span class="badge" id="mxState">carregando…</span></div>` : ''}
              <div class="stems" id="stemList"></div>
              ${previewMarkup(files, !sep)}
              ${sep ? `<div class="note info"><b>Mixagem personalizada</b><p id="mixSummary" aria-live="polite"></p><div class="row" style="flex-wrap:wrap;gap:12px"><label>Formato <select id="mixFormat"><option value="wav">WAV</option><option value="flac">FLAC</option><option value="mp3">MP3 · 320 kbps</option></select></label>${S.cfg.drive ? `<label><input type="checkbox" id="mixDrive" ${j.save_drive ? 'checked' : ''}> Salvar também no Drive</label>` : ''}<button class="btn primary" id="mixBtn">🎛 Renderizar mixagem</button></div><p class="small" style="margin:8px 0 0">Respeita M, Solo e volumes. O limitador protege a soma contra picos excessivos.</p></div>` : ''}`;
            const list = $('#stemList');
            const rows = files.map(f => ({...f}));
            if(sep && j.original_preview) rows.push({stem:'Original', label:'Original', icon:'💿', color:'#e2e8f0', preview:j.original_preview, rel:null, original:true});
            list.innerHTML = rows.map((f, i) => `<div class="stem ${f.original?'muted':''}" style="--c:${f.color}" data-i="${i}"><div class="ico">${f.icon}</div><div><div class="nm">${esc(f.label)}${f.original ? '<span class="badge">para comparar</span>' : `<span class="badge">${esc(f.size_label||'')}</span>`}${f.drive_path ? '<span class="badge ok">Drive</span>' : ''}</div><div class="fi">${esc(f.file || 'música original (prévia)')}</div></div><div class="ctl">${sep ? `<div class="meter"><i></i><i></i><i></i><i></i></div><button class="mbtn m ${f.original?'on':''}" data-mute="${i}" title="Silenciar">M</button><button class="mbtn s" data-solo="${i}" title="Solo (ouvir só esta)">S</button><input type="range" class="vol" data-vol="${i}" min="0" max="100" value="100">` : ''}${f.rel ? `<button class="btn sm primary" data-dl="${esc(f.rel)}">⬇ Baixar</button>` : ''}</div></div>`).join('');
            $('#resultActions').innerHTML = `<button class="btn ghost" id="newBtn">↺ Nova tarefa</button><div class="right">${sep && currentSource() && (currentSource().source_id || currentSource().id) === j.source_id ? '<button class="btn" id="againBtn">🔀 Separar de outro jeito</button>' : ''}${j.kind === 'mix' ? '<button class="btn" id="backStemsBtn">🎛 Voltar às faixas</button>' : ''}${files.length > 1 ? '<button class="btn" id="zipBtn">🗜 Baixar tudo (.zip)</button>' : ''}</div>`;
            $('#newBtn').addEventListener('click', resetToStart);
            const ab = $('#againBtn'); if(ab) ab.addEventListener('click', () => { S.action = 'separate'; renderActions(); goto(3); });
            const zb = $('#zipBtn'); if(zb) zb.addEventListener('click', async () => { busy(zb, true, 'Compactando…'); const r = await api('zip_job', j.id); busy(zb, false); if(!r.ok) return toast(r.error, 'err'); toast(`ZIP pronto (${r.size_label}).`, 'ok'); downloadRel(r.rel, zb); });
            mountResultPlayers(files);
            confetti(card); resize();
            if(sep) S.mixer = await buildMixer(rows, j.duration);
            const backStems = $('#backStemsBtn'); if(backStems) backStems.addEventListener('click', () => viewJob(j.parent_id));
            const mixBtn = $('#mixBtn'); if(mixBtn) mixBtn.addEventListener('click', async () => {
              const tracks = S.mixer ? S.mixer.selection() : [];
              if(!tracks.length) return toast('Deixe ao menos uma faixa audível, com volume acima de zero.', 'warn');
              busy(mixBtn, true, 'Preparando mixagem…');
              try {
                const r = await api('mix_job', j.id, tracks, $('#mixFormat').value, !!$('#mixDrive')?.checked);
                if(!r.ok) throw new Error(r.error);
                if(S.mixer){ S.mixer.destroy(); S.mixer = null; }
                S.jobId = r.job_id; renderProc(null); goto(4); poll(); startQueuePolling();
              } catch(e){ toast(e.message, 'err'); }
              finally { busy(mixBtn, false); }
            });
          }
          function clearResultPlayers(){
            (S.resultPlayers || []).forEach(p => p.destroy()); S.resultPlayers = [];
          }
          function previewMarkup(files, autoOpen=false){
            return `<div class="result-previews">${files.map((f,i) => {
              const video = f.preview_kind === 'video';
              if(!f.preview) return `<p class="small">Prévia indisponível para ${esc(f.label || f.file)}. Use Baixar para abrir o arquivo.</p>`;
              return `<details class="studio-preview" data-preview-index="${i}" ${autoOpen && files.length === 1 ? 'open' : ''}>
                <summary><span class="preview-icon">${video ? '▷' : '♫'}</span><span><b>${video ? 'Ver vídeo' : 'Ouvir áudio'}</b><small>${esc(f.file || f.label)}</small></span><span class="badge">PRÉVIA</span></summary>
                <div class="preview-body">
                  ${video ? '<video playsinline preload="none" aria-label="Prévia do vídeo"></video>' : '<audio preload="none" aria-label="Prévia do áudio"></audio>'}
                  <div class="preview-transport"><button class="preview-play" aria-label="Reproduzir prévia" disabled>▶</button><div class="preview-timeline"><div class="preview-time"><span data-current>0:00</span><span data-duration>0:00</span></div><input data-seek type="range" min="0" max="1000" value="0" aria-label="Posição da prévia" disabled></div></div>
                  <div class="preview-options"><label>Volume <input data-volume type="range" min="0" max="100" value="100" aria-label="Volume da prévia"></label><label>Velocidade <select data-speed aria-label="Velocidade da prévia"><option value="0.5">0,5×</option><option value="0.75">0,75×</option><option value="1" selected>1×</option><option value="1.25">1,25×</option><option value="1.5">1,5×</option><option value="2">2×</option></select></label></div>
                  <div class="preview-status" role="status" aria-live="polite">Abra para carregar a prévia.</div><button class="btn sm preview-retry" hidden>Tentar carregar novamente</button>
                  <p class="small">${video ? 'Prévia completa em até 480p. O download mantém a resolução escolhida.' : 'Prévia de escuta em MP3. O download mantém o formato escolhido.'}</p>
                </div></details>`;
            }).join('')}</div>`;
          }
          function mountResultPlayers(files){
            S.resultPlayers = [];
            $$('.studio-preview').forEach(panel => {
              const f = files[Number(panel.dataset.previewIndex)];
              const controller = mountMediaPreview(panel, f.preview);
              S.resultPlayers.push(controller);
            });
          }
          function mountMediaPreview(panel, rel){
            const audio = panel.querySelector('audio,video'), status = panel.querySelector('.preview-status');
            const retry = panel.querySelector('.preview-retry'), play = panel.querySelector('.preview-play');
            const seek = panel.querySelector('[data-seek]'), volume = panel.querySelector('[data-volume]');
            const speed = panel.querySelector('[data-speed]'), current = panel.querySelector('[data-current]');
            const duration = panel.querySelector('[data-duration]');
            let destroyed=false, generation=0, viaBridge=false, ready=false, started=false;
            let timer=null, objectUrl=null;
            const events=[];
            function listen(target, event, fn){ target.addEventListener(event,fn); events.push([target,event,fn]); }
            function release(){ clearTimeout(timer); audio.pause(); audio.removeAttribute('src'); audio.load(); if(objectUrl){ URL.revokeObjectURL(objectUrl); objectUrl=null; } }
            function fail(message){ if(destroyed) return; clearTimeout(timer); play.disabled=true; status.textContent=message+' O arquivo continua disponível no botão Baixar.'; retry.hidden=false; }
            function deadline(run){
              clearTimeout(timer);
              timer=setTimeout(() => { if(!destroyed && run===generation && !ready){ generation++; fail('O carregamento demorou demais.'); } },30000);
            }
            async function fallback(run){
              if(destroyed || run!==generation || viaBridge) return;
              viaBridge=true; ready=false; deadline(run);
              status.textContent='Carregando prévia por conexão alternativa…';
              try {
                const chunks=[]; let offset=0, total=null, mime='audio/mpeg';
                do {
                  const r=await api('preview_chunk',rel,offset);
                  if(destroyed || run!==generation) return;
                  if(!r?.ok || !r.b64 || r.offset!==offset || r.next_offset<=offset || r.next_offset>r.total || (total!==null && total!==r.total)) throw new Error('Prévia inválida');
                  chunks.push(Uint8Array.from(atob(r.b64),c=>c.charCodeAt(0)));
                  offset=r.next_offset; total=r.total; mime=r.mime;
                  status.textContent=`Carregando prévia… ${Math.round(offset/total*100)}%`;
                  deadline(run);
                } while(offset<total);
                objectUrl=URL.createObjectURL(new Blob(chunks,{type:mime}));
                audio.src=objectUrl; audio.load();
              } catch(e){ if(!destroyed && run===generation) fail('Não foi possível carregar a prévia.'); }
            }
            function onReady(){
              if(destroyed) return;
              ready=true; clearTimeout(timer); play.disabled=false; seek.disabled=false; retry.hidden=true;
              duration.textContent=fmtDur(audio.duration);
              status.textContent='Pronto para reproduzir.';
            }
            function onError(){ if(destroyed) return; if(viaBridge) fail('O navegador não conseguiu abrir a prévia.'); else fallback(generation); }
            async function start(){
              const run=++generation; started=true; release(); viaBridge=false; ready=false;
              play.disabled=true; seek.disabled=true; retry.hidden=true; status.textContent='Carregando prévia…'; audio.preload='auto';
              timer=setTimeout(()=>{ if(!ready) fallback(run); },8000);
              try { const url=await fileUrl(rel); if(destroyed || run!==generation || viaBridge) return; audio.src=url; audio.load(); }
              catch(e){ fallback(run); }
            }
            listen(audio,'canplay',onReady); listen(audio,'error',onError);
            listen(audio,'loadedmetadata',()=>{duration.textContent=fmtDur(audio.duration);});
            listen(audio,'timeupdate',()=>{ current.textContent=fmtDur(audio.currentTime); seek.value=audio.duration ? audio.currentTime/audio.duration*1000 : 0; seek.style.setProperty('--p',seek.value/10+'%'); });
            listen(audio,'play',()=>{ play.textContent='❚❚'; play.setAttribute('aria-label','Pausar prévia'); panel.classList.add('is-playing'); status.textContent='Reproduzindo.'; });
            listen(audio,'pause',()=>{ play.textContent='▶'; play.setAttribute('aria-label','Reproduzir prévia'); panel.classList.remove('is-playing'); if(ready) status.textContent='Pausado.'; });
            listen(audio,'ended',()=>{status.textContent='Reprodução concluída.';});
            listen(play,'click',async()=>{
              if(!audio.paused){ audio.pause(); return; }
              (S.resultPlayers||[]).forEach(p=>p.pause()); if(S.mixer) S.mixer.pause();
              try { if(audio.ended) audio.currentTime=0; await audio.play(); }
              catch(e){ if(!destroyed){status.textContent='Não foi possível iniciar. Clique em reproduzir novamente.';} }
            });
            listen(seek,'input',()=>{if(Number.isFinite(audio.duration)) audio.currentTime=Number(seek.value)/1000*audio.duration;});
            listen(volume,'input',()=>{audio.volume=Number(volume.value)/100;});
            listen(speed,'change',()=>{audio.playbackRate=Number(speed.value);});
            listen(retry,'click',start);
            listen(panel,'toggle',()=>{ if(panel.open){if(!started) start();} else audio.pause(); resize(); });
            if(panel.open) start();
            return {pause(){audio.pause();},destroy(){destroyed=true;generation++;events.forEach(([t,e,f])=>t.removeEventListener(e,f));release();}};
          }
          async function buildMixer(rows, duration){
            const audios = []; let ready = 0, destroyed = false; const withPrev = rows.map((f, i) => ({f, i})).filter(x => x.f.preview || !x.f.original);
            for(const {f, i} of withPrev){
              const a = new Audio(); a.preload = 'auto'; a.volume = 1; a.muted = !!f.original;
              audios.push({a, i, muted: !!f.original, solo:false, vol:1, rel: f.preview, viaB64:false});
            }
            // Carrega cada prévia: primeiro pelo proxy de portas do Colab; se não responder em 8 s (ou der erro),
            // busca o áudio em base64 pela ponte do Colab (mais lento e sujeito aos limites da ponte).
            async function loadViaB64(x){
              if(destroyed || x.viaB64) return; x.viaB64 = true;
              try{ const r = await api('preview_b64', x.rel); if(!r.ok) throw new Error(r.error); if(destroyed) return; x.a.src = `data:${r.mime};base64,${r.b64}`; x.a.load(); }
              catch(e){ x.failed = true; }
            }
            audios.forEach(x => {
              if(!x.rel){ x.failed = true; return; }
              let settled = false;
              x.a.addEventListener('canplay', () => { settled = true; }, {once:true});
              x.a.addEventListener('error', () => { if(!settled && !x.viaB64) loadViaB64(x); }, {once:true});
              setTimeout(() => { if(!settled && !x.viaB64) loadViaB64(x); }, 8000);
              fileUrl(x.rel).then(u => { if(!destroyed && !x.viaB64){ x.a.src = u; x.a.load(); } }).catch(() => loadViaB64(x));
            });
            const play = $('#mxPlay'), seek = $('#mxSeek'), cur = $('#mxCur'), state = $('#mxState');
            const master = audios[0] ? audios[0].a : null;
            let playing = false, seeking = false, dur = duration || 0;
            const list = $('#stemList');
            function selection(){
              const anySolo = audios.some(x => x.solo);
              return audios.filter(x => !rows[x.i].original && (anySolo ? x.solo : !x.muted) && x.vol > 0).map(x => ({stem: rows[x.i].stem, gain: x.vol}));
            }
            function applyGains(){
              const anySolo = audios.some(x => x.solo);
              audios.forEach(x => { const audible = anySolo ? x.solo : !x.muted; x.a.muted = !audible; x.a.volume = x.vol; const row = list.querySelector(`.stem[data-i="${x.i}"]`); if(row){ row.classList.toggle('muted', !audible); row.classList.toggle('playing', playing && audible); } });
              list.querySelectorAll('[data-mute]').forEach(b => b.classList.toggle('on', !!audios.find(x => x.i == b.dataset.mute)?.muted));
              list.querySelectorAll('[data-solo]').forEach(b => b.classList.toggle('on', !!audios.find(x => x.i == b.dataset.solo)?.solo));
              const selected = selection();
              const summary = $('#mixSummary'); if(summary) summary.textContent = selected.length ? 'Incluir: ' + selected.map(t => (rows.find(r => r.stem === t.stem)?.label || t.stem) + ' (' + Math.round(t.gain * 100) + '%)').join(', ') : 'Nenhuma faixa selecionada. Desative M ou Solo no Original e aumente o volume de uma faixa.';
              const button = $('#mixBtn'); if(button) button.disabled = !selected.length;
            }
            list.addEventListener('click', e => {
              const m = e.target.closest('[data-mute]'), s = e.target.closest('[data-solo]');
              if(m){ const x = audios.find(x => x.i == m.dataset.mute); if(x){ x.muted = !x.muted; applyGains(); } }
              if(s){ const x = audios.find(x => x.i == s.dataset.solo); if(x){ x.solo = !x.solo; applyGains(); } }
            });
            list.addEventListener('input', e => { const v = e.target.closest('[data-vol]'); if(!v) return; const x = audios.find(x => x.i == v.dataset.vol); if(x){ x.vol = v.value/100; v.style.setProperty('--p', v.value+'%'); applyGains(); } });
            function sync(){ if(!master) return; audios.forEach(x => { if(x.a !== master && Math.abs(x.a.currentTime - master.currentTime) > 0.12) x.a.currentTime = master.currentTime; }); }
            async function doPlay(){ (S.resultPlayers || []).forEach(p => p.pause()); if(!master || destroyed) return; if(ready < audios.length){ toast('Aguarde todas as prévias carregarem antes de reproduzir.', 'info', 4000); return; } play.disabled = true; try { sync(); const result = await Promise.allSettled(audios.map(x => x.a.play())); if(destroyed || result.some(r => r.status === 'rejected')){ doPause(); if(!destroyed) toast('Não foi possível reproduzir todas as faixas. Tente novamente ou baixe os arquivos.', 'err'); return; } playing = true; play.textContent = '❚❚'; state.textContent = 'tocando'; applyGains(); } finally { play.disabled = false; } }
            function doPause(){ audios.forEach(x => x.a.pause()); playing = false; play.textContent = '▶'; state.textContent = 'pausado'; applyGains(); }
            play.addEventListener('click', () => playing ? doPause() : doPlay());
            if(master){
              master.addEventListener('timeupdate', () => { if(seeking) return; const d = master.duration || dur; if(d) { seek.value = Math.round(master.currentTime / d * 1000); seek.style.setProperty('--p', (master.currentTime/d*100)+'%'); } cur.textContent = fmtDur(master.currentTime); if(playing && (master.currentTime|0) % 3 === 0) sync(); });
              master.addEventListener('ended', () => { doPause(); audios.forEach(x => x.a.currentTime = 0); });
              master.addEventListener('loadedmetadata', () => { dur = master.duration || dur; $('#mxDur').textContent = fmtDur(dur); });
              audios.forEach(x => { x.a.addEventListener('canplay', () => { ready++; if(ready >= audios.length){ state.textContent = 'pronto'; state.className = 'badge ok'; } else state.textContent = `carregando ${ready}/${audios.length}…`; }, {once:true}); });
              setTimeout(() => { if(ready < audios.length && state.textContent.startsWith('carregando')){ state.textContent = 'carregando pela ponte do Colab…'; } }, 9000);
            }
            seek.addEventListener('input', () => { seeking = true; const d = master && (master.duration || dur); if(d) cur.textContent = fmtDur(seek.value/1000*d); seek.style.setProperty('--p', (seek.value/10)+'%'); });
            seek.addEventListener('change', () => { const d = master && (master.duration || dur); if(d){ const t = seek.value/1000*d; audios.forEach(x => x.a.currentTime = t); } seeking = false; });
            applyGains();
            return { selection, pause: doPause, destroy(){ destroyed = true; doPause(); audios.forEach(x => { x.a.src = ''; x.a.load(); }); } };
          }
          root.addEventListener('click', e => { const b = e.target.closest('[data-dl]'); if(b && !b.disabled) downloadRel(b.dataset.dl, b); });
          function resetToStart(){ clearResultPlayers(); if(S.mixer){ S.mixer.destroy(); S.mixer = null; } S.jobId = null; S.source = null; S.analysis = null; S.choice = null; resetSelection(); $('#urlInput').value = ''; S.maxStep = 1; stopWatchModels(); goto(1); $$('.step').forEach(s=>s.classList.remove('done')); $('#urlInput').focus(); }

          // ---------- fila (visível em todas as etapas) ----------
          function startQueuePolling(){ if(S.queueTimer) return; pollQueue(); S.queueTimer = setInterval(pollQueue, 1500); }
          async function pollQueue(){
            let r; try { r = await api('jobs_list'); } catch(e){ return; }
            if(!r || !r.ok) return;
            renderQueue(r.jobs);
            r.jobs.forEach(j => {
              const prev = S.knownJobs[j.id];
              if(prev && prev !== j.status && ['done','error','cancelled'].includes(j.status) && !((S.step === 4 || S.step === 5) && S.jobId === j.id)){
                toast(j.status === 'cancelled' ? `Cancelado: ${j.title}` : (j.status === 'error' ? `Falhou: ${j.title} — ${j.error||''}` : (j.errors ? `Concluído com ${j.errors} erro(s): ${j.title}` : `Pronto: ${j.title}. Clique em Ver para ouvir e baixar.`)), j.status === 'done' && !j.errors ? 'ok' : 'warn', 8000);
              }
              S.knownJobs[j.id] = j.status;
            });
            if(!r.active){ clearInterval(S.queueTimer); S.queueTimer = null; }
          }
          const KIND_ICON = {download:'⬇', separate:'🎧', mix:'🎛'};
          function renderQueue(jobs){
            const bar = $('#queueBar'), list = $('#queueList');
            if(!jobs.length){ bar.hidden = true; return; }
            bar.hidden = false;
            const active = jobs.filter(j => j.status === 'queued' || j.status === 'running').length;
            $('#queueCount').textContent = active ? `${active} em andamento` : 'tudo concluído';
            $('#queueHint').textContent = active ? 'Downloads e separações rodam em paralelo. Você pode preparar outra tarefa enquanto isso.' : 'Clique em Ver para ouvir e baixar.';
            list.innerHTML = jobs.slice(-8).reverse().map(j => {
              const st = j.status === 'running' ? 'run' : (j.status === 'queued' ? '' : (j.status === 'done' ? (j.errors ? 'warn' : 'done') : (j.status === 'error' ? 'err' : 'warn')));
              const icon = j.status === 'running' ? '' : (j.status === 'queued' ? '…' : (j.status === 'done' ? (j.errors ? '!' : '✓') : (j.status === 'error' ? '!' : '–')));
              const sub = j.status === 'queued' ? 'Na fila, aguardando' : (j.status === 'running' ? esc(j.stage_label||'') + (j.kind === 'download' && j.detail ? ' · ' + esc(j.detail) : '') : (j.status === 'done' ? `${j.files} arquivo(s) pronto(s)${j.errors ? ` · ${j.errors} com erro` : ''}` : (j.status === 'error' ? esc(j.error||'Erro') : 'Cancelado')));
              const viewing = S.jobId === j.id && (S.step === 4 || S.step === 5);
              return `<div class="qjob ${viewing ? 'viewing' : ''}"><div class="st ${st}">${icon}</div><div><div class="ql" title="${esc(j.title)}">${KIND_ICON[j.kind]||''} ${esc(j.title||j.id)} <span class="muted" style="font-weight:500">· ${esc(j.label||'')}</span></div><div class="qs">${sub}</div></div><div class="bar ${j.status==='done' ? 'ok' : ''} ${j.status==='running' ? 'striped' : ''}"><i style="width:${j.status==='done'?100:j.percent}%"></i></div><button class="btn sm ${viewing ? 'ghost' : 'primary'}" data-view="${j.id}">${viewing ? 'Vendo' : 'Ver'}</button></div>`;
            }).join('');
            resize();
          }
          $('#queueList').addEventListener('click', e => { const b = e.target.closest('[data-view]'); if(b) viewJob(b.dataset.view); });
          async function viewJob(id){ S.jobId = id; let j; try { j = await api('job_status', id); } catch(e){ return; } if(S.jobId !== id) return; if(!j.ok) return toast(j.error, 'err'); if(j.status === 'queued' || j.status === 'running'){ goto(4); renderProc(j); poll(); } else { goto(5); renderResult(j); } pollQueue(); }

          if(session) window.addEventListener('pageshow', () => { if(!session.connected()) showFatal('Reabra o Studio pela célula 2.'); });
          bootstrap().catch(e => { toast('Erro ao iniciar: ' + e.message, 'err', 8000); showFatal('Falha ao chamar o Python (stemtube.bootstrap). Verifique se a célula 2 terminou sem erro.\n' + (e.stack || e.message)); });
        })();
        </script>
        <link rel="preconnect" href="https://fonts.googleapis.com">
        <link href="https://fonts.googleapis.com/css2?family=Sora:wght@500;600;700;800&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
        """
    except Exception as _e:
        import traceback as _tb
        _pronto = False
        _erro('Erro ao preparar o aplicativo. Copie a mensagem abaixo e envie para suporte.')
        print(_tb.format_exc())
LAUNCHER = '<div id="studio-launcher" style="font:15px/1.5 system-ui,sans-serif;padding:12px">\n  <button type="button" style="background:#573ac0;color:white;border:0;border-radius:9px;padding:12px 18px;font:inherit;cursor:pointer">Abrir StemTube Studio em nova aba</button>\n  <p role="status" aria-live="polite">Mantenha este notebook aberto e a sessão conectada enquanto usa o Studio.</p>\n</div>\n<script>\n(() => {\n  const launcher = document.getElementById(\'studio-launcher\');\n  const message = launcher.querySelector(\'[role=status]\');\n  const page = __STUDIO_DOCUMENT__;\n  let tab;\n  launcher.querySelector(\'button\').onclick = () => {\n    try {\n      if (tab && !tab.closed && tab.document.getElementById(\'stemtube\')) {\n        tab.focus();\n        return;\n      }\n      const kernel = window.google?.colab?.kernel;\n      if (!kernel) throw new Error(\'A conexão com o Colab está indisponível.\');\n      tab = window.open(\'\', \'_blank\');\n      if (!tab) throw new Error(\'O navegador bloqueou a abertura da aba.\');\n      tab.document.open();\n      tab.opener = null;\n      tab.stemTubeSession = Object.freeze({\n        invoke: (name, args) => kernel.invokeFunction(\'stemtube.\' + name, args, {}),\n        mediaBase: port => kernel.proxyPort(port, {cache:true}),\n        connected: () => launcher.isConnected && !window.closed\n      });\n      tab.document.write(page);\n      tab.document.close();\n      message.textContent = \'Studio aberto. Mantenha este notebook conectado. Este botão também reabre o aplicativo.\';\n    } catch (error) {\n      // Uma aba que já navegou para outro site não pertence mais ao Studio.\n      try { if (tab && tab.location.href === \'about:blank\') tab.close(); } catch (_) {}\n      tab = null;\n      message.textContent = error.message + \' Permita pop-ups e tente novamente, ou desmarque abrir_em_nova_aba e execute a célula 2.\';\n    }\n  };\n})();\n</script>\n'

if _pronto:
    try:
        if abrir_em_nova_aba:
            _page = ('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
                     '<meta name="viewport" content="width=device-width, initial-scale=1">'
                     '<title>StemTube Studio</title></head><body class="studio-window">' + APP_HTML + '</body></html>')
            # A página contém scripts próprios. Escape "<" antes de inseri-la no script lançador.
            _literal = _json.dumps(_page, ensure_ascii=True).replace('<', r'\u003c')
            _display(_HTML(LAUNCHER.replace('__STUDIO_DOCUMENT__', _literal)))
        else:
            _display(_HTML(APP_HTML))
    except Exception as _e:
        import traceback as _tb
        _erro('Falha ao abrir o aplicativo. Copie a mensagem abaixo e envie para suporte.')
        print(_tb.format_exc())


In [ ]:
#@title 3️⃣ (Opcional) Limpar arquivos temporários com segurança { display-mode: "form" }
#@markdown Apaga downloads, faixas e áudios temporários desta sessão do Colab. Não limpa durante tarefas ativas e preserva os modelos de IA, os cookies e as cópias no Drive.
confirmar_limpeza = False  #@param {type:"boolean"}
if not confirmar_limpeza:
    print('Limpeza desativada. Marque confirmar_limpeza e execute esta célula para apagar os temporários.')
elif 'cb_cleanup' not in globals():
    print('Execute a célula 2 antes de usar a limpeza.')
else:
    _cleanup_result = cb_cleanup().data
    if not _cleanup_result.get('ok'):
        print('⚠️ ' + _cleanup_result.get('error', 'Não foi possível limpar.'))
    else:
        print('✅ Arquivos temporários e histórico limpos. Execute a célula 2 para renovar a interface.')


## ℹ️ Sobre o StemTube Studio

O StemTube Studio reúne, em um único aplicativo, as ferramentas **StemLab** (separação de faixas com IA e mixer) e **YT Downloader Pessoal** (vídeos, playlists e áudio do YouTube), sem repetir nada: um só login, uma só seção de cookies, uma só fila e um só destino no Drive (`Meu Drive/StemTube`).

### O que acontece por baixo dos panos

| Recurso | Como funciona |
|---|---|
| Downloads | `yt-dlp` com runtime JavaScript (deno), `ffmpeg` para converter, empacotar em MP4 e embutir capa/metadados |
| Recorte por tempo | O arquivo completo é baixado e cortado com `ffmpeg`; vídeo é recodificado (H.264) para o corte ser exato, áudio é recodificado no formato escolhido; original usa corte aproximado por pacotes |
| Separação de faixas | [audio-separator](https://github.com/nomadkaraoke/python-audio-separator) com os modelos `model_bs_roformer_ep_317_sdr_12.9755.ckpt` (voz), `htdemucs_ft.yaml` (4 faixas) e `htdemucs_6s.yaml` (6 faixas), na GPU quando disponível |
| Mixer | Prévias MP3 de cada faixa tocadas em sincronia no navegador; a mixagem final é renderizada pelo `ffmpeg` a partir das faixas completas, com limitador de picos |
| Fila | Duas pistas independentes: downloads e processamento na GPU rodam em paralelo, um item por vez em cada pista |
| Segurança | Login Google obrigatório, validação de links e de cookies, bloqueio de travessia de diretórios, limites de upload e de duração |

### Solução de problemas

- **“Confirme que você não é um robô” / vídeo privado / +18** → carregue os cookies da sua conta na seção *Cookies* e tente de novo.
- **Sem GPU** → *Ambiente de execução ▸ Alterar o tipo de ambiente de execução ▸ T4 GPU*, depois rode as células 1 e 2 novamente. Downloads não precisam de GPU.
- **A memória da GPU estourou** → use uma música mais curta (limite de 25 min) ou o modo *Voz + Playback*.
- **A nova aba não abriu**: clique no botão da célula 2. Se o navegador bloquear pop-ups, permita a abertura ou desmarque `abrir_em_nova_aba` e execute a célula 2 novamente. Se a sessão expirou, repita as células 1 e 2. Para voltar após fechar ou atualizar a aba do Studio, use novamente o botão do notebook.
- **Mixes automáticos do YouTube (list=RD…)** não podem ser listados fora do player: baixe o vídeo ou salve os itens em uma playlist sua.

### Limites e observações

- Separação: áudios de até **25 minutos** (limite operacional da GPU gratuita). Arquivos locais de até **1 GB**.
- Playlists: até **1000 vídeos** por análise.
- O áudio do YouTube tem no máximo ~130–160 kbps (256 kbps em alguns casos): FLAC e WAV preservam o que foi recebido, mas não recuperam qualidade perdida.
- As estimativas de tempo são aproximadas. Os arquivos ficam em `/content/stemtube` só enquanto a sessão estiver ativa; use a célula 3 para limpar.

Código-fonte, testes e servidor de desenvolvimento: [github.com/magoupdown/stemtube-studio](https://github.com/magoupdown/stemtube-studio)

---
*Desenvolvido por **@emersonms** - 2026*
